# 업무 프로세스 추출 PoC — Deep Unsupervised / Self-Supervised V6.2

V6.2는 **V6.0의 모델 구조와 V6.2의 고객용 시나리오 화면을 유지**하면서,
`sessions.user_id`를 이용한 **사용자 업무 궤적(User Trajectory) 분석 계층**을 추가합니다.

핵심 원칙:

- `user_id`를 event embedding의 feature로 넣지 않습니다.
- `user_id`를 업무 label처럼 학습하지 않습니다.
- 기존 event → segment → workflow 모델은 그대로 유지합니다.
- `sessions.session_id → sessions.user_id`를 통해 이미 추출된 process instance를 사용자별 시간 순서로 연결합니다.
- 같은 사용자가 여러 session을 이동하며 수행한 업무를 상위 시나리오 후보로 구성합니다.
- 서로 무관한 장시간 작업을 억지로 이어붙이지 않도록 generic temporal episode/window를 사용합니다.
- 사전 정의 7개 PoC Ground Truth는 여전히 **post-hoc 비교에만 사용**합니다.

### V6.2 구조

```text
Raw Events
→ V6.0 Event / Segment / Workflow Model
→ V5.7 Calibrated Threshold
→ V6 Consensus Process Models
→ Session-safe Process Instances
              │
              └─ sessions.session_id → user_id
                        ↓
                User Trajectory Episodes
                        ↓
             Cross-session Workflow Sequence
                        ↓
          User-level Scenario Candidate Windows
                        ↓
             7 PoC Ground Truth 사후 비교
```

### 고객 화면

PoC 시나리오를 선택하면 다음 순서로 보여줍니다.

```text
Ground Truth
→ 동일 사용자의 대표 업무 궤적
→ 선택한 실제 실행의 상세 compressed path
→ Aggregate process graph
→ step/input evidence
```

하위 수준 단계는 유지하고, `SP03`, `WP02`처럼 고객이 해석할 수 없는 prototype ID 단독 단계만 숨깁니다.

## Colab 실행 안내

권장 런타임: **T4 GPU**

입력 경로는 아래 두 형태를 자동 지원합니다.

```text
/content/drive/MyDrive/Colab Notebooks/data/events.csv
```

또는 dataset별 폴더가 있으면:

```text
/content/drive/MyDrive/Colab Notebooks/data/urangi/events.csv
```

`events.csv`만 필수입니다.

V6.2 결과는 선택된 데이터 폴더 아래에 저장됩니다.

```text
artifacts/business_flow_deep_v6_2/
```

semantic embedding은 cache로 재사용합니다.

## 1. Setup / Google Drive / Config

In [1]:
import os, re, json, math, random, shutil, hashlib, sys, subprocess, gc, html as html_lib
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
    v_measure_score,
    silhouette_score,
)


SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODEL_VERSION = "v6.2"

# ============================================================
# V6.2 full-data execution + user trajectory scenario layer
# ============================================================
# V6.2 keeps V6.0 modeling unchanged and adds session.user_id based higher-level trajectory analysis.
USE_SEMANTIC_ENCODER = True
SEMANTIC_MODEL_NAME = "intfloat/multilingual-e5-small"
SEMANTIC_BATCH_SIZE = 128
SEMANTIC_FALLBACK_DIM = 384
SEMANTIC_CACHE = True

# Semantic influence ablation.
# 0.00 ≈ semantic branch disabled at fusion
# 1.00 = V5.2
# V5.3 default = 0.20
SEMANTIC_WEIGHT = 0.20
SEMANTIC_WEIGHT_CANDIDATES = [0.00, 0.10, 0.20, 0.30, 0.50, 1.00]

# Known-process evaluation is POST-HOC ONLY.
ENABLE_GOLD_EVAL = True
GOLD_EVAL_DATASET = "poc_full"

# ============================================================
# Structural-aware boundary
# ============================================================
USE_STRUCTURAL_BOUNDARY = True
STRUCT_DIM = 48
STRUCT_TARGET_WEIGHT = 0.35
STRUCT_BOUNDARY_SPARSITY_WEIGHT = 0.01
STRUCTURAL_FIELDS = [
    'struct_api_family',
    'struct_page_path',
    'struct_page_title',
]

# ============================================================
# V5.6 Transition-Contrastive Workflow Boundary
# ============================================================
# Gold 업무명/API 업무 mapping을 사용하지 않는다.
# 실제 인접 segment pair만 self-supervised positive relation으로 사용하고
# 같은 batch의 다른 target segment를 contrastive negatives로 사용한다.
USE_TRANSITION_CONTRASTIVE_WORKFLOW_BOUNDARY = True

WORKFLOW_TRANSITION_EPOCHS = 8
WORKFLOW_TRANSITION_BATCH_SIZE = 256
WORKFLOW_TRANSITION_LR = 3e-4
WORKFLOW_TRANSITION_TEMPERATURE = 0.10

# 최종 workflow boundary score:
# (1-w) * V5.4 contextual boundary
# + w * transition surprise
WORKFLOW_TRANSITION_BLEND_WEIGHT = 0.45

# ============================================================
# V5.7 Boundary Threshold Calibration
# ============================================================
THRESHOLD_MODE = "calibrated_gmm"
THRESHOLD_POSTERIOR_MIN = 0.80
THRESHOLD_BIC_GAIN_MIN = 10.0
THRESHOLD_MIN_SEPARATION_SIGMA = 1.00
THRESHOLD_ROBUST_Z_FALLBACK = 2.50

# ============================================================
# V5.7 Evaluation v2
# ============================================================
EVAL_V2_PRIMARY_MODE = "context_ge1"
EVAL_V2_PERSISTENT_MIN_HITS = 2
EVAL_V2_PERSISTENT_MIN_DWELL_SEC = 3.0

# Prototype assignment stability — evaluation only
ENABLE_PROTOTYPE_STABILITY_EVAL = True
PROTOTYPE_STABILITY_SEEDS = [41, 42, 43]

# ============================================================
# V6.0 Stable Prototype Ensemble / Consensus
# ============================================================
USE_CONSENSUS_PROTOTYPES = True

PROTOTYPE_ENSEMBLE_SEEDS = [41, 42, 43, 44, 45]
PROTOTYPE_OPT_STEPS = 300
PROTOTYPE_LOG_EVERY = 50
PROTOTYPE_EARLY_STOP_PATIENCE = 80
PROTOTYPE_MIN_DELTA = 1e-5

# Consensus cluster count search.
# Workflow는 고객 납품 목표를 반영해 7개 이상 후보를 탐색하지만
# 어떤 gold 업무 label도 사용하지 않는다.
SEGMENT_CONSENSUS_K_MIN = 10
SEGMENT_CONSENSUS_K_MAX = 18
WORKFLOW_CONSENSUS_K_MIN = 7
WORKFLOW_CONSENSUS_K_MAX = 12

# ============================================================
# V6.0 Customer Report
# ============================================================
REPORT_TARGET_MIN_WORKFLOWS = 7
REPORT_MAX_WORKFLOWS = 10
REPORT_MIN_INSTANCE_SUPPORT = 5
REPORT_SCORE_THRESHOLD = 0.55

# Explicitly disabled V5.0/V5.1 experimental branches
USE_SALIENCE_GATE = False
USE_VICREG = False

# ============================================================
# Original V4.2 deep unsupervised / self-supervised config
# ============================================================
FAST_DEV_RUN = False

# V6.1 full-data run.
# 모델은 동일하고, 입력 파일의 전체 row를 그대로 사용합니다.
FULL_DATA_RUN = True
DATASET_NAME = "poc_full"

# 자동 탐색 순서:
#   data/poc_full/events.csv
#   data/full/events.csv
#   data/events.csv
#   data/urangi/events.csv (fallback)
FULL_DATA_SUBDIR_CANDIDATES = [
    "poc_full",
    "full",
    "",
    "urangi",
]

# ============================================================
# V6.2 User Trajectory Analysis
# ============================================================
USE_USER_TRAJECTORY = True

# sessions.csv에서 session_id -> user_id를 가져오되
# user_id는 base event representation / prototype 학습 feature로 사용하지 않는다.
SESSION_USER_ID_COLUMN = "user_id"
SESSION_KEY_CANDIDATES = [
    "session_id",
    "id",
]

# 같은 사용자의 서로 가까운 process instance만 하나의 상위 trajectory episode로 묶는다.
# 장시간 떨어진 업무를 억지로 연결하지 않기 위한 generic temporal constraint.
USER_TRAJECTORY_GAP_MINUTES = 180
USER_TRAJECTORY_BREAK_ON_DATE_CHANGE = True

# Ground Truth 사후 비교 시 한 사용자의 trajectory에서 탐색하는
# contiguous process-instance window의 최대 크기/시간.
USER_TRAJECTORY_MAX_WINDOW_INSTANCES = 12
USER_TRAJECTORY_MAX_WINDOW_MINUTES = 240

# 시나리오가 여러 사용자에게서 반복되었다고 볼 내부 support 기준.
USER_TRAJECTORY_SUPPORT_SCORE = 0.40

EVENT_WINDOW=128; EVENT_STRIDE=64; MAX_TEXT_BYTES=192; MASK_PROB=0.15
D_MODEL=128; NHEAD=4; EVENT_LAYERS=3; FF=256; DROPOUT=0.1
BATCH_SIZE=16; EVENT_EPOCHS=12; BOUNDARY_EPOCHS=6

SEGMENT_WINDOW=32; SEGMENT_STRIDE=16
SEGMENT_EPOCHS=8; WORKFLOW_BOUNDARY_EPOCHS=6

MIN_SEGMENT_EVENTS=3; MIN_WORKFLOW_SEGMENTS=2

PROTOTYPE_EPOCHS=20; PROTOTYPE_LR=1e-3
MAX_SEGMENT_PROTOTYPES=32; MAX_WORKFLOW_PROTOTYPES=16

# ============================================================
# Output policy
# ============================================================
KEEP_MODEL_BUNDLE = True
PREVIEW_MAX_MODELS = 12
PREVIEW_INSTANCES_PER_MODEL = 20
PREVIEW_MAX_INPUTS_PER_INSTANCE = 120

if FAST_DEV_RUN:
    EVENT_WINDOW=64; EVENT_STRIDE=32; MAX_TEXT_BYTES=128
    D_MODEL=64; EVENT_LAYERS=2; FF=128
    BATCH_SIZE=8; EVENT_EPOCHS=2; BOUNDARY_EPOCHS=2
    SEGMENT_WINDOW=16; SEGMENT_STRIDE=8
    SEGMENT_EPOCHS=2; WORKFLOW_BOUNDARY_EPOCHS=2
    PROTOTYPE_EPOCHS=3
    SEMANTIC_BATCH_SIZE=64
    WORKFLOW_TRANSITION_EPOCHS=2
    WORKFLOW_TRANSITION_BATCH_SIZE=128
    PROTOTYPE_ENSEMBLE_SEEDS=[41,42,43]
    PROTOTYPE_OPT_STEPS=40
    SEGMENT_CONSENSUS_K_MAX=12
    WORKFLOW_CONSENSUS_K_MAX=8

# ============================================================
# Colab bootstrap
# ============================================================
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    try:
        import sentence_transformers  # noqa: F401
    except Exception:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "sentence-transformers>=3.0.0"
            ]
        )

    from google.colab import drive
    drive.mount("/content/drive")

    DATA_ROOT = Path(
        "/content/drive/MyDrive/Colab Notebooks/data"
    )
else:
    DATA_ROOT = Path("/mnt/data")

# ============================================================
# V6.1 full-data source auto-discovery
# ============================================================
# 전체 데이터 파일을 우선 탐색합니다.
# 어떤 후보에서도 찾지 못하면 명확하게 오류를 발생시킵니다.
_data_candidates = []

for _subdir in FULL_DATA_SUBDIR_CANDIDATES:
    _base = (
        DATA_ROOT / _subdir
        if _subdir
        else DATA_ROOT
    )

    _data_candidates.append(
        _base
    )

BASE_DIR = None

for _candidate in _data_candidates:
    _events = _candidate / "events.csv"

    if _events.exists():
        BASE_DIR = _candidate
        break

if BASE_DIR is None:
    raise FileNotFoundError(
        "V6.1 full-data events.csv를 찾지 못했습니다.\n"
        + "확인한 위치:\n"
        + "\n".join(
            str(x / "events.csv")
            for x in _data_candidates
        )
    )

EVENTS_PATH = BASE_DIR / "events.csv"
TASKS_PATH = BASE_DIR / "tasks.csv"
SESSIONS_PATH = BASE_DIR / "sessions.csv"

OUTPUT_DIR = BASE_DIR / "artifacts" / "business_flow_deep_v6_2"

RESET_OUTPUT_DIR = True
if RESET_OUTPUT_DIR and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_RESULTS_DIR = OUTPUT_DIR / "main_results"
MAIN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DIAGNOSTICS_DIR = OUTPUT_DIR / "diagnostics"
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)

# semantic cache는 OUTPUT_DIR 바깥에 유지
SEMANTIC_CACHE_DIR = BASE_DIR / "semantic_cache"
SEMANTIC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

if not EVENTS_PATH.exists():
    raise FileNotFoundError(
        f"events file not found: {EVENTS_PATH}"
    )

print("MODEL_VERSION:", MODEL_VERSION)
print("FULL_DATA_RUN:", FULL_DATA_RUN)
print("DATASET_NAME:", DATASET_NAME)
print("selected full-data BASE_DIR:", BASE_DIR)
print("device:", device)
print("IN_COLAB:", IN_COLAB)
print("EVENTS_PATH:", EVENTS_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("semantic encoder:", USE_SEMANTIC_ENCODER, SEMANTIC_MODEL_NAME)
print("semantic weight:", SEMANTIC_WEIGHT)
print("gold evaluation:", ENABLE_GOLD_EVAL, GOLD_EVAL_DATASET)
print("structural boundary:", USE_STRUCTURAL_BOUNDARY)
print("struct target weight:", STRUCT_TARGET_WEIGHT)
print("transition-contrastive workflow boundary:", USE_TRANSITION_CONTRASTIVE_WORKFLOW_BOUNDARY)
print("transition blend weight:", WORKFLOW_TRANSITION_BLEND_WEIGHT)
print("transition epochs:", WORKFLOW_TRANSITION_EPOCHS)
print("threshold mode:", THRESHOLD_MODE)
print("threshold posterior min:", THRESHOLD_POSTERIOR_MIN)
print("evaluation v2 primary:", EVAL_V2_PRIMARY_MODE)
print("prototype stability seeds:", PROTOTYPE_STABILITY_SEEDS)
print("V6 consensus prototypes:", USE_CONSENSUS_PROTOTYPES)
print("user trajectory analysis:", USE_USER_TRAJECTORY)
print("trajectory gap minutes:", USER_TRAJECTORY_GAP_MINUTES)
print("trajectory max window instances:", USER_TRAJECTORY_MAX_WINDOW_INSTANCES)
print("prototype ensemble seeds:", PROTOTYPE_ENSEMBLE_SEEDS)
print("prototype optimization steps:", PROTOTYPE_OPT_STEPS)
print("workflow consensus k range:", WORKFLOW_CONSENSUS_K_MIN, WORKFLOW_CONSENSUS_K_MAX)
print("customer report target:", REPORT_TARGET_MIN_WORKFLOWS)
print("salience gate:", USE_SALIENCE_GATE)
print("VICReg:", USE_VICREG)


Mounted at /content/drive
MODEL_VERSION: v6.2
FULL_DATA_RUN: True
DATASET_NAME: poc_full
selected full-data BASE_DIR: /content/drive/MyDrive/Colab Notebooks/data/urangi
device: cuda
IN_COLAB: True
EVENTS_PATH: /content/drive/MyDrive/Colab Notebooks/data/urangi/events.csv
OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/data/urangi/artifacts/business_flow_deep_v6_2
semantic encoder: True intfloat/multilingual-e5-small
semantic weight: 0.2
gold evaluation: True poc_full
structural boundary: True
struct target weight: 0.35
transition-contrastive workflow boundary: True
transition blend weight: 0.45
transition epochs: 8
threshold mode: calibrated_gmm
threshold posterior min: 0.8
evaluation v2 primary: context_ge1
prototype stability seeds: [41, 42, 43]
V6 consensus prototypes: True
user trajectory analysis: True
trajectory gap minutes: 180
trajectory max window instances: 12
prototype ensemble seeds: [41, 42, 43, 44, 45]
prototype optimization steps: 300
workflow consensus k range: 7 12


### 입력 파일 확인

In [2]:
for name, path in {
    "events": EVENTS_PATH,
    "tasks": TASKS_PATH,
    "sessions": SESSIONS_PATH,
}.items():
    print(
        f"{name:8s}",
        "OK" if path.exists() else "MISSING",
        "→",
        path
    )

if not EVENTS_PATH.exists():
    raise FileNotFoundError(
        "events.csv가 없습니다. BASE_DIR 또는 파일명을 확인해주세요."
    )

events   OK → /content/drive/MyDrive/Colab Notebooks/data/urangi/events.csv
tasks    OK → /content/drive/MyDrive/Colab Notebooks/data/urangi/tasks.csv
sessions OK → /content/drive/MyDrive/Colab Notebooks/data/urangi/sessions.csv


## 2. Raw event preprocessing / Byte + Frozen Semantic encoding

In [3]:
def safe_str(x):
    if x is None:
        return ''

    try:
        if pd.isna(x):
            return ''
    except Exception:
        pass

    s = str(x).strip()

    return (
        ''
        if s.lower() in {
            'nan',
            'none',
            'null',
            'nat',
            '[null]'
        }
        else s
    )


def url_path(x):
    s = safe_str(x)

    if not s:
        return ''

    try:
        p = urlparse(s)
        return p.path or s.split('?')[0]

    except Exception:
        return s.split('?')[0]


def structural_api_family(x):
    p=url_path(x)
    if not p:
        return ''
    parts=[z for z in p.split('/') if z]
    if len(parts)<=1:
        return p
    return '/' + '/'.join(parts[:-1])

def structural_page_path(x):
    return url_path(x)

def structural_title(x):
    return safe_str(x)


def derive_worker_id(row):
    for col in [
        'actor_key',
        'login_id',
        'user_id',
        'worker_id',
        'employee_id'
    ]:
        if col in row.index:
            v = safe_str(
                row.get(col)
            )

            if v:
                return v

    wk = safe_str(
        row.get('workflow_key')
    )

    if wk:
        # workflow 번호 자체는 정답으로 사용하지 않고 actor prefix만 stream id로 사용
        return wk.split('#W')[0]

    sid = safe_str(
        row.get('session_id')
    )

    if sid:
        return f"session:{sid}"

    return 'worker_0'


def build_event_text(row):
    """
    V4.2 Byte CNN 입력.
    technical string + UI text를 모두 유지한다.
    """
    fields = [
        ('EVT', row.get('event_type')),
        ('INT', row.get('interaction_type')),
        ('TAG', row.get('element_tag')),
        ('TEXT', row.get('element_text')),
        ('LABEL', row.get('associated_label')),
        ('API', row.get('api_path')),
        ('PAGE', row.get('page_title')),
        ('PATH', url_path(row.get('page_url'))),
        ('TESTID', row.get('data_testid')),
    ]

    parts = []

    for k, v in fields:
        s = safe_str(v)

        if s:
            parts.append(
                f'[{k}] {s}'
            )

    return ' '.join(parts)[:1200]


def build_semantic_text(row):
    """
    V5.2의 유일한 representation 추가.

    Byte CNN은 API/path/technical string을 그대로 처리하고,
    semantic encoder는 사람이 읽는 UI 의미만 보강한다.
    """
    fields = [
        ('PAGE', row.get('page_title')),
        ('LABEL', row.get('associated_label')),
        ('TEXT', row.get('element_text')),
    ]

    parts = []

    for k, v in fields:
        s = safe_str(v)

        if s:
            parts.append(
                f'{k}: {s}'
            )

    return ' | '.join(parts)[:700]


def byte_encode(text, max_len):
    b = safe_str(text).encode(
        'utf-8',
        errors='ignore'
    )[:max_len]

    arr = np.zeros(
        max_len,
        dtype=np.int64
    )

    if b:
        arr[:len(b)] = (
            np.frombuffer(
                b,
                dtype=np.uint8
            ).astype(np.int64)
            + 1
        )

    return arr


class Vocab:
    def __init__(
        self,
        vals,
        min_freq=1
    ):
        c = Counter(
            safe_str(v)
            for v in vals
            if safe_str(v)
        )

        self.itos = (
            [
                '<PAD>',
                '<UNK>'
            ]
            + [
                t
                for t, n in c.most_common()
                if n >= min_freq
            ]
        )

        self.stoi = {
            t: i
            for i, t in enumerate(
                self.itos
            )
        }

    def enc(self, x):
        return self.stoi.get(
            safe_str(x),
            1
        )

    def __len__(self):
        return len(
            self.itos
        )


def prep(df):
    df = df.copy()

    df[
        'event_time_parsed'
    ] = pd.to_datetime(
        df[
            'event_time'
        ],
        errors='coerce'
    )

    sort_cols = [
        'event_time_parsed'
    ]

    if 'id' in df.columns:
        sort_cols.append(
            'id'
        )

    df = df.sort_values(
        sort_cols,
        kind='stable'
    ).reset_index(
        drop=True
    )

    df[
        'global_event_idx'
    ] = np.arange(
        len(df)
    )

    df[
        'stream_id'
    ] = df.apply(
        derive_worker_id,
        axis=1
    )

    df[
        'event_text'
    ] = df.apply(
        build_event_text,
        axis=1
    )

    df[
        'semantic_text'
    ] = df.apply(
        build_semantic_text,
        axis=1
    )

    df['struct_api_family'] = df['api_path'].map(structural_api_family) if 'api_path' in df.columns else ''
    df['struct_page_path'] = df['page_url'].map(structural_page_path) if 'page_url' in df.columns else ''
    df['struct_page_title'] = df['page_title'].map(structural_title) if 'page_title' in df.columns else ''

    df[
        'delta_sec'
    ] = (
        df
        .groupby(
            'stream_id'
        )[
            'event_time_parsed'
        ]
        .diff()
        .dt
        .total_seconds()
        .clip(
            lower=0
        )
        .fillna(
            0
        )
    )

    for c in [
        'step_duration_ms',
        'api_latency_ms',
        'api_status_code'
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(
                df[c],
                errors='coerce'
            ).fillna(
                0
            )
        else:
            df[c] = 0.0

    return df


def temporal_split(
    df,
    ratio=.8
):
    train = []
    valid = []

    for sid, g in df.groupby(
        'stream_id',
        sort=False
    ):
        g = g.sort_values(
            [
                'event_time_parsed',
                'global_event_idx'
            ],
            kind='stable'
        )

        if len(g) >= 2:
            cut = max(
                1,
                min(
                    len(g) - 1,
                    int(
                        len(g)
                        * ratio
                    )
                )
            )
        else:
            cut = len(g)

        train.extend(
            g.iloc[
                :cut
            ].index.tolist()
        )

        valid.extend(
            g.iloc[
                cut:
            ].index.tolist()
        )

    return (
        df.loc[
            train
        ].copy(),
        df.loc[
            valid
        ].copy()
    )


def read_table(path):
    if path is None:
        return None

    path = Path(path)

    if not path.exists():
        return None

    suffix = path.suffix.lower()

    if suffix == '.csv':
        return pd.read_csv(
            path,
            encoding='utf-8-sig',
            low_memory=False
        )

    if suffix in {
        '.xlsx',
        '.xls'
    }:
        return pd.read_excel(
            path
        )

    raise ValueError(
        f"지원하지 않는 파일 형식: {path}"
    )


events_raw = read_table(
    EVENTS_PATH
)

# V6.1: no row cap / no sample / no account whitelist.
# events.csv에 들어있는 전체 row를 그대로 모델 입력으로 사용합니다.
if events_raw is not None:
    print(
        "V6.1 full-data events loaded:",
        f"{len(events_raw):,}",
        "rows from",
        EVENTS_PATH
    )

if events_raw is None:
    raise FileNotFoundError(
        f"events file not found: {EVENTS_PATH}"
    )

tasks = read_table(
    TASKS_PATH
)

sessions = read_table(
    SESSIONS_PATH
)

print(
    "events:",
    events_raw.shape
)

print(
    "tasks:",
    tasks.shape
    if tasks is not None
    else None
)

print(
    "sessions:",
    sessions.shape
    if sessions is not None
    else None
)

if tasks is not None:
    print(
        "task columns:",
        tasks.columns.tolist()
    )


df = prep(
    events_raw
)


# ============================================================
# V6.2 session_id -> user_id mapping for higher-level trajectory only
# ============================================================
def _build_session_user_map(
    sessions_df
):
    if (
        sessions_df is None
        or not len(
            sessions_df
        )
        or SESSION_USER_ID_COLUMN not in sessions_df.columns
    ):
        return (
            {},
            None
        )

    session_key=None

    for candidate in SESSION_KEY_CANDIDATES:
        if candidate in sessions_df.columns:
            session_key=candidate
            break

    if session_key is None:
        return (
            {},
            None
        )

    tmp=sessions_df[
        [
            session_key,
            SESSION_USER_ID_COLUMN
        ]
    ].copy()

    tmp[
        session_key
    ]=tmp[
        session_key
    ].map(
        safe_str
    )

    tmp[
        SESSION_USER_ID_COLUMN
    ]=tmp[
        SESSION_USER_ID_COLUMN
    ].map(
        safe_str
    )

    tmp=tmp[
        (
            tmp[
                session_key
            ]!=''
        )
        & (
            tmp[
                SESSION_USER_ID_COLUMN
            ]!=''
        )
    ].drop_duplicates(
        session_key,
        keep='last'
    )

    return (
        dict(
            zip(
                tmp[
                    session_key
                ],
                tmp[
                    SESSION_USER_ID_COLUMN
                ]
            )
        ),
        session_key
    )


session_user_map,session_user_key=(
    _build_session_user_map(
        sessions
    )
)

df[
    'trajectory_user_id'
]=''

if 'session_id' in df.columns:
    df[
        'trajectory_user_id'
    ]=(
        df[
            'session_id'
        ]
        .map(
            safe_str
        )
        .map(
            session_user_map
        )
        .fillna(
            ''
        )
        .map(
            safe_str
        )
    )

# sessions.user_id를 최우선으로 사용하되,
# mapping이 없는 event는 events 자체에 존재하는 actor/user identity로 보완한다.
for _fallback_col in [
    'user_id',
    'actor_key',
    'login_id',
    'worker_id',
    'employee_id',
]:
    if _fallback_col in df.columns:
        _empty=(
            df[
                'trajectory_user_id'
            ]==''
        )

        df.loc[
            _empty,
            'trajectory_user_id'
        ]=df.loc[
            _empty,
            _fallback_col
        ].map(
            safe_str
        )

print(
    "session user mapping:",
    f"{len(session_user_map):,}",
    "sessions",
    "key=",
    session_user_key
)

print(
    "trajectory_user_id coverage:",
    f"{int((df['trajectory_user_id']!='').sum()):,}"
    "/"
    f"{len(df):,}",
    (
        f"({(df['trajectory_user_id']!='').mean()*100:.1f}%)"
        if len(
            df
        )
        else ""
    )
)

# IMPORTANT:
# trajectory_user_id는 이 시점 이후 higher-level grouping/export에만 사용한다.
# stream_id / event embedding / training feature에는 추가하지 않는다.

trdf, vdf = temporal_split(
    df
)

cat_cols = [
    'event_type',
    'interaction_type',
    'element_tag',
    'api_method'
]

vocabs = {
    c: Vocab(
        trdf[c]
        if c in trdf.columns
        else []
    )
    for c in cat_cols
}


struct_cols=list(STRUCTURAL_FIELDS)
struct_vocabs={c:Vocab(trdf[c] if c in trdf.columns else []) for c in struct_cols}
print('structural vocab sizes:',{c:len(struct_vocabs[c]) for c in struct_cols})


def raw_num(frame):
    x = np.stack(
        [
            np.log1p(
                frame[
                    'delta_sec'
                ].values
            ),

            np.log1p(
                frame[
                    'step_duration_ms'
                ].values
            ),

            np.log1p(
                frame[
                    'api_latency_ms'
                ].values
            ),

            frame[
                'api_status_code'
            ].values
            / 600.0
        ],
        axis=1
    ).astype(
        'float32'
    )

    return x


tr_num = raw_num(
    trdf
)

mu = tr_num.mean(
    0
)

sd = (
    tr_num.std(
        0
    )
    + 1e-6
)


def _ensure_sentence_transformers():
    try:
        import sentence_transformers  # noqa: F401
        return

    except Exception:
        print(
            "sentence-transformers not found. Installing..."
        )

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "sentence-transformers>=3.0.0"
            ]
        )


def _semantic_cache_path(
    texts,
    model_name
):
    h = hashlib.sha1()

    h.update(
        model_name.encode(
            'utf-8'
        )
    )

    h.update(
        str(
            len(texts)
        ).encode(
            'utf-8'
        )
    )

    for s in texts:
        h.update(
            safe_str(s)[:700].encode(
                'utf-8',
                errors='ignore'
            )
        )
        h.update(
            b'\0'
        )

    key = h.hexdigest()[
        :20
    ]

    return (
        SEMANTIC_CACHE_DIR
        / f"semantic_e5_{DATASET_NAME}_{key}.npy"
    )


def build_semantic_embeddings(
    texts
):
    texts = [
        safe_str(x)
        for x in texts
    ]

    if not USE_SEMANTIC_ENCODER:
        return np.zeros(
            (
                len(texts),
                SEMANTIC_FALLBACK_DIM
            ),
            dtype=np.float32
        )

    cache_path = _semantic_cache_path(
        texts,
        SEMANTIC_MODEL_NAME
    )

    if (
        SEMANTIC_CACHE
        and cache_path.exists()
    ):
        arr = np.load(
            cache_path
        )

        if len(arr) == len(texts):
            print(
                "semantic cache loaded:",
                cache_path,
                arr.shape
            )

            return arr.astype(
                np.float32
            )

    _ensure_sentence_transformers()

    from sentence_transformers import SentenceTransformer

    nonempty_idx = [
        i
        for i, s in enumerate(
            texts
        )
        if s
    ]

    # E5 retrieval family 권장 prefix.
    prefixed = [
        "passage: "
        + texts[i]
        for i in nonempty_idx
    ]

    semantic_device = (
        'cuda'
        if torch.cuda.is_available()
        else 'cpu'
    )

    encoder = None

    try:
        print(
            "loading semantic encoder:",
            SEMANTIC_MODEL_NAME,
            "device=",
            semantic_device
        )

        encoder = SentenceTransformer(
            SEMANTIC_MODEL_NAME,
            device=semantic_device
        )

        dim = int(
            encoder.get_sentence_embedding_dimension()
        )

        out = np.zeros(
            (
                len(texts),
                dim
            ),
            dtype=np.float32
        )

        if prefixed:
            emb = encoder.encode(
                prefixed,
                batch_size=SEMANTIC_BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True
            ).astype(
                np.float32
            )

            out[
                nonempty_idx
            ] = emb

    except RuntimeError as e:
        print(
            "semantic GPU encoding failed; retrying on CPU:",
            repr(e)
        )

        if encoder is not None:
            del encoder

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        encoder = SentenceTransformer(
            SEMANTIC_MODEL_NAME,
            device='cpu'
        )

        dim = int(
            encoder.get_sentence_embedding_dimension()
        )

        out = np.zeros(
            (
                len(texts),
                dim
            ),
            dtype=np.float32
        )

        if prefixed:
            emb = encoder.encode(
                prefixed,
                batch_size=max(
                    16,
                    SEMANTIC_BATCH_SIZE // 2
                ),
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True
            ).astype(
                np.float32
            )

            out[
                nonempty_idx
            ] = emb

    finally:
        if encoder is not None:
            del encoder

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if SEMANTIC_CACHE:
        np.save(
            cache_path,
            out
        )

        print(
            "semantic cache saved:",
            cache_path
        )

    return out


N = len(
    df
)

arrays = {}

arrays[
    'bytes'
] = np.stack(
    [
        byte_encode(
            x,
            MAX_TEXT_BYTES
        )
        for x in df[
            'event_text'
        ]
    ]
)

arrays[
    'semantic'
] = build_semantic_embeddings(
    df[
        'semantic_text'
    ].tolist()
)

SEMANTIC_DIM = int(
    arrays[
        'semantic'
    ].shape[
        1
    ]
)

for c in cat_cols:
    arrays[c] = np.array(
        [
            vocabs[c].enc(
                x
            )
            for x in df[c]
        ],
        dtype=np.int64
    )


for c in struct_cols:
    arrays[c]=np.array([struct_vocabs[c].enc(x) for x in df[c]],dtype=np.int64)

allnum = raw_num(
    df
)

arrays[
    'num'
] = (
    (
        allnum
        - mu
    )
    / sd
).astype(
    'float32'
)

print(
    "prepared events:",
    N
)

print(
    "byte array:",
    arrays[
        'bytes'
    ].shape
)

print(
    "semantic array:",
    arrays[
        'semantic'
    ].shape
)

print(
    "numeric array:",
    arrays[
        'num'
    ].shape
)


V6.1 full-data events loaded: 32,273 rows from /content/drive/MyDrive/Colab Notebooks/data/urangi/events.csv
events: (32273, 25)
tasks: (19843, 8)
sessions: (484, 11)
task columns: ['id', 'session_id', 'task_name', 'status', 'start_time', 'end_time', 'duration_ms', 'step_id']
session user mapping: 484 sessions key= id
trajectory_user_id coverage: 32,273/32,273 (100.0%)
structural vocab sizes: {'struct_api_family': 119, 'struct_page_path': 4, 'struct_page_title': 5}
loading semantic encoder: intfloat/multilingual-e5-small device= cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

/tmp/ipykernel_1271/1397509921.py:842: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  encoder.get_sentence_embedding_dimension()


Batches:   0%|          | 0/253 [00:00<?, ?it/s]

semantic cache saved: /content/drive/MyDrive/Colab Notebooks/data/urangi/semantic_cache/semantic_e5_poc_full_65550eebd78c9e0da573.npy
prepared events: 32273
byte array: (32273, 192)
semantic array: (32273, 384)
numeric array: (32273, 4)


## 3. Sliding windows / V4.2 Event Transformer + Semantic branch

In [4]:
def make_windows(frame,window,stride):
    out=[]
    for sid,g in frame.groupby('stream_id',sort=False):
        ids=g.sort_values('event_time_parsed')['global_event_idx'].to_numpy()
        if len(ids)==0: continue
        if len(ids)<=window:
            out.append((sid,ids))
        else:
            starts=list(range(0,len(ids)-window+1,stride))
            if starts[-1]!=len(ids)-window: starts.append(len(ids)-window)
            for st in starts: out.append((sid,ids[st:st+window]))
    return out

train_windows=make_windows(trdf,EVENT_WINDOW,EVENT_STRIDE)
valid_windows=make_windows(vdf,EVENT_WINDOW,EVENT_STRIDE)
full_windows=make_windows(df,EVENT_WINDOW,EVENT_STRIDE)

print('windows',len(train_windows),len(valid_windows),len(full_windows))


class EventWindowDataset(Dataset):
    def __init__(self,windows,arrays,mask_prob=0.0):
        self.windows=windows
        self.a=arrays
        self.mask_prob=mask_prob

    def __len__(self):
        return len(self.windows)

    def __getitem__(self,i):
        sid,ids=self.windows[i]
        L=len(ids)

        mask=np.zeros(L,dtype=bool)

        if self.mask_prob>0 and L>1:
            mask=np.random.rand(L)<self.mask_prob

            if not mask.any():
                mask[np.random.randint(0,L)]=True

        return {
            'stream_id':sid,
            'global_idx':torch.tensor(ids,dtype=torch.long),
            'bytes':torch.tensor(self.a['bytes'][ids],dtype=torch.long),
            'semantic':torch.tensor(self.a['semantic'][ids],dtype=torch.float32),
            **{
                c:torch.tensor(
                    self.a[c][ids],
                    dtype=torch.long
                )
                for c in cat_cols
            },
            **{
                c:torch.tensor(
                    self.a[c][ids],
                    dtype=torch.long
                )
                for c in struct_cols
            },
            'num':torch.tensor(
                self.a['num'][ids],
                dtype=torch.float32
            ),
            'mask_pos':torch.tensor(mask,dtype=torch.bool),
            'length':L
        }


def collate(batch):
    B=len(batch)
    L=max(x['length'] for x in batch)
    T=batch[0]['bytes'].shape[1]
    S=batch[0]['semantic'].shape[1]

    out={
        'stream_id':[x['stream_id'] for x in batch],
        'lengths':torch.tensor(
            [x['length'] for x in batch],
            dtype=torch.long
        )
    }

    out['global_idx']=torch.full((B,L),-1,dtype=torch.long)
    out['bytes']=torch.zeros((B,L,T),dtype=torch.long)
    out['semantic']=torch.zeros((B,L,S),dtype=torch.float32)
    out['num']=torch.zeros((B,L,4),dtype=torch.float32)
    out['mask_pos']=torch.zeros((B,L),dtype=torch.bool)
    out['attention_mask']=torch.zeros((B,L),dtype=torch.bool)

    for c in cat_cols:
        out[c]=torch.zeros((B,L),dtype=torch.long)

    for c in struct_cols:
        out[c]=torch.zeros((B,L),dtype=torch.long)

    for i,x in enumerate(batch):
        n=x['length']

        out['global_idx'][i,:n]=x['global_idx']
        out['bytes'][i,:n]=x['bytes']
        out['semantic'][i,:n]=x['semantic']
        out['num'][i,:n]=x['num']
        out['mask_pos'][i,:n]=x['mask_pos']
        out['attention_mask'][i,:n]=True

        for c in cat_cols:
            out[c][i,:n]=x[c]

        for c in struct_cols:
            out[c][i,:n]=x[c]

    return out


def todev(batch):
    return {
        k:(
            v.to(device)
            if torch.is_tensor(v)
            else v
        )
        for k,v in batch.items()
    }


class ByteCNN(nn.Module):
    def __init__(self,d_model):
        super().__init__()
        self.emb=nn.Embedding(257,32,padding_idx=0)
        self.conv=nn.Sequential(
            nn.Conv1d(32,64,3,padding=1),
            nn.GELU(),
            nn.Conv1d(64,d_model,5,padding=2),
            nn.GELU()
        )

    def forward(self,x):
        B,L,T=x.shape
        y=self.emb(
            x.reshape(B*L,T)
        ).transpose(1,2)

        y=self.conv(y)

        mask=x.reshape(
            B*L,T
        ).ne(0).unsqueeze(1)

        y=y.masked_fill(
            ~mask,
            -1e4
        ).amax(-1)

        empty=~mask.any(-1).squeeze(1)

        y=torch.where(
            empty.unsqueeze(-1),
            torch.zeros_like(y),
            y
        )

        return y.reshape(B,L,-1)


class EventTransformer(nn.Module):
    """
    V5.2 구조를 유지하되 semantic embedding 영향력을 축소한다.

    기존 V4.2:
        byte + numeric + categorical

    V5.3:
        byte + SEMANTIC_WEIGHT × semantic + numeric + categorical

    이후 Transformer / SSL heads 구조는 V4.2와 동일.
    """
    def __init__(self,vocabs,semantic_dim):
        super().__init__()

        self.text=ByteCNN(D_MODEL)

        self.semantic=nn.Sequential(
            nn.LayerNorm(semantic_dim),
            nn.Linear(
                semantic_dim,
                D_MODEL,
                bias=False
            )
        )

        self.cat=nn.ModuleDict(
            {
                c:nn.Embedding(
                    len(vocabs[c]),
                    D_MODEL,
                    padding_idx=0
                )
                for c in cat_cols
            }
        )

        self.num=nn.Sequential(
            nn.Linear(4,D_MODEL),
            nn.GELU(),
            nn.Linear(D_MODEL,D_MODEL)
        )

        self.pos=nn.Embedding(
            EVENT_WINDOW,
            D_MODEL
        )

        self.mask_token=nn.Parameter(
            torch.randn(D_MODEL)*.02
        )

        layer=nn.TransformerEncoderLayer(
            D_MODEL,
            NHEAD,
            FF,
            DROPOUT,
            batch_first=True,
            norm_first=True
        )

        self.tr=nn.TransformerEncoder(
            layer,
            EVENT_LAYERS
        )

        self.norm=nn.LayerNorm(
            D_MODEL
        )

        self.mask_event=nn.Linear(
            D_MODEL,
            len(vocabs['event_type'])
        )

        self.mask_int=nn.Linear(
            D_MODEL,
            len(vocabs['interaction_type'])
        )

        self.next_event=nn.Linear(
            D_MODEL,
            len(vocabs['event_type'])
        )

        self.next_int=nn.Linear(
            D_MODEL,
            len(vocabs['interaction_type'])
        )

    def forward(self,b,mask_events=True):
        # 핵심: V4.2 additive fusion을 유지하면서 semantic branch만 추가
        # V5.3:
        # semantic representation은 보조 신호로만 사용.
        # exact/technical identity(Byte/API/categorical)가 주 표현을 유지한다.
        x=(
            self.text(
                b['bytes']
            )
            + SEMANTIC_WEIGHT
            * self.semantic(
                b['semantic']
            )
            + self.num(
                b['num']
            )
        )

        for c in cat_cols:
            x=x+self.cat[c](b[c])

        if mask_events:
            x=torch.where(
                b['mask_pos'].unsqueeze(-1),
                self.mask_token.view(1,1,-1),
                x
            )

        L=x.shape[1]

        x=x+self.pos(
            torch.arange(
                L,
                device=x.device
            )
        ).view(1,L,-1)

        h=self.norm(
            self.tr(
                x,
                src_key_padding_mask=~b['attention_mask']
            )
        )

        return h

    def pooled(self,h,m):
        return (
            h*m.unsqueeze(-1)
        ).sum(1)/m.sum(
            1,
            keepdim=True
        ).clamp_min(1)


model=EventTransformer(
    vocabs,
    SEMANTIC_DIM
).to(
    device
)

print(model)


windows 407 114 507
EventTransformer(
  (text): ByteCNN(
    (emb): Embedding(257, 32, padding_idx=0)
    (conv): Sequential(
      (0): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): GELU(approximate='none')
      (2): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
      (3): GELU(approximate='none')
    )
  )
  (semantic): Sequential(
    (0): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=384, out_features=128, bias=False)
  )
  (cat): ModuleDict(
    (event_type): Embedding(4, 128, padding_idx=0)
    (interaction_type): Embedding(19, 128, padding_idx=0)
    (element_tag): Embedding(8, 128, padding_idx=0)
    (api_method): Embedding(3, 128, padding_idx=0)
  )
  (num): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (pos): Embedding(128, 128)
  (tr): TransformerEncoder(
    (layers): Modu

/tmp/ipykernel_1271/2444463755.py:227: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr=nn.TransformerEncoder(


## 4. Self-supervised Event Transformer training — V4.2 objective retained

In [5]:
def random_mask(att,p=.15):
    m=(torch.rand_like(att.float())<p)&att
    for i in range(m.shape[0]):
        if att[i].sum()>1 and not m[i].any(): m[i,torch.where(att[i])[0][0]]=True
    return m

def ntxent(z1,z2,temp=.2):
    z1=F.normalize(z1,dim=-1); z2=F.normalize(z2,dim=-1); B=z1.size(0)
    if B<2: return torch.tensor(0.,device=z1.device)
    z=torch.cat([z1,z2],0); sim=z@z.T/temp; eye=torch.eye(2*B,device=z.device,dtype=torch.bool); sim=sim.masked_fill(eye,-1e9)
    tgt=torch.arange(2*B,device=z.device); tgt=(tgt+B)%(2*B)
    return F.cross_entropy(sim,tgt)

def ssl_loss(model,b):
    orig_mask=b['mask_pos'].clone(); h1=model(b,True)
    mask=b['mask_pos']&b['attention_mask']; loss=torch.tensor(0.,device=device)
    if mask.any():
        loss=loss+F.cross_entropy(model.mask_event(h1)[mask],b['event_type'][mask])+.5*F.cross_entropy(model.mask_int(h1)[mask],b['interaction_type'][mask])
    valid=b['attention_mask'][:,:-1]&b['attention_mask'][:,1:]
    if valid.any():
        loss=loss+.5*F.cross_entropy(model.next_event(h1[:,:-1])[valid],b['event_type'][:,1:][valid])+.25*F.cross_entropy(model.next_int(h1[:,:-1])[valid],b['interaction_type'][:,1:][valid])
    b['mask_pos']=random_mask(b['attention_mask'],MASK_PROB); h2=model(b,True); b['mask_pos']=orig_mask
    loss=loss+.1*ntxent(model.pooled(h1,b['attention_mask']),model.pooled(h2,b['attention_mask']))
    return loss

train_loader=DataLoader(EventWindowDataset(train_windows,arrays,MASK_PROB),batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate)
valid_loader=DataLoader(EventWindowDataset(valid_windows,arrays,MASK_PROB),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate) if valid_windows else None

@torch.no_grad()
def eval_ssl(loader):
    if loader is None: return None
    model.eval(); vals=[]
    for b in loader:
        b=todev(b); vals.append(float(ssl_loss(model,b).item()))
    return float(np.mean(vals)) if vals else None

opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=0.01)
best=float('inf'); ssl_metrics=[]
for ep in range(1,EVENT_EPOCHS+1):
    model.train(); ls=[]
    for b in train_loader:
        b=todev(b); loss=ssl_loss(model,b); opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step(); ls.append(loss.item())
    tr=float(np.mean(ls)) if ls else float('nan')
    va=eval_ssl(valid_loader)
    score=va if va is not None else tr
    print(f'event_ssl epoch={ep:02d} train={tr:.4f} valid={va}')
    ssl_metrics.append({'epoch':ep,'train_loss':tr,'valid_loss':va})
    if score < best:
        best=score
        torch.save(
            {
                'model_state_dict':model.state_dict(),
                'best_score':best,
                'model_version':MODEL_VERSION,
                'semantic_model_name':SEMANTIC_MODEL_NAME,
                'semantic_dim':SEMANTIC_DIM,
                'semantic_weight':SEMANTIC_WEIGHT,
                'controlled_ablation':'V5.2 + semantic weight 0.20 + post-hoc gold evaluation'
            },
            OUTPUT_DIR/'event_transformer.pt'
        )
pd.DataFrame(ssl_metrics).to_csv(OUTPUT_DIR/'event_ssl_metrics.csv',index=False,encoding='utf-8-sig')
ckpt=torch.load(OUTPUT_DIR/'event_transformer.pt',map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print('restored best event model:',ckpt['best_score'])


event_ssl epoch=01 train=2.5115 valid=2.069085583090782
event_ssl epoch=02 train=1.9836 valid=1.9874114096164703
event_ssl epoch=03 train=1.8274 valid=1.7828194499015808
event_ssl epoch=04 train=1.7387 valid=1.765583261847496
event_ssl epoch=05 train=1.6782 valid=1.6748241037130356
event_ssl epoch=06 train=1.6531 valid=1.678338572382927
event_ssl epoch=07 train=1.6093 valid=1.6659545451402664
event_ssl epoch=08 train=1.5650 valid=1.5732783079147339
event_ssl epoch=09 train=1.5589 valid=1.6013760566711426
event_ssl epoch=10 train=1.5472 valid=1.5622435957193375
event_ssl epoch=11 train=1.5338 valid=1.5074570178985596
event_ssl epoch=12 train=1.5035 valid=1.539689153432846
restored best event model: 1.5074570178985596


## 5. Neural event-boundary detector

---



In [6]:
class StructuralIdentityEncoder(nn.Module):
    def __init__(self,struct_vocabs,d_struct=STRUCT_DIM):
        super().__init__()
        self.emb=nn.ModuleDict({
            c:nn.Embedding(len(struct_vocabs[c]),d_struct,padding_idx=0)
            for c in struct_cols
        })
        self.norm=nn.LayerNorm(d_struct)

    def forward(self,b):
        vecs=[]; masks=[]
        for c in struct_cols:
            ids=b[c]
            vecs.append(self.emb[c](ids))
            masks.append(ids.ne(0).float())
        stacked=torch.stack(vecs,dim=-2)
        present=torch.stack(masks,dim=-1)
        denom=present.sum(-1,keepdim=True).clamp_min(1.0)
        pooled=(stacked*present.unsqueeze(-1)).sum(-2)/denom
        pooled=self.norm(pooled)
        any_present=present.sum(-1).gt(0)
        return torch.where(any_present.unsqueeze(-1),pooled,torch.zeros_like(pooled))


class NeuralBoundary(nn.Module):
    def __init__(self,d,struct_vocabs):
        super().__init__()
        self.context_conv=nn.Sequential(
            nn.Conv1d(d,d,3,padding=1),nn.GELU(),
            nn.Conv1d(d,d,3,padding=1),nn.GELU()
        )
        self.struct_encoder=StructuralIdentityEncoder(struct_vocabs,STRUCT_DIM)
        self.struct_conv=nn.Sequential(
            nn.Conv1d(STRUCT_DIM,STRUCT_DIM,3,padding=1),nn.GELU(),
            nn.Conv1d(STRUCT_DIM,STRUCT_DIM,3,padding=1),nn.GELU()
        )
        feat_dim=d*3+STRUCT_DIM*3+len(struct_cols)
        self.mlp=nn.Sequential(
            nn.Linear(feat_dim,d),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(d,1)
        )

    @staticmethod
    def raw_structural_change_features(b):
        changes=[]
        for c in struct_cols:
            left=b[c][:,:-1]; right=b[c][:,1:]
            available=left.ne(0)|right.ne(0)
            changes.append((left.ne(right)&available).float())
        return torch.stack(changes,dim=-1)

    @staticmethod
    def structural_change_score(b):
        feat=NeuralBoundary.raw_structural_change_features(b)
        available=[]
        for c in struct_cols:
            left=b[c][:,:-1]; right=b[c][:,1:]
            available.append((left.ne(0)|right.ne(0)).float())
        avail=torch.stack(available,dim=-1)
        return feat.sum(-1)/avail.sum(-1).clamp_min(1.0)

    def forward(self,h,b):
        c=self.context_conv(h.transpose(1,2)).transpose(1,2)
        s=self.struct_encoder(b)
        s=self.struct_conv(s.transpose(1,2)).transpose(1,2)
        a,bb=c[:,:-1],c[:,1:]
        sa,sb=s[:,:-1],s[:,1:]
        raw=self.raw_structural_change_features(b)
        feat=torch.cat([a,bb,torch.abs(a-bb),sa,sb,torch.abs(sa-sb),raw],-1)
        return self.mlp(feat).squeeze(-1)


def contextual_multiscale_target(h,mask,k=4):
    B,L,D=h.shape
    out=torch.zeros(B,L-1,device=h.device)
    valid=torch.zeros(B,L-1,dtype=torch.bool,device=h.device)
    for t in range(L-1):
        ls=max(0,t-k+1); re=min(L,t+1+k)
        lm=mask[:,ls:t+1].float(); rm=mask[:,t+1:re].float()
        lv=(h[:,ls:t+1]*lm.unsqueeze(-1)).sum(1)/lm.sum(1,keepdim=True).clamp_min(1)
        rv=(h[:,t+1:re]*rm.unsqueeze(-1)).sum(1)/rm.sum(1,keepdim=True).clamp_min(1)
        out[:,t]=1-F.cosine_similarity(lv,rv,dim=-1)
        valid[:,t]=mask[:,t]&mask[:,t+1]&lm.sum(1).gt(0)&rm.sum(1).gt(0)
    vals=out[valid]
    med=vals.median() if vals.numel() else torch.tensor(0.,device=h.device)
    mad=(vals-med).abs().median().clamp_min(1e-3) if vals.numel() else torch.tensor(1.,device=h.device)
    soft=torch.sigmoid((out-med)/(1.4826*mad))
    return soft,valid


def structural_aware_target(h,mask,b,k=4):
    contextual_soft,valid=contextual_multiscale_target(h,mask,k)
    if not USE_STRUCTURAL_BOUNDARY:
        return contextual_soft,valid,torch.zeros_like(contextual_soft)
    structural_score=NeuralBoundary.structural_change_score(b)
    target=(contextual_soft+STRUCT_TARGET_WEIGHT*structural_score*(1-contextual_soft)).clamp(0,1)
    return target,valid,structural_score


bd=NeuralBoundary(D_MODEL,struct_vocabs).to(device)
bopt=torch.optim.AdamW(bd.parameters(),lr=8e-4,weight_decay=0.01)
model.eval(); bd_metrics=[]
for ep in range(1,BOUNDARY_EPOCHS+1):
    bd.train(); ls=[]; sm=[]; tm=[]
    for b in train_loader:
        b=todev(b); b['mask_pos'].zero_()
        with torch.no_grad():
            h=model(b,False)
            target,valid,struct_score=structural_aware_target(h,b['attention_mask'],b,k=4)
        logits=bd(h,b)
        if valid.any():
            pred=torch.sigmoid(logits[valid])
            loss=F.binary_cross_entropy_with_logits(logits[valid],target[valid]) + STRUCT_BOUNDARY_SPARSITY_WEIGHT*pred.mean()
            bopt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(bd.parameters(),1.0)
            bopt.step(); ls.append(loss.item())
            sm.append(float(struct_score[valid].mean().item()))
            tm.append(float(target[valid].mean().item()))
    v=float(np.mean(ls)) if ls else float('nan')
    sv=float(np.mean(sm)) if sm else float('nan')
    tv=float(np.mean(tm)) if tm else float('nan')
    print(f'event_boundary epoch={ep:02d} loss={v:.4f} struct_mean={sv:.4f} target_mean={tv:.4f}')
    bd_metrics.append({'epoch':ep,'loss':v,'structural_change_mean':sv,'target_mean':tv})

pd.DataFrame(bd_metrics).to_csv(OUTPUT_DIR/'event_boundary_metrics.csv',index=False,encoding='utf-8-sig')
torch.save({
    'boundary_state_dict':bd.state_dict(),
    'structural_fields':struct_cols,
    'structural_vocab':{c:struct_vocabs[c].itos for c in struct_cols},
    'struct_dim':STRUCT_DIM,
    'struct_target_weight':STRUCT_TARGET_WEIGHT,
    'use_structural_boundary':USE_STRUCTURAL_BOUNDARY,
},OUTPUT_DIR/'event_boundary_model.pt')


event_boundary epoch=01 loss=0.6728 struct_mean=0.1352 target_mean=0.5546
event_boundary epoch=02 loss=0.6448 struct_mean=0.1353 target_mean=0.5532
event_boundary epoch=03 loss=0.6282 struct_mean=0.1364 target_mean=0.5549
event_boundary epoch=04 loss=0.6254 struct_mean=0.1374 target_mean=0.5541
event_boundary epoch=05 loss=0.6227 struct_mean=0.1357 target_mean=0.5540
event_boundary epoch=06 loss=0.6237 struct_mean=0.1360 target_mean=0.5547


## 6. Global event embeddings / latent business segments

In [7]:
# global inference
full_loader=DataLoader(EventWindowDataset(full_windows,arrays,0),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate)
emb_sum=np.zeros((N,D_MODEL),np.float32); emb_cnt=np.zeros(N,np.float32)
prob_sum=np.zeros(N,np.float32); prob_cnt=np.zeros(N,np.float32)
struct_sum=np.zeros(N,np.float32); struct_cnt=np.zeros(N,np.float32)
model.eval(); bd.eval()
with torch.no_grad():
    for b in full_loader:
        b=todev(b); h=model(b,False)
        p=torch.sigmoid(bd(h,b)).cpu().numpy()
        ss=bd.structural_change_score(b).cpu().numpy()
        hh=h.cpu().numpy(); gids=b['global_idx'].cpu().numpy(); lens=b['lengths'].cpu().numpy()
        for bi,n in enumerate(lens):
            ids=gids[bi,:n]
            emb_sum[ids]+=hh[bi,:n]; emb_cnt[ids]+=1
            for t in range(n-1):
                gi=ids[t+1]
                prob_sum[gi]+=p[bi,t]; prob_cnt[gi]+=1
                struct_sum[gi]+=ss[bi,t]; struct_cnt[gi]+=1
emb=emb_sum/np.maximum(emb_cnt[:,None],1)
probs=prob_sum/np.maximum(prob_cnt,1); probs[prob_cnt==0]=0
structural_scores=struct_sum/np.maximum(struct_cnt,1); structural_scores[struct_cnt==0]=0

np.save(OUTPUT_DIR/'event_embeddings.npy',emb)

# ============================================================
# V5.4 representation + structural-boundary diagnostics
# - training / assignment에는 사용하지 않음
# ============================================================
def sampled_pair_cosine(
    vectors,
    n_pairs=5000,
    seed=42
):
    if len(vectors) < 2:
        return float('nan')

    rng=np.random.default_rng(seed)

    n=min(
        n_pairs,
        max(
            len(vectors)*2,
            100
        )
    )

    a=rng.integers(
        0,
        len(vectors),
        size=n
    )

    b=rng.integers(
        0,
        len(vectors),
        size=n
    )

    keep=a!=b

    a=a[keep]
    b=b[keep]

    va=vectors[a]
    vb=vectors[b]

    denom=(
        np.linalg.norm(
            va,
            axis=1
        )
        * np.linalg.norm(
            vb,
            axis=1
        )
        + 1e-9
    )

    return float(
        np.mean(
            (
                va*vb
            ).sum(
                axis=1
            )
            / denom
        )
    )


event_embedding_cosine_mean=sampled_pair_cosine(
    emb
)

print(
    'event embedding cosine mean:',
    event_embedding_cosine_mean
)


# ============================================================
# V5.7 calibrated boundary threshold
# ============================================================

def threshold_gmm_legacy(vals):
    """
    V5.3~V5.6 비교용 기존 threshold.
    최종 segmentation에는 사용하지 않는다.
    """
    vals=np.asarray(vals,float)
    vals=vals[np.isfinite(vals)]

    if len(vals)<8 or np.std(vals)<1e-6:
        return (
            float(
                np.median(vals)+np.std(vals)
            )
            if len(vals)
            else .5
        )

    try:
        gm=GaussianMixture(
            2,
            random_state=SEED
        ).fit(
            vals.reshape(-1,1)
        )

        means=np.sort(
            gm.means_.ravel()
        )

        return float(
            means.mean()
        )

    except Exception:
        return float(
            np.quantile(
                vals,
                .85
            )
        )


def _robust_center_scale(vals):
    vals=np.asarray(
        vals,
        dtype=float
    )

    vals=vals[
        np.isfinite(
            vals
        )
    ]

    if len(
        vals
    ) == 0:
        return (
            .0,
            1.0
        )

    med=float(
        np.median(
            vals
        )
    )

    mad=float(
        np.median(
            np.abs(
                vals-med
            )
        )
    )

    scale=max(
        1.4826*mad,
        float(
            np.std(
                vals
            )
        )*.25,
        1e-6
    )

    return (
        med,
        scale
    )


def threshold_calibrated(
    vals,
    return_meta=False
):
    """
    1-component vs 2-component GMM을 비교하고,
    충분히 분리된 boundary component가 있을 때만
    high-component posterior를 cut 근거로 사용한다.

    분리 구조가 약하면 median-relative quantile이 아니라
    robust absolute upper-tail threshold로 fallback한다.
    """
    vals=np.asarray(
        vals,
        dtype=float
    )

    vals=vals[
        np.isfinite(
            vals
        )
    ]

    meta={
        'n':
            int(
                len(
                    vals
                )
            ),
        'mode':
            'empty',
        'threshold':
            .5,
        'legacy_threshold':
            None,
        'bic_1':
            None,
        'bic_2':
            None,
        'bic_gain':
            None,
        'low_mean':
            None,
        'high_mean':
            None,
        'separation_sigma':
            None,
        'posterior_min':
            float(
                THRESHOLD_POSTERIOR_MIN
            ),
    }

    if len(
        vals
    ) == 0:
        return (
            (
                .5,
                meta
            )
            if return_meta
            else .5
        )

    legacy=threshold_gmm_legacy(
        vals
    )

    meta[
        'legacy_threshold'
    ]=float(
        legacy
    )

    med,scale=_robust_center_scale(
        vals
    )

    robust_threshold=float(
        med
        + THRESHOLD_ROBUST_Z_FALLBACK
        * scale
    )

    if (
        len(
            vals
        ) < 8
        or np.std(
            vals
        ) < 1e-6
    ):
        threshold=robust_threshold

        meta.update(
            {
                'mode':
                    'robust_tail_small_n',
                'threshold':
                    float(
                        threshold
                    ),
            }
        )

        return (
            (
                threshold,
                meta
            )
            if return_meta
            else threshold
        )

    try:
        gm1=GaussianMixture(
            1,
            random_state=SEED
        ).fit(
            vals.reshape(
                -1,
                1
            )
        )

        gm2=GaussianMixture(
            2,
            random_state=SEED
        ).fit(
            vals.reshape(
                -1,
                1
            )
        )

        bic1=float(
            gm1.bic(
                vals.reshape(
                    -1,
                    1
                )
            )
        )

        bic2=float(
            gm2.bic(
                vals.reshape(
                    -1,
                    1
                )
            )
        )

        bic_gain=(
            bic1
            - bic2
        )

        means=gm2.means_.ravel()

        order=np.argsort(
            means
        )

        low_idx=int(
            order[
                0
            ]
        )

        high_idx=int(
            order[
                1
            ]
        )

        low_mean=float(
            means[
                low_idx
            ]
        )

        high_mean=float(
            means[
                high_idx
            ]
        )

        cov=np.asarray(
            gm2.covariances_
        ).reshape(
            -1
        )

        pooled_sd=math.sqrt(
            max(
                (
                    float(
                        cov[
                            low_idx
                        ]
                    )
                    + float(
                        cov[
                            high_idx
                        ]
                    )
                )
                / 2.0,
                1e-8
            )
        )

        separation_sigma=(
            abs(
                high_mean
                - low_mean
            )
            / pooled_sd
        )

        meta.update(
            {
                'bic_1':
                    bic1,
                'bic_2':
                    bic2,
                'bic_gain':
                    float(
                        bic_gain
                    ),
                'low_mean':
                    low_mean,
                'high_mean':
                    high_mean,
                'separation_sigma':
                    float(
                        separation_sigma
                    ),
            }
        )

        if (
            bic_gain
            >= THRESHOLD_BIC_GAIN_MIN
            and separation_sigma
            >= THRESHOLD_MIN_SEPARATION_SIGMA
        ):
            posterior=gm2.predict_proba(
                vals.reshape(
                    -1,
                    1
                )
            )[
                :,
                high_idx
            ]

            accepted=vals[
                posterior
                >= THRESHOLD_POSTERIOR_MIN
            ]

            if len(
                accepted
            ):
                threshold=float(
                    np.min(
                        accepted
                    )
                )

                meta[
                    'mode'
                ]='gmm_high_posterior'

            else:
                threshold=robust_threshold
                meta[
                    'mode'
                ]='robust_tail_no_posterior'

        else:
            threshold=robust_threshold
            meta[
                'mode'
            ]='robust_tail_weak_mixture'

    except Exception as e:
        threshold=robust_threshold

        meta[
            'mode'
        ]='robust_tail_exception'

        meta[
            'exception'
        ]=safe_str(
            e
        )

    meta[
        'threshold'
    ]=float(
        threshold
    )

    return (
        (
            threshold,
            meta
        )
        if return_meta
        else threshold
    )


def threshold_gmm(vals):
    """
    Existing call sites compatibility wrapper.
    V5.7 final segmentation uses calibrated threshold.
    """
    return threshold_calibrated(
        vals,
        return_meta=False
    )

def cosine(a,b):
    na=np.linalg.norm(a); nb=np.linalg.norm(b); return float(a@b/(na*nb+1e-9))

event_threshold_rows=[]

def make_segments(df,emb,probs):
    segrows=[]
    members=[]
    sid_counter=0

    for stream,g in df.groupby(
        'stream_id',
        sort=False
    ):
        g=g.sort_values(
            'event_time_parsed'
        )

        ids=g.global_event_idx.to_numpy()

        vals=(
            probs[
                ids
            ][
                1:
            ]
            if len(
                ids
            ) > 1
            else np.asarray(
                []
            )
        )

        th,meta=threshold_calibrated(
            vals,
            return_meta=True
        )

        legacy_th=(
            threshold_gmm_legacy(
                vals
            )
            if len(
                vals
            )
            else 1.0
        )

        raw_starts=[0]+[
            i
            for i in range(
                1,
                len(
                    ids
                )
            )
            if probs[
                ids[
                    i
                ]
            ]>=th
        ]

        legacy_starts=[0]+[
            i
            for i in range(
                1,
                len(
                    ids
                )
            )
            if probs[
                ids[
                    i
                ]
            ]>=legacy_th
        ]

        # enforce min by skipping too-close starts
        filt=[0]

        for s in raw_starts[
            1:
        ]:
            if (
                s-filt[
                    -1
                ]
                >=MIN_SEGMENT_EVENTS
            ):
                filt.append(
                    s
                )

        if (
            len(
                ids
            )
            - filt[
                -1
            ]
            < MIN_SEGMENT_EVENTS
            and len(
                filt
            ) > 1
        ):
            filt.pop()

        legacy_filt=[0]

        for s in legacy_starts[
            1:
        ]:
            if (
                s-legacy_filt[
                    -1
                ]
                >=MIN_SEGMENT_EVENTS
            ):
                legacy_filt.append(
                    s
                )

        if (
            len(
                ids
            )
            - legacy_filt[
                -1
            ]
            < MIN_SEGMENT_EVENTS
            and len(
                legacy_filt
            ) > 1
        ):
            legacy_filt.pop()

        event_threshold_rows.append(
            {
                'stream_id':
                    safe_str(
                        stream
                    ),
                'event_count':
                    int(
                        len(
                            ids
                        )
                    ),
                'legacy_threshold':
                    float(
                        legacy_th
                    ),
                'calibrated_threshold':
                    float(
                        th
                    ),
                'legacy_segment_count':
                    int(
                        len(
                            legacy_filt
                        )
                    ),
                'calibrated_segment_count':
                    int(
                        len(
                            filt
                        )
                    ),
                **meta,
            }
        )

        for j,s in enumerate(
            filt
        ):
            e=(
                filt[
                    j+1
                ]-1
                if j+1<len(
                    filt
                )
                else len(
                    ids
                )-1
            )

            mids=ids[
                s:e+1
            ]

            sid_counter+=1

            txt=g.iloc[
                s:e+1
            ]

            evidence=[]

            for col in [
                'element_text',
                'associated_label',
                'api_path',
                'interaction_type'
            ]:
                for v in (
                    txt[
                        col
                    ].tolist()
                    if col in txt
                    else []
                ):
                    v=safe_str(
                        v
                    )

                    if (
                        v
                        and v not in evidence
                    ):
                        evidence.append(
                            v
                        )

                    if len(
                        evidence
                    )>=8:
                        break

                if len(
                    evidence
                )>=8:
                    break

            segrows.append(
                {
                    'segment_idx':
                        sid_counter-1,
                    'segment_id':
                        f'S{sid_counter:05d}',
                    'stream_id':
                        stream,
                    'start_global_idx':
                        int(
                            mids[
                                0
                            ]
                        ),
                    'end_global_idx':
                        int(
                            mids[
                                -1
                            ]
                        ),
                    'event_count':
                        len(
                            mids
                        ),
                    'start_time':
                        txt.event_time_parsed.iloc[
                            0
                        ],
                    'end_time':
                        txt.event_time_parsed.iloc[
                            -1
                        ],
                    'boundary_threshold':
                        th,
                    'boundary_prob_start':
                        (
                            1.0
                            if s==0
                            else float(
                                probs[
                                    mids[
                                        0
                                    ]
                                ]
                            )
                        ),
                    'evidence':
                        ' | '.join(
                            evidence
                        )
                }
            )

            members.append(
                mids
            )

    return (
        pd.DataFrame(
            segrows
        ),
        members
    )


segdf,members=make_segments(
    df,
    emb,
    probs
)

pd.DataFrame(
    event_threshold_rows
).to_csv(
    DIAGNOSTICS_DIR
    / 'event_threshold_diagnostics.csv',
    index=False,
    encoding='utf-8-sig'
)
print('segments',len(segdf)); print(segdf.head())
seg_base=np.stack([emb[m].mean(0) for m in members]) if members else np.empty((0,D_MODEL))
segdf.to_csv(OUTPUT_DIR/'latent_segments.csv',index=False,encoding='utf-8-sig')
np.save(OUTPUT_DIR/'base_segment_embeddings.npy',seg_base)
event_boundary_export=df[['global_event_idx','stream_id','event_time_parsed','event_type','interaction_type','element_text','associated_label','api_path']].copy()
event_boundary_export['boundary_prob_before']=probs
event_boundary_export['structural_change_score_before']=structural_scores
event_boundary_export.to_csv(OUTPUT_DIR/'event_boundary_scores.csv',index=False,encoding='utf-8-sig')

_valid_struct=structural_scores[struct_cnt>0]
structural_boundary_summary={
    'enabled':bool(USE_STRUCTURAL_BOUNDARY),
    'fields':list(struct_cols),
    'struct_dim':int(STRUCT_DIM),
    'target_weight':float(STRUCT_TARGET_WEIGHT),
    'mean_structural_change_score':float(np.mean(_valid_struct)) if len(_valid_struct) else None,
    'p50_structural_change_score':float(np.quantile(_valid_struct,.50)) if len(_valid_struct) else None,
    'p90_structural_change_score':float(np.quantile(_valid_struct,.90)) if len(_valid_struct) else None,
    'mean_boundary_probability':float(np.mean(probs[prob_cnt>0])) if np.any(prob_cnt>0) else None,
}
print('structural boundary summary:',structural_boundary_summary)


event embedding cosine mean: 0.22350436449050903
segments 5167
   segment_idx segment_id                                          stream_id  \
0            0     S00001  login:8aca60cbf9848351f929b026e6f9d2b7b502c0cc...   
1            1     S00002  login:8aca60cbf9848351f929b026e6f9d2b7b502c0cc...   
2            2     S00003  login:8aca60cbf9848351f929b026e6f9d2b7b502c0cc...   
3            3     S00004  login:8aca60cbf9848351f929b026e6f9d2b7b502c0cc...   
4            4     S00005  login:8aca60cbf9848351f929b026e6f9d2b7b502c0cc...   

   start_global_idx  end_global_idx  event_count              start_time  \
0                 0               5            6 2026-08-03 13:54:14.245   
1                 6               8            3 2026-08-03 13:54:26.446   
2                 9              41           33 2026-08-03 13:54:26.470   
3                42              44            3 2026-08-03 13:56:08.630   
4                45              55           11 2026-08-03 13:56:21.785   


## 7. Hierarchical Segment Transformer / workflow boundary

V5.6은 이 단계에서 **V5.4의 Segment Transformer와 workflow mean pooling을 유지**합니다.

변경되는 것은 workflow boundary signal 하나입니다.

```text
V5.4 contextual boundary probability
              +
transition contrastive surprise
              ↓
combined workflow boundary score
```

Transition model은 실제 인접 segment sequence만 이용해
“현재 segment 다음에 어떤 segment representation이 오는가”를 self-supervised하게 학습합니다.

6개 실제 업무명은 이 학습에 사용하지 않습니다.

In [8]:
class SegmentSeqDataset(Dataset):
    def __init__(self, segdf, vectors, window, stride, mask_prob=.15):
        self.v = vectors
        self.items = []
        self.mask_prob = mask_prob

        for sid, g in segdf.groupby('stream_id', sort=False):
            idx = (
                g.sort_values('start_global_idx')
                .segment_idx
                .to_numpy()
            )

            if len(idx) <= window:
                self.items.append((sid, idx))
            else:
                starts = list(
                    range(
                        0,
                        len(idx) - window + 1,
                        stride
                    )
                )

                if starts[-1] != len(idx) - window:
                    starts.append(len(idx) - window)

                for st in starts:
                    self.items.append(
                        (sid, idx[st:st + window])
                    )

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        sid, ids = self.items[i]
        L = len(ids)

        mask = (
            np.random.rand(L) < self.mask_prob
            if self.mask_prob
            else np.zeros(L, bool)
        )

        if self.mask_prob and L > 1 and not mask.any():
            mask[np.random.randint(L)] = True

        return {
            'stream_id': sid,
            'segment_idx': torch.tensor(
                ids,
                dtype=torch.long
            ),
            'vectors': torch.tensor(
                self.v[ids],
                dtype=torch.float32
            ),
            'mask_pos': torch.tensor(
                mask,
                dtype=torch.bool
            ),
            'length': L
        }


def seg_collate(batch):
    B = len(batch)
    L = max(x['length'] for x in batch)
    D = batch[0]['vectors'].shape[-1]

    out = {
        'stream_id': [
            x['stream_id']
            for x in batch
        ],

        'lengths': torch.tensor(
            [
                x['length']
                for x in batch
            ],
            dtype=torch.long
        ),

        'segment_idx': torch.full(
            (B, L),
            -1,
            dtype=torch.long
        ),

        'vectors': torch.zeros(
            B,
            L,
            D,
            dtype=torch.float32
        ),

        'mask_pos': torch.zeros(
            B,
            L,
            dtype=torch.bool
        ),

        'attention_mask': torch.zeros(
            B,
            L,
            dtype=torch.bool
        )
    }

    for i, x in enumerate(batch):
        n = x['length']

        out['segment_idx'][i, :n] = x['segment_idx']
        out['vectors'][i, :n] = x['vectors']
        out['mask_pos'][i, :n] = x['mask_pos']
        out['attention_mask'][i, :n] = True

    return out


class SegmentTransformer(nn.Module):
    def __init__(self, d):
        super().__init__()

        self.inp = nn.Linear(d, d)

        self.mask = nn.Parameter(
            torch.randn(d) * .02
        )

        self.pos = nn.Embedding(
            SEGMENT_WINDOW,
            d
        )

        layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=NHEAD,
            dim_feedforward=FF,
            dropout=DROPOUT,
            batch_first=True,
            norm_first=True
        )

        self.tr = nn.TransformerEncoder(
            layer,
            num_layers=2
        )

        self.norm = nn.LayerNorm(d)

        self.recon = nn.Linear(
            d,
            d
        )

        self.next = nn.Linear(
            d,
            d
        )

    def forward(self, b, mask=True):
        x = self.inp(
            b['vectors']
        )

        if mask:
            x = torch.where(
                b['mask_pos'].unsqueeze(-1),
                self.mask.view(1, 1, -1),
                x
            )

        L = x.size(1)

        pos = torch.arange(
            L,
            device=x.device
        )

        x = x + self.pos(pos)

        x = self.tr(
            x,
            src_key_padding_mask=~b['attention_mask']
        )

        return self.norm(x)


# ============================================================
# Segment Transformer 초기화
# ============================================================

sm = None
sh = seg_base.copy()
wbd = None

wprobs = np.zeros(
    len(segdf),
    dtype=np.float32
)

workflow_df = pd.DataFrame()

workflow_members = []

workflow_embeddings = np.empty(
    (0, D_MODEL),
    dtype=np.float32
)


workflow_contextual_probs = np.zeros(
    len(segdf),
    dtype=np.float32
)

workflow_transition_scores = np.zeros(
    len(segdf),
    dtype=np.float32
)

workflow_transition_raw_surprise = np.zeros(
    len(segdf),
    dtype=np.float32
)

workflow_transition_diagnostics = {
    'enabled':
        bool(
            USE_TRANSITION_CONTRASTIVE_WORKFLOW_BOUNDARY
        ),
    'pair_count':
        0,
    'blend_weight':
        float(
            WORKFLOW_TRANSITION_BLEND_WEIGHT
        ),
    'temperature':
        float(
            WORKFLOW_TRANSITION_TEMPERATURE
        ),
    'epochs':
        int(
            WORKFLOW_TRANSITION_EPOCHS
        ),
}


# ============================================================
# Segment-level Transformer 학습
# ============================================================

if len(segdf) >= 2:

    sds = SegmentSeqDataset(
        segdf,
        seg_base,
        SEGMENT_WINDOW,
        SEGMENT_STRIDE,
        MASK_PROB
    )

    sl = DataLoader(
        sds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=seg_collate
    )

    sm = SegmentTransformer(
        D_MODEL
    ).to(device)

    sopt = torch.optim.AdamW(
        sm.parameters(),
        lr=3e-4
    )

    # ========================================================
    # Segment SSL Training
    # ========================================================

    for ep in range(SEGMENT_EPOCHS):

        ls = []

        sm.train()

        for b in sl:

            b = todev(b)

            h = sm(
                b,
                True
            )

            mask = (
                b['mask_pos']
                & b['attention_mask']
            )

            valid = (
                b['attention_mask'][:, :-1]
                & b['attention_mask'][:, 1:]
            )

            loss_terms = []

            # ------------------------------------------------
            # 1. Masked segment reconstruction
            # ------------------------------------------------

            if mask.any():

                recon_pred = sm.recon(h)[mask]

                recon_target = (
                    b['vectors'][mask]
                )

                recon_loss = (
                    1
                    - F.cosine_similarity(
                        recon_pred,
                        recon_target,
                        dim=-1
                    )
                ).mean()

                loss_terms.append(
                    recon_loss
                )

            # ------------------------------------------------
            # 2. Next segment prediction
            # ------------------------------------------------

            if valid.any():

                next_pred = (
                    sm.next(
                        h[:, :-1]
                    )[valid]
                )

                next_target = (
                    b['vectors'][:, 1:][valid]
                )

                next_loss = (
                    1
                    - F.cosine_similarity(
                        next_pred,
                        next_target,
                        dim=-1
                    )
                ).mean()

                loss_terms.append(
                    0.5 * next_loss
                )

            # ------------------------------------------------
            # loss가 없는 batch는 skip
            # ------------------------------------------------

            if not loss_terms:
                continue

            loss = torch.stack(
                loss_terms
            ).sum()

            sopt.zero_grad()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                sm.parameters(),
                max_norm=1.0
            )

            sopt.step()

            ls.append(
                loss.item()
            )

        mean_loss = (
            float(np.mean(ls))
            if ls
            else float('nan')
        )

        print(
            f"segssl epoch={ep + 1:02d} "
            f"loss={mean_loss:.4f} "
            f"batches={len(ls)}"
        )


    # ========================================================
    # Segment Transformer 저장
    # ========================================================

    torch.save(
        {
            'segment_model_state_dict':
                sm.state_dict()
        },
        OUTPUT_DIR / 'segment_transformer.pt'
    )


    # ========================================================
    # Segment Hidden Representation 추론
    # ========================================================

    infds = SegmentSeqDataset(
        segdf,
        seg_base,
        SEGMENT_WINDOW,
        SEGMENT_STRIDE,
        0
    )

    infl = DataLoader(
        infds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=seg_collate
    )

    sh_sum = np.zeros(
        (
            len(segdf),
            D_MODEL
        ),
        dtype=np.float32
    )

    sh_cnt = np.zeros(
        len(segdf),
        dtype=np.float32
    )

    sm.eval()

    with torch.no_grad():

        for b in infl:

            b = todev(b)

            h = (
                sm(
                    b,
                    False
                )
                .cpu()
                .numpy()
            )

            ids = (
                b['segment_idx']
                .cpu()
                .numpy()
            )

            lens = (
                b['lengths']
                .cpu()
                .numpy()
            )

            for bi, n in enumerate(lens):

                valid_ids = ids[
                    bi,
                    :n
                ]

                sh_sum[
                    valid_ids
                ] += h[
                    bi,
                    :n
                ]

                sh_cnt[
                    valid_ids
                ] += 1


    sh = (
        sh_sum
        / np.maximum(
            sh_cnt[:, None],
            1
        )
    )

    np.save(
        OUTPUT_DIR
        / 'segment_embeddings.npy',
        sh
    )


    # ========================================================
    # Workflow Boundary Detector
    # ========================================================

    class WorkflowBoundary(nn.Module):
        """
        Workflow-level boundary detector.

        V5.4 structural identity is intentionally applied ONLY to the event boundary.
        At this level the inputs are latent segment embeddings, so we keep the
        original V5.3/V4.2 contextual boundary architecture unchanged.
        """
        def __init__(self, d):
            super().__init__()

            self.conv = nn.Sequential(
                nn.Conv1d(
                    d,
                    d,
                    3,
                    padding=1
                ),
                nn.GELU(),
                nn.Conv1d(
                    d,
                    d,
                    3,
                    padding=1
                ),
                nn.GELU()
            )

            self.mlp = nn.Sequential(
                nn.Linear(
                    d * 3,
                    d
                ),
                nn.GELU(),
                nn.Linear(
                    d,
                    1
                )
            )

        def forward(self, h):
            c = self.conv(
                h.transpose(
                    1,
                    2
                )
            ).transpose(
                1,
                2
            )

            a = c[
                :,
                :-1
            ]

            b = c[
                :,
                1:
            ]

            feat = torch.cat(
                [
                    a,
                    b,
                    torch.abs(
                        a - b
                    )
                ],
                dim=-1
            )

            return self.mlp(
                feat
            ).squeeze(
                -1
            )

    wbd = WorkflowBoundary(
        D_MODEL
    ).to(device)

    wopt = torch.optim.AdamW(
        wbd.parameters(),
        lr=1e-3
    )


    # ========================================================
    # Workflow Boundary Training
    # ========================================================

    for ep in range(
        WORKFLOW_BOUNDARY_EPOCHS
    ):

        ls = []

        for sid, g in segdf.groupby(
            'stream_id',
            sort=False
        ):

            ids = (
                g
                .sort_values(
                    'start_global_idx'
                )
                .segment_idx
                .to_numpy()
            )

            if len(ids) < 2:
                continue

            for st in range(
                0,
                len(ids),
                SEGMENT_WINDOW
            ):

                sub = ids[
                    st:st + SEGMENT_WINDOW
                ]

                if len(sub) < 2:
                    continue

                h = torch.tensor(
                    sh[sub],
                    dtype=torch.float32,
                    device=device
                ).unsqueeze(0)

                attention_mask = torch.ones(
                    1,
                    len(sub),
                    dtype=torch.bool,
                    device=device
                )

                target, valid = (
                    contextual_multiscale_target(
                        h,
                        attention_mask,
                        k=2
                    )
                )

                logits = wbd(h)

                if not valid.any():
                    continue

                loss = (
                    F.binary_cross_entropy_with_logits(
                        logits[valid],
                        target[valid]
                    )
                    + 0.01
                    * torch.sigmoid(
                        logits[valid]
                    ).mean()
                )

                wopt.zero_grad()

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    wbd.parameters(),
                    max_norm=1.0
                )

                wopt.step()

                ls.append(
                    loss.item()
                )

        print(
            f"wbd epoch={ep + 1:02d} "
            f"loss="
            f"{np.mean(ls) if ls else None}"
        )


    # ========================================================
    # V5.6 Transition-Contrastive Workflow Boundary Signal
    # ========================================================

    class WorkflowTransitionContrastive(nn.Module):
        """
        Self-supervised transition model.

        source segment → expected next segment representation

        같은 mini-batch의 다른 target segment를 contrastive negatives로 사용한다.
        실제 업무명 / 메뉴 / API 의미 mapping은 사용하지 않는다.
        """
        def __init__(
            self,
            d
        ):
            super().__init__()

            self.source_encoder = nn.Sequential(
                nn.Linear(
                    d,
                    d
                ),
                nn.GELU(),
                nn.LayerNorm(
                    d
                ),
                nn.Linear(
                    d,
                    d
                )
            )

            self.target_encoder = nn.Sequential(
                nn.Linear(
                    d,
                    d
                ),
                nn.GELU(),
                nn.LayerNorm(
                    d
                ),
                nn.Linear(
                    d,
                    d
                )
            )

        def encode_source(
            self,
            x
        ):
            return F.normalize(
                self.source_encoder(
                    x
                ),
                dim=-1
            )

        def encode_target(
            self,
            x
        ):
            return F.normalize(
                self.target_encoder(
                    x
                ),
                dim=-1
            )

        def forward(
            self,
            source,
            target
        ):
            return (
                self.encode_source(
                    source
                ),
                self.encode_target(
                    target
                )
            )


    def _collect_transition_pairs():
        src_ids = []
        tgt_ids = []

        for _, g in segdf.groupby(
            'stream_id',
            sort=False
        ):
            ids = (
                g
                .sort_values(
                    'start_global_idx'
                )
                .segment_idx
                .to_numpy(
                    dtype=int
                )
            )

            if len(
                ids
            ) < 2:
                continue

            src_ids.extend(
                ids[
                    :-1
                ].tolist()
            )

            tgt_ids.extend(
                ids[
                    1:
                ].tolist()
            )

        return (
            np.asarray(
                src_ids,
                dtype=int
            ),
            np.asarray(
                tgt_ids,
                dtype=int
            )
        )


    transition_model = None

    transition_src_ids, transition_tgt_ids = (
        _collect_transition_pairs()
    )

    workflow_transition_diagnostics[
        'pair_count'
    ] = int(
        len(
            transition_src_ids
        )
    )

    transition_training_rows = []

    if (
        USE_TRANSITION_CONTRASTIVE_WORKFLOW_BOUNDARY
        and len(
            transition_src_ids
        ) >= 2
    ):
        transition_model = WorkflowTransitionContrastive(
            D_MODEL
        ).to(
            device
        )

        transition_opt = torch.optim.AdamW(
            transition_model.parameters(),
            lr=WORKFLOW_TRANSITION_LR,
            weight_decay=.01
        )

        rng = np.random.default_rng(
            SEED
        )

        for ep in range(
            1,
            WORKFLOW_TRANSITION_EPOCHS
            + 1
        ):
            transition_model.train()

            order = rng.permutation(
                len(
                    transition_src_ids
                )
            )

            losses = []
            accs = []

            for st in range(
                0,
                len(
                    order
                ),
                WORKFLOW_TRANSITION_BATCH_SIZE
            ):
                batch_idx = order[
                    st:
                    st
                    + WORKFLOW_TRANSITION_BATCH_SIZE
                ]

                if len(
                    batch_idx
                ) < 2:
                    continue

                src = torch.tensor(
                    sh[
                        transition_src_ids[
                            batch_idx
                        ]
                    ],
                    dtype=torch.float32,
                    device=device
                )

                tgt = torch.tensor(
                    sh[
                        transition_tgt_ids[
                            batch_idx
                        ]
                    ],
                    dtype=torch.float32,
                    device=device
                )

                q, k = transition_model(
                    src,
                    tgt
                )

                logits = (
                    q
                    @ k.T
                    / WORKFLOW_TRANSITION_TEMPERATURE
                )

                labels = torch.arange(
                    len(
                        batch_idx
                    ),
                    device=device
                )

                # symmetric contrastive objective
                loss_forward = F.cross_entropy(
                    logits,
                    labels
                )

                loss_reverse = F.cross_entropy(
                    logits.T,
                    labels
                )

                loss = .5 * (
                    loss_forward
                    + loss_reverse
                )

                transition_opt.zero_grad()

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    transition_model.parameters(),
                    1.0
                )

                transition_opt.step()

                losses.append(
                    float(
                        loss.item()
                    )
                )

                with torch.no_grad():
                    pred = logits.argmax(
                        dim=-1
                    )

                    accs.append(
                        float(
                            pred.eq(
                                labels
                            ).float().mean().item()
                        )
                    )

            row = {
                'epoch':
                    ep,
                'loss':
                    (
                        float(
                            np.mean(
                                losses
                            )
                        )
                        if losses
                        else None
                    ),
                'inbatch_top1_accuracy':
                    (
                        float(
                            np.mean(
                                accs
                            )
                        )
                        if accs
                        else None
                    ),
            }

            transition_training_rows.append(
                row
            )

            print(
                f"workflow_transition epoch={ep:02d} "
                f"loss={row['loss']:.4f} "
                f"top1={row['inbatch_top1_accuracy']:.4f}"
            )

        pd.DataFrame(
            transition_training_rows
        ).to_csv(
            DIAGNOSTICS_DIR
            / 'workflow_transition_training.csv',
            index=False,
            encoding='utf-8-sig'
        )

        torch.save(
            {
                'state_dict':
                    transition_model.state_dict(),
                'd_model':
                    D_MODEL,
                'temperature':
                    WORKFLOW_TRANSITION_TEMPERATURE,
                'blend_weight':
                    WORKFLOW_TRANSITION_BLEND_WEIGHT,
            },
            OUTPUT_DIR
            / 'workflow_transition_model.pt'
        )

    else:
        print(
            'workflow transition contrastive training skipped'
        )


    # ========================================================
    # Workflow Boundary Model 저장
    # ========================================================

    torch.save(
        {
            'workflow_boundary_state_dict':
                wbd.state_dict()
        },
        OUTPUT_DIR
        / 'workflow_boundary_model.pt'
    )


    # ========================================================
    # Workflow Boundary Probability 추론
    # ========================================================

    wprobs = np.zeros(
        len(segdf),
        dtype=np.float32
    )

    wcnt = np.zeros(
        len(segdf),
        dtype=np.float32
    )

    wbd.eval()

    with torch.no_grad():

        for sid, g in segdf.groupby(
            'stream_id',
            sort=False
        ):

            ids = (
                g
                .sort_values(
                    'start_global_idx'
                )
                .segment_idx
                .to_numpy()
            )

            for st in range(
                0,
                len(ids),
                SEGMENT_STRIDE
            ):

                sub = ids[
                    st:st + SEGMENT_WINDOW
                ]

                if len(sub) < 2:
                    continue

                h = torch.tensor(
                    sh[sub],
                    dtype=torch.float32,
                    device=device
                ).unsqueeze(0)

                p = (
                    torch
                    .sigmoid(
                        wbd(h)
                    )
                    .squeeze(0)
                    .cpu()
                    .numpy()
                )

                for t in range(
                    len(sub) - 1
                ):

                    wprobs[
                        sub[t + 1]
                    ] += p[t]

                    wcnt[
                        sub[t + 1]
                    ] += 1


    wprobs = (
        wprobs
        / np.maximum(
            wcnt,
            1
        )
    )

    workflow_contextual_probs = wprobs.copy()


    # ========================================================
    # V5.6 Transition Surprise Inference
    # ========================================================

    def _robust_surprise_to_score(
        values
    ):
        values = np.asarray(
            values,
            dtype=np.float32
        )

        if len(
            values
        ) == 0:
            return values

        med = float(
            np.median(
                values
            )
        )

        mad = float(
            np.median(
                np.abs(
                    values
                    - med
                )
            )
        )

        scale = max(
            1.4826
            * mad,
            1e-3
        )

        z = (
            values
            - med
        ) / scale

        return (
            1.0
            / (
                1.0
                + np.exp(
                    -np.clip(
                        z,
                        -12,
                        12
                    )
                )
            )
        ).astype(
            np.float32
        )


    transition_compatibilities = []

    if (
        transition_model is not None
        and len(
            transition_src_ids
        )
    ):
        transition_model.eval()

        with torch.no_grad():
            src_tensor = torch.tensor(
                sh[
                    transition_src_ids
                ],
                dtype=torch.float32,
                device=device
            )

            tgt_tensor = torch.tensor(
                sh[
                    transition_tgt_ids
                ],
                dtype=torch.float32,
                device=device
            )

            batch_scores = []

            for st in range(
                0,
                len(
                    transition_src_ids
                ),
                WORKFLOW_TRANSITION_BATCH_SIZE
            ):
                src = src_tensor[
                    st:
                    st
                    + WORKFLOW_TRANSITION_BATCH_SIZE
                ]

                tgt = tgt_tensor[
                    st:
                    st
                    + WORKFLOW_TRANSITION_BATCH_SIZE
                ]

                q, k = transition_model(
                    src,
                    tgt
                )

                comp = (
                    q
                    * k
                ).sum(
                    -1
                )

                batch_scores.append(
                    comp.cpu().numpy()
                )

            transition_compatibilities = np.concatenate(
                batch_scores
            ).astype(
                np.float32
            )

        raw_surprise = (
            1.0
            - transition_compatibilities
        ).astype(
            np.float32
        )

        # stream별 local distribution으로 normalize.
        pair_cursor = 0

        for _, g in segdf.groupby(
            'stream_id',
            sort=False
        ):
            ids = (
                g
                .sort_values(
                    'start_global_idx'
                )
                .segment_idx
                .to_numpy(
                    dtype=int
                )
            )

            n_pairs = max(
                len(
                    ids
                )
                - 1,
                0
            )

            if n_pairs == 0:
                continue

            local_raw = raw_surprise[
                pair_cursor:
                pair_cursor
                + n_pairs
            ]

            local_score = _robust_surprise_to_score(
                local_raw
            )

            target_ids = ids[
                1:
            ]

            workflow_transition_raw_surprise[
                target_ids
            ] = local_raw

            workflow_transition_scores[
                target_ids
            ] = local_score

            pair_cursor += n_pairs

        w = float(
            WORKFLOW_TRANSITION_BLEND_WEIGHT
        )

        wprobs = (
            (
                1.0
                - w
            )
            * workflow_contextual_probs
            + w
            * workflow_transition_scores
        ).astype(
            np.float32
        )

        valid_transition = workflow_transition_scores[
            workflow_transition_scores
            > 0
        ]

        workflow_transition_diagnostics.update(
            {
                'training_final':
                    (
                        transition_training_rows[
                            -1
                        ]
                        if transition_training_rows
                        else None
                    ),
                'mean_positive_compatibility':
                    float(
                        np.mean(
                            transition_compatibilities
                        )
                    ),
                'p10_positive_compatibility':
                    float(
                        np.quantile(
                            transition_compatibilities,
                            .10
                        )
                    ),
                'p50_positive_compatibility':
                    float(
                        np.quantile(
                            transition_compatibilities,
                            .50
                        )
                    ),
                'mean_raw_surprise':
                    float(
                        np.mean(
                            raw_surprise
                        )
                    ),
                'p90_raw_surprise':
                    float(
                        np.quantile(
                            raw_surprise,
                            .90
                        )
                    ),
                'mean_transition_boundary_score':
                    (
                        float(
                            np.mean(
                                valid_transition
                            )
                        )
                        if len(
                            valid_transition
                        )
                        else None
                    ),
                'mean_contextual_boundary_score':
                    float(
                        np.mean(
                            workflow_contextual_probs[
                                wcnt
                                > 0
                            ]
                        )
                    )
                    if np.any(
                        wcnt
                        > 0
                    )
                    else None,
                'mean_combined_boundary_score':
                    float(
                        np.mean(
                            wprobs[
                                wcnt
                                > 0
                            ]
                        )
                    )
                    if np.any(
                        wcnt
                        > 0
                    )
                    else None,
            }
        )

    else:
        wprobs = workflow_contextual_probs.copy()

        workflow_transition_diagnostics[
            'fallback'
        ] = 'V5.4 contextual workflow boundary only'


    pd.DataFrame(
        {
            'segment_idx':
                segdf[
                    'segment_idx'
                ].to_numpy(),
            'stream_id':
                segdf[
                    'stream_id'
                ].astype(
                    str
                ).to_numpy(),
            'contextual_boundary_prob_before':
                workflow_contextual_probs,
            'transition_raw_surprise_before':
                workflow_transition_raw_surprise,
            'transition_boundary_score_before':
                workflow_transition_scores,
            'combined_boundary_prob_before':
                wprobs,
        }
    ).to_csv(
        DIAGNOSTICS_DIR
        / 'workflow_boundary_scores.csv',
        index=False,
        encoding='utf-8-sig'
    )

    print(
        'workflow transition diagnostics:',
        workflow_transition_diagnostics
    )


    # ========================================================
    # V5.7 Workflow threshold calibration diagnostics
    # ========================================================

    def _workflow_cut_set(
        signal,
        threshold_method
    ):
        cut_set=set()
        workflow_count=0
        per_stream=[]

        for sid,g in segdf.groupby(
            'stream_id',
            sort=False
        ):
            ids=(
                g
                .sort_values(
                    'start_global_idx'
                )
                .segment_idx
                .to_numpy(
                    dtype=int
                )
            )

            if len(
                ids
            ) == 0:
                continue

            vals=(
                signal[
                    ids
                ][
                    1:
                ]
                if len(
                    ids
                ) > 1
                else np.asarray(
                    []
                )
            )

            if len(
                ids
            ) > 1:
                if threshold_method=='legacy':
                    th=threshold_gmm_legacy(
                        vals
                    )

                    meta={
                        'mode':
                            'legacy_gmm_midpoint'
                    }

                else:
                    th,meta=threshold_calibrated(
                        vals,
                        return_meta=True
                    )

                starts=[0]+[
                    i
                    for i in range(
                        1,
                        len(
                            ids
                        )
                    )
                    if signal[
                        ids[
                            i
                        ]
                    ]>=th
                ]

            else:
                th=1.0
                meta={
                    'mode':
                        'single_segment'
                }
                starts=[0]

            filt=[0]

            for pos in starts[
                1:
            ]:
                if (
                    pos-filt[
                        -1
                    ]
                    >= MIN_WORKFLOW_SEGMENTS
                ):
                    filt.append(
                        pos
                    )

            if (
                len(
                    ids
                )
                - filt[
                    -1
                ]
                < MIN_WORKFLOW_SEGMENTS
                and len(
                    filt
                ) > 1
            ):
                filt.pop()

            for pos in filt[
                1:
            ]:
                cut_set.add(
                    int(
                        ids[
                            pos
                        ]
                    )
                )

            workflow_count+=len(
                filt
            )

            per_stream.append(
                {
                    'stream_id':
                        safe_str(
                            sid
                        ),
                    'threshold':
                        float(
                            th
                        ),
                    'threshold_mode':
                        safe_str(
                            meta.get(
                                'mode'
                            )
                        ),
                    'workflow_count':
                        int(
                            len(
                                filt
                            )
                        ),
                    'cut_count':
                        int(
                            max(
                                len(
                                    filt
                                )-1,
                                0
                            )
                        ),
                }
            )

        return (
            cut_set,
            int(
                workflow_count
            ),
            per_stream
        )


    threshold_signals={
        'contextual':
            workflow_contextual_probs,
        'transition':
            workflow_transition_scores,
        'combined':
            wprobs,
    }

    workflow_threshold_sets={}
    workflow_threshold_summary_rows=[]

    for signal_name,signal in threshold_signals.items():
        valid_vals=[]

        for _,g in segdf.groupby(
            'stream_id',
            sort=False
        ):
            ids=(
                g
                .sort_values(
                    'start_global_idx'
                )
                .segment_idx
                .to_numpy(
                    dtype=int
                )
            )

            if len(
                ids
            )>1:
                valid_vals.extend(
                    signal[
                        ids
                    ][
                        1:
                    ].tolist()
                )

        valid_vals=np.asarray(
            valid_vals,
            dtype=float
        )

        for method in [
            'legacy',
            'calibrated'
        ]:
            cuts,n_workflows,_=_workflow_cut_set(
                signal,
                method
            )

            workflow_threshold_sets[
                (
                    signal_name,
                    method
                )
            ]=cuts

            workflow_threshold_summary_rows.append(
                {
                    'signal':
                        signal_name,
                    'threshold_method':
                        method,
                    'workflow_count':
                        int(
                            n_workflows
                        ),
                    'cut_count':
                        int(
                            len(
                                cuts
                            )
                        ),
                    'score_median':
                        (
                            float(
                                np.median(
                                    valid_vals
                                )
                            )
                            if len(
                                valid_vals
                            )
                            else None
                        ),
                    'score_mean':
                        (
                            float(
                                np.mean(
                                    valid_vals
                                )
                            )
                            if len(
                                valid_vals
                            )
                            else None
                        ),
                    'score_p90':
                        (
                            float(
                                np.quantile(
                                    valid_vals,
                                    .90
                                )
                            )
                            if len(
                                valid_vals
                            )
                            else None
                        ),
                }
            )


    def _set_jaccard(
        a,
        b
    ):
        a=set(
            a
        )
        b=set(
            b
        )

        union=a|b

        if not union:
            return 1.0

        return (
            len(
                a&b
            )
            / len(
                union
            )
        )


    workflow_jaccard_rows=[]

    signal_names=list(
        threshold_signals.keys()
    )

    for method in [
        'legacy',
        'calibrated'
    ]:
        for i in range(
            len(
                signal_names
            )
        ):
            for j in range(
                i+1,
                len(
                    signal_names
                )
            ):
                a=signal_names[
                    i
                ]
                b=signal_names[
                    j
                ]

                aset=workflow_threshold_sets[
                    (
                        a,
                        method
                    )
                ]

                bset=workflow_threshold_sets[
                    (
                        b,
                        method
                    )
                ]

                workflow_jaccard_rows.append(
                    {
                        'comparison_type':
                            'between_signals',
                        'threshold_method':
                            method,
                        'signal_a':
                            a,
                        'signal_b':
                            b,
                        'cuts_a':
                            int(
                                len(
                                    aset
                                )
                            ),
                        'cuts_b':
                            int(
                                len(
                                    bset
                                )
                            ),
                        'shared_cuts':
                            int(
                                len(
                                    aset&bset
                                )
                            ),
                        'jaccard':
                            float(
                                _set_jaccard(
                                    aset,
                                    bset
                                )
                            ),
                    }
                )

    for signal_name in signal_names:
        legacy_set=workflow_threshold_sets[
            (
                signal_name,
                'legacy'
            )
        ]

        calibrated_set=workflow_threshold_sets[
            (
                signal_name,
                'calibrated'
            )
        ]

        workflow_jaccard_rows.append(
            {
                'comparison_type':
                    'legacy_vs_calibrated',
                'threshold_method':
                    'mixed',
                'signal_a':
                    f'{signal_name}_legacy',
                'signal_b':
                    f'{signal_name}_calibrated',
                'cuts_a':
                    int(
                        len(
                            legacy_set
                        )
                    ),
                'cuts_b':
                    int(
                        len(
                            calibrated_set
                        )
                    ),
                'shared_cuts':
                    int(
                        len(
                            legacy_set
                            & calibrated_set
                        )
                    ),
                'jaccard':
                    float(
                        _set_jaccard(
                            legacy_set,
                            calibrated_set
                        )
                    ),
            }
        )

    workflow_threshold_signal_summary=pd.DataFrame(
        workflow_threshold_summary_rows
    )

    workflow_cut_jaccard=pd.DataFrame(
        workflow_jaccard_rows
    )

    workflow_threshold_signal_summary.to_csv(
        DIAGNOSTICS_DIR
        / 'workflow_threshold_signal_summary.csv',
        index=False,
        encoding='utf-8-sig'
    )

    workflow_cut_jaccard.to_csv(
        DIAGNOSTICS_DIR
        / 'workflow_cut_jaccard.csv',
        index=False,
        encoding='utf-8-sig'
    )

    print(
        '\\n=== V5.7 THRESHOLD SIGNAL SUMMARY ==='
    )

    print(
        workflow_threshold_signal_summary.to_string(
            index=False
        )
    )

    print(
        '\\n=== CUT LOCATION JACCARD ==='
    )

    print(
        workflow_cut_jaccard.to_string(
            index=False
        )
    )


    # ========================================================
    # Workflow 생성
    # ========================================================

    workflows = []

    workflow_members = []

    workflow_threshold_rows = []

    wid = 0


    for sid, g in segdf.groupby(
        'stream_id',
        sort=False
    ):

        ids = (
            g
            .sort_values(
                'start_global_idx'
            )
            .segment_idx
            .to_numpy()
        )

        if len(ids) == 0:
            continue

        if len(ids) > 1:

            vals=wprobs[
                ids
            ][
                1:
            ]

            th,threshold_meta=threshold_calibrated(
                vals,
                return_meta=True
            )

            legacy_th=threshold_gmm_legacy(
                vals
            )

            starts = [0] + [
                i
                for i in range(
                    1,
                    len(ids)
                )
                if wprobs[
                    ids[i]
                ] >= th
            ]

        else:
            th = 1.0
            legacy_th = 1.0
            threshold_meta={
                'mode':
                    'single_segment',
                'bic_gain':
                    None,
                'separation_sigma':
                    None,
            }
            starts = [0]


        # ----------------------------------------------------
        # 지나치게 짧은 workflow 제거
        # ----------------------------------------------------

        filt = [0]

        for pos in starts[1:]:

            if (
                pos - filt[-1]
                >= MIN_WORKFLOW_SEGMENTS
            ):
                filt.append(pos)


        if (
            len(ids) - filt[-1]
            < MIN_WORKFLOW_SEGMENTS
            and len(filt) > 1
        ):
            filt.pop()

        workflow_threshold_rows.append(
            {
                'stream_id':
                    safe_str(
                        sid
                    ),
                'segment_count':
                    int(
                        len(
                            ids
                        )
                    ),
                'legacy_threshold':
                    float(
                        legacy_th
                    ),
                'calibrated_threshold':
                    float(
                        th
                    ),
                'threshold_mode':
                    safe_str(
                        threshold_meta.get(
                            'mode'
                        )
                    ),
                'bic_gain':
                    threshold_meta.get(
                        'bic_gain'
                    ),
                'separation_sigma':
                    threshold_meta.get(
                        'separation_sigma'
                    ),
                'calibrated_workflow_count':
                    int(
                        len(
                            filt
                        )
                    ),
            }
        )


        # ----------------------------------------------------
        # Workflow 저장
        # ----------------------------------------------------

        for j, start in enumerate(filt):

            end = (
                filt[j + 1] - 1
                if j + 1 < len(filt)
                else len(ids) - 1
            )

            segids = ids[
                start:end + 1
            ]

            if len(segids) == 0:
                continue

            wid += 1

            workflows.append(
                {
                    'workflow_idx':
                        wid - 1,

                    'workflow_id':
                        f'W{wid:05d}',

                    'stream_id':
                        sid,

                    'segment_start':
                        int(
                            segids[0]
                        ),

                    'segment_end':
                        int(
                            segids[-1]
                        ),

                    'segment_count':
                        len(segids),

                    'start_time':
                        segdf.loc[
                            segids[0],
                            'start_time'
                        ],

                    'end_time':
                        segdf.loc[
                            segids[-1],
                            'end_time'
                        ],

                    'boundary_threshold':
                        th
                }
            )

            workflow_members.append(
                segids
            )


    pd.DataFrame(
        workflow_threshold_rows
    ).to_csv(
        DIAGNOSTICS_DIR
        / 'workflow_threshold_diagnostics.csv',
        index=False,
        encoding='utf-8-sig'
    )


    # ========================================================
    # Workflow Embedding 생성
    # ========================================================

    workflow_df = pd.DataFrame(
        workflows
    )


    if workflow_members:

        workflow_embeddings = np.stack(
            [
                sh[x].mean(0)
                for x in workflow_members
            ]
        )

    else:

        workflow_embeddings = np.empty(
            (
                0,
                D_MODEL
            ),
            dtype=np.float32
        )


    # ========================================================
    # 결과 저장
    # ========================================================

    workflow_df.to_csv(
        OUTPUT_DIR
        / 'latent_workflows.csv',
        index=False,
        encoding='utf-8-sig'
    )

    np.save(
        OUTPUT_DIR
        / 'workflow_embeddings.npy',
        workflow_embeddings
    )

    print(
        "workflows:",
        len(workflow_df)
    )

    print(
        workflow_df.head()
    )

/tmp/ipykernel_1271/771061633.py:147: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tr = nn.TransformerEncoder(


segssl epoch=01 loss=1.0641 batches=21
segssl epoch=02 loss=0.6205 batches=21
segssl epoch=03 loss=0.4802 batches=21
segssl epoch=04 loss=0.4341 batches=21
segssl epoch=05 loss=0.3974 batches=21
segssl epoch=06 loss=0.3608 batches=21
segssl epoch=07 loss=0.3376 batches=21
segssl epoch=08 loss=0.3039 batches=21
wbd epoch=01 loss=0.6581058121291843
wbd epoch=02 loss=0.6371847916636947
wbd epoch=03 loss=0.6260016206100848
wbd epoch=04 loss=0.6157583881764722
wbd epoch=05 loss=0.6141270414964687
wbd epoch=06 loss=0.6110615934845964
workflow_transition epoch=01 loss=3.6739 top1=0.1823
workflow_transition epoch=02 loss=2.7860 top1=0.4615
workflow_transition epoch=03 loss=2.3466 top1=0.7124
workflow_transition epoch=04 loss=1.6749 top1=0.8268
workflow_transition epoch=05 loss=1.1693 top1=0.8726
workflow_transition epoch=06 loss=0.9300 top1=0.9001
workflow_transition epoch=07 loss=0.7910 top1=0.9057
workflow_transition epoch=08 loss=0.7274 top1=0.9252
workflow transition diagnostics: {'enabled

## 8. V6.2 Stable Prototype Ensemble / Consensus

V5.7에서 prototype assignment가 seed에 따라 크게 흔들리는 것이 확인되었습니다.

V6.2에서는 single-seed prototype을 최종 workflow ID로 사용하지 않습니다.

```text
Seed 41 ─┐
Seed 42 ─┤
Seed 43 ─┤
Seed 44 ─┤
Seed 45 ─┘
          ↓
label-invariant assignment feature
          ↓
consensus clustering
          ↓
stable SP / WP
```

각 seed의 prototype model은 기존 V4.2 objective를 유지하되,
20 full-batch step 대신 최대 300 optimization step과 early stopping을 사용합니다.

Workflow consensus의 cluster 수는 gold 업무 수를 보지 않고
`7~12` 후보에서 silhouette / seed agreement / balance / small-cluster penalty를 이용해 선택합니다.

`7`이라는 하한은 고객 납품 목표이며, 알려진 6개 업무 label과는 무관합니다.

In [9]:
# ============================================================
# V6.0 Stable Multi-seed Prototype Ensemble / Consensus
# ============================================================

class NeuralPrototypeModel(nn.Module):
    def __init__(
        self,
        input_dim,
        k,
        latent_dim=64
    ):
        super().__init__()

        self.projector=nn.Sequential(
            nn.Linear(
                input_dim,
                input_dim
            ),
            nn.GELU(),
            nn.Dropout(
                .15
            ),
            nn.Linear(
                input_dim,
                latent_dim
            )
        )

        self.prototypes=nn.Parameter(
            torch.randn(
                k,
                latent_dim
            )
        )

    def forward(
        self,
        x,
        temp=.15
    ):
        z=F.normalize(
            self.projector(
                x
            ),
            dim=-1
        )

        p=F.normalize(
            self.prototypes,
            dim=-1
        )

        return (
            z,
            z@p.T/temp
        )


def choose_k(
    n,
    max_k
):
    if n<4:
        return 1

    return min(
        max_k,
        max(
            2,
            int(
                round(
                    math.sqrt(
                        n
                    )
                )
            )
        )
    )


def _train_prototype_seed(
    vectors,
    max_k,
    seed,
    prefix
):
    """
    V4.2 objective를 유지하되 20 full-batch step에서
    충분한 optimization steps로 늘린다.
    """
    n=len(
        vectors
    )

    if n==0:
        return {
            'seed':
                int(
                    seed
                ),
            'k':
                0,
            'assignment':
                np.array(
                    [],
                    dtype=int
                ),
            'confidence':
                np.array(
                    [],
                    dtype=np.float32
                ),
            'margin':
                np.array(
                    [],
                    dtype=np.float32
                ),
            'loss':
                None,
            'steps':
                0,
            'state_dict':
                None,
        }

    k=choose_k(
        n,
        max_k
    )

    if k==1:
        return {
            'seed':
                int(
                    seed
                ),
            'k':
                1,
            'assignment':
                np.zeros(
                    n,
                    dtype=int
                ),
            'confidence':
                np.ones(
                    n,
                    dtype=np.float32
                ),
            'margin':
                np.ones(
                    n,
                    dtype=np.float32
                ),
            'loss':
                0.0,
            'steps':
                0,
            'state_dict':
                None,
        }

    random.seed(
        int(
            seed
        )
    )

    np.random.seed(
        int(
            seed
        )
    )

    torch.manual_seed(
        int(
            seed
        )
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(
                seed
            )
        )

    pm=NeuralPrototypeModel(
        vectors.shape[
            1
        ],
        k,
        min(
            64,
            vectors.shape[
                1
            ]
        )
    ).to(
        device
    )

    x=torch.tensor(
        vectors,
        dtype=torch.float32,
        device=device
    )

    opt=torch.optim.AdamW(
        pm.parameters(),
        lr=PROTOTYPE_LR,
        weight_decay=.01
    )

    best_loss=float(
        'inf'
    )

    best_state=None
    no_improve=0
    last_loss=None
    actual_steps=0

    for step in range(
        1,
        PROTOTYPE_OPT_STEPS+1
    ):
        pm.train()

        x1=(
            F.dropout(
                x,
                p=.15,
                training=True
            )
            + .01
            * torch.randn_like(
                x
            )
        )

        x2=(
            F.dropout(
                x,
                p=.15,
                training=True
            )
            + .01
            * torch.randn_like(
                x
            )
        )

        _,l1=pm(
            x1
        )

        _,l2=pm(
            x2
        )

        q1=torch.softmax(
            l1,
            -1
        )

        q2=torch.softmax(
            l2,
            -1
        )

        consistency=-.5*(
            (
                q1.detach()
                * torch.log_softmax(
                    l2,
                    -1
                )
            ).sum(
                -1
            ).mean()
            + (
                q2.detach()
                * torch.log_softmax(
                    l1,
                    -1
                )
            ).sum(
                -1
            ).mean()
        )

        marginal=.5*(
            q1.mean(
                0
            )
            + q2.mean(
                0
            )
        )

        uniform=torch.full_like(
            marginal,
            1.0/k
        )

        balance=(
            marginal
            * (
                torch.log(
                    marginal.clamp_min(
                        1e-8
                    )
                )
                - torch.log(
                    uniform
                )
            )
        ).sum()

        proto=F.normalize(
            pm.prototypes,
            dim=-1
        )

        gram=proto@proto.T

        eye=torch.eye(
            k,
            device=device,
            dtype=torch.bool
        )

        separation=(
            gram.masked_select(
                ~eye
            )**2
        ).mean()

        # Original V4.2 objective retained exactly.
        loss=consistency+.2*balance+.02*separation

        opt.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            pm.parameters(),
            1.0
        )

        opt.step()

        actual_steps=step
        last_loss=float(
            loss.item()
        )

        if (
            best_loss
            - last_loss
            > PROTOTYPE_MIN_DELTA
        ):
            best_loss=last_loss
            best_state={
                k:
                    v.detach()
                    .cpu()
                    .clone()
                for k,v in pm.state_dict().items()
            }
            no_improve=0

        else:
            no_improve+=1

        if (
            step==1
            or step%PROTOTYPE_LOG_EVERY==0
        ):
            print(
                f"{prefix} seed={seed} "
                f"step={step:03d} "
                f"loss={last_loss:.4f} "
                f"k={k}"
            )

        if (
            step>=100
            and no_improve
            >= PROTOTYPE_EARLY_STOP_PATIENCE
        ):
            print(
                f"{prefix} seed={seed} early-stop at {step}"
            )
            break

    if best_state is not None:
        pm.load_state_dict(
            best_state
        )

    pm.eval()

    with torch.no_grad():
        _,logits=pm(
            x
        )

        pr=torch.softmax(
            logits,
            -1
        )

        assignment=pr.argmax(
            -1
        ).cpu().numpy()

        confidence=pr.max(
            -1
        ).values.cpu().numpy()

        if k>=2:
            top2=torch.topk(
                pr,
                k=2,
                dim=-1
            ).values

            margin=(
                top2[
                    :,
                    0
                ]
                - top2[
                    :,
                    1
                ]
            ).cpu().numpy()

        else:
            margin=np.ones(
                n,
                dtype=np.float32
            )

    return {
        'seed':
            int(
                seed
            ),
        'k':
            int(
                k
            ),
        'assignment':
            assignment.astype(
                int
            ),
        'confidence':
            confidence.astype(
                np.float32
            ),
        'margin':
            margin.astype(
                np.float32
            ),
        'loss':
            float(
                best_loss
            )
            if np.isfinite(
                best_loss
            )
            else last_loss,
        'steps':
            int(
                actual_steps
            ),
        'state_dict':
            best_state,
    }


def _assignment_feature(
    runs
):
    """
    Seed별 prototype label 번호는 서로 비교할 수 없으므로,
    각 seed assignment를 one-hot으로 바꾸어 이어 붙인다.

    Euclidean distance는 여러 seed에서 얼마나 자주 같은 prototype에
    묶였는지를 label permutation과 무관하게 반영한다.
    """
    if not runs:
        return np.empty(
            (
                0,
                0
            ),
            dtype=np.float32
        )

    features=[]

    for run in runs:
        a=np.asarray(
            run[
                'assignment'
            ],
            dtype=int
        )

        k=int(
            run[
                'k'
            ]
        )

        onehot=np.zeros(
            (
                len(
                    a
                ),
                max(
                    k,
                    1
                )
            ),
            dtype=np.float32
        )

        if len(
            a
        ):
            onehot[
                np.arange(
                    len(
                        a
                    )
                ),
                a
            ]=1.0

        features.append(
            onehot
        )

    x=np.concatenate(
        features,
        axis=1
    )

    x/=math.sqrt(
        max(
            len(
                features
            ),
            1
        )
    )

    return x


def _within_seed_agreement(
    labels,
    runs
):
    """
    consensus cluster 내부의 item pair가 원래 seed prototype에서도
    같이 묶이는 비율.
    """
    labels=np.asarray(
        labels,
        dtype=int
    )

    numer=0.0
    denom=0.0

    for cluster_id in np.unique(
        labels
    ):
        ids=np.flatnonzero(
            labels==cluster_id
        )

        m=len(
            ids
        )

        if m<2:
            continue

        pair_total=(
            m
            * (
                m-1
            )
            / 2
        )

        for run in runs:
            a=np.asarray(
                run[
                    'assignment'
                ]
            )[
                ids
            ]

            counts=np.bincount(
                a,
                minlength=max(
                    int(
                        run[
                            'k'
                        ]
                    ),
                    1
                )
            )

            same=float(
                (
                    counts
                    * (
                        counts-1
                    )
                    / 2
                ).sum()
            )

            numer+=same
            denom+=pair_total

    return (
        float(
            numer/denom
        )
        if denom>0
        else 1.0
    )


def _cluster_balance_entropy(
    labels
):
    labels=np.asarray(
        labels,
        dtype=int
    )

    if len(
        labels
    )==0:
        return 0.0

    counts=np.bincount(
        labels
    )

    counts=counts[
        counts>0
    ]

    if len(
        counts
    )<=1:
        return 0.0

    p=counts/counts.sum()

    return float(
        -(
            p*np.log(
                p
            )
        ).sum()
        / math.log(
            len(
                counts
            )
        )
    )


def _small_cluster_share(
    labels,
    min_support
):
    labels=np.asarray(
        labels,
        dtype=int
    )

    if len(
        labels
    )==0:
        return 1.0

    counts=np.bincount(
        labels
    )

    small_ids=np.flatnonzero(
        counts<min_support
    )

    return float(
        np.isin(
            labels,
            small_ids
        ).mean()
    )


def _fit_agglomerative(
    x,
    k
):
    # sklearn compatibility across versions.
    try:
        model=AgglomerativeClustering(
            n_clusters=int(
                k
            ),
            linkage='ward'
        )

        return model.fit_predict(
            x
        )

    except TypeError:
        model=AgglomerativeClustering(
            n_clusters=int(
                k
            ),
            linkage='ward'
        )

        return model.fit_predict(
            x
        )


def _consensus_k_search(
    feature,
    runs,
    k_min,
    k_max,
    prefix,
    min_support
):
    n=len(
        feature
    )

    if n==0:
        return (
            np.array(
                [],
                dtype=int
            ),
            pd.DataFrame(),
            {
                'k':
                    0,
                'score':
                    None,
            }
        )

    k_min=max(
        2,
        min(
            int(
                k_min
            ),
            n
        )
    )

    k_max=max(
        k_min,
        min(
            int(
                k_max
            ),
            n
        )
    )

    rows=[]
    candidates={}

    for k in range(
        k_min,
        k_max+1
    ):
        labels=_fit_agglomerative(
            feature,
            k
        )

        candidates[
            k
        ]=labels

        if (
            len(
                np.unique(
                    labels
                )
            )>1
            and len(
                labels
            )>len(
                np.unique(
                    labels
                )
            )
        ):
            sample_size=min(
                1500,
                len(
                    labels
                )
            )

            sil=float(
                silhouette_score(
                    feature,
                    labels,
                    metric='euclidean',
                    sample_size=(
                        sample_size
                        if sample_size<len(
                            labels
                        )
                        else None
                    ),
                    random_state=SEED
                )
            )

        else:
            sil=-1.0

        within=_within_seed_agreement(
            labels,
            runs
        )

        entropy=_cluster_balance_entropy(
            labels
        )

        small_share=_small_cluster_share(
            labels,
            min_support
        )

        sil01=(
            sil+1.0
        )/2.0

        score=(
            .50*sil01
            + .35*within
            + .15*entropy
            - .30*small_share
        )

        rows.append(
            {
                'level':
                    prefix,
                'k':
                    int(
                        k
                    ),
                'silhouette':
                    sil,
                'within_seed_agreement':
                    within,
                'balance_entropy':
                    entropy,
                'small_cluster_share':
                    small_share,
                'consensus_score':
                    float(
                        score
                    ),
            }
        )

    search_df=pd.DataFrame(
        rows
    )

    best_row=(
        search_df
        .sort_values(
            [
                'consensus_score',
                'within_seed_agreement',
                'silhouette'
            ],
            ascending=[
                False,
                False,
                False
            ]
        )
        .iloc[
            0
        ]
    )

    best_k=int(
        best_row[
            'k'
        ]
    )

    labels=np.asarray(
        candidates[
            best_k
        ],
        dtype=int
    )

    # support가 큰 cluster부터 번호를 다시 부여해
    # WP/SP numbering이 seed가 아니라 실제 support에 안정적으로 대응.
    counts=Counter(
        labels.tolist()
    )

    ordered=[
        old
        for old,_ in sorted(
            counts.items(),
            key=lambda x:(
                -x[
                    1
                ],
                x[
                    0
                ]
            )
        )
    ]

    remap={
        old:
            new
        for new,old in enumerate(
            ordered
        )
    }

    labels=np.asarray(
        [
            remap[
                int(
                    x
                )
            ]
            for x in labels
        ],
        dtype=int
    )

    summary={
        'k':
            best_k,
        'score':
            float(
                best_row[
                    'consensus_score'
                ]
            ),
        'silhouette':
            float(
                best_row[
                    'silhouette'
                ]
            ),
        'within_seed_agreement':
            float(
                best_row[
                    'within_seed_agreement'
                ]
            ),
        'balance_entropy':
            float(
                best_row[
                    'balance_entropy'
                ]
            ),
        'small_cluster_share':
            float(
                best_row[
                    'small_cluster_share'
                ]
            ),
    }

    return (
        labels,
        search_df,
        summary
    )


def _consensus_item_metrics(
    feature,
    labels
):
    labels=np.asarray(
        labels,
        dtype=int
    )

    if len(
        labels
    )==0:
        return (
            np.array(
                [],
                dtype=np.float32
            ),
            np.array(
                [],
                dtype=np.float32
            )
        )

    unique=np.unique(
        labels
    )

    centroids=[]

    for c in unique:
        centroids.append(
            feature[
                labels==c
            ].mean(
                axis=0
            )
        )

    centroids=np.asarray(
        centroids,
        dtype=np.float32
    )

    fnorm=feature/np.maximum(
        np.linalg.norm(
            feature,
            axis=1,
            keepdims=True
        ),
        1e-8
    )

    cnorm=centroids/np.maximum(
        np.linalg.norm(
            centroids,
            axis=1,
            keepdims=True
        ),
        1e-8
    )

    sim=fnorm@cnorm.T

    own=np.zeros(
        len(
            labels
        ),
        dtype=np.float32
    )

    margin=np.zeros(
        len(
            labels
        ),
        dtype=np.float32
    )

    cluster_to_pos={
        int(
            c
        ):
            i
        for i,c in enumerate(
            unique
        )
    }

    for i,c in enumerate(
        labels
    ):
        pos=cluster_to_pos[
            int(
                c
            )
        ]

        own[
            i
        ]=sim[
            i,
            pos
        ]

        if sim.shape[
            1
        ]>=2:
            others=np.delete(
                sim[
                    i
                ],
                pos
            )

            second=float(
                others.max()
            )

        else:
            second=0.0

        margin[
            i
        ]=own[
            i
        ]-second

    # cosine similarity는 이 one-hot consensus feature에서 0~1 범위.
    confidence=np.clip(
        own,
        0.0,
        1.0
    )

    return (
        confidence.astype(
            np.float32
        ),
        margin.astype(
            np.float32
        )
    )


def _ensemble_level(
    vectors,
    max_k,
    prefix,
    consensus_k_min,
    consensus_k_max,
    consensus_min_support
):
    runs=[]
    run_rows=[]

    for seed in PROTOTYPE_ENSEMBLE_SEEDS:
        run=_train_prototype_seed(
            vectors,
            max_k,
            seed,
            prefix
        )

        runs.append(
            run
        )

        run_rows.append(
            {
                'level':
                    prefix,
                'seed':
                    int(
                        seed
                    ),
                'n':
                    int(
                        len(
                            run[
                                'assignment'
                            ]
                        )
                    ),
                'prototype_bank_k':
                    int(
                        run[
                            'k'
                        ]
                    ),
                'used':
                    int(
                        np.unique(
                            run[
                                'assignment'
                            ]
                        ).size
                    )
                    if len(
                        run[
                            'assignment'
                        ]
                    )
                    else 0,
                'steps':
                    int(
                        run[
                            'steps'
                        ]
                    ),
                'best_loss':
                    run[
                        'loss'
                    ],
                'mean_confidence':
                    (
                        float(
                            np.mean(
                                run[
                                    'confidence'
                                ]
                            )
                        )
                        if len(
                            run[
                                'confidence'
                            ]
                        )
                        else None
                    ),
                'mean_probability_margin':
                    (
                        float(
                            np.mean(
                                run[
                                    'margin'
                                ]
                            )
                        )
                        if len(
                            run[
                                'margin'
                            ]
                        )
                        else None
                    ),
            }
        )

    pairwise=[]

    for i in range(
        len(
            runs
        )
    ):
        for j in range(
            i+1,
            len(
                runs
            )
        ):
            a=runs[
                i
            ][
                'assignment'
            ]

            b=runs[
                j
            ][
                'assignment'
            ]

            if (
                len(
                    a
                )
                and len(
                    a
                )==len(
                    b
                )
            ):
                pairwise.append(
                    {
                        'level':
                            prefix,
                        'seed_a':
                            int(
                                runs[
                                    i
                                ][
                                    'seed'
                                ]
                            ),
                        'seed_b':
                            int(
                                runs[
                                    j
                                ][
                                    'seed'
                                ]
                            ),
                        'ari':
                            float(
                                adjusted_rand_score(
                                    a,
                                    b
                                )
                            ),
                        'nmi':
                            float(
                                normalized_mutual_info_score(
                                    a,
                                    b
                                )
                            ),
                    }
                )

    feature=_assignment_feature(
        runs
    )

    labels,k_search,consensus_summary=(
        _consensus_k_search(
            feature,
            runs,
            consensus_k_min,
            consensus_k_max,
            prefix,
            consensus_min_support
        )
    )

    confidence,margin=_consensus_item_metrics(
        feature,
        labels
    )

    stability_summary={
        'level':
            prefix,
        'seeds':
            [
                int(
                    x
                )
                for x in PROTOTYPE_ENSEMBLE_SEEDS
            ],
        'pairwise_ari_mean':
            (
                float(
                    np.mean(
                        [
                            x[
                                'ari'
                            ]
                            for x in pairwise
                        ]
                    )
                )
                if pairwise
                else None
            ),
        'pairwise_ari_min':
            (
                float(
                    np.min(
                        [
                            x[
                                'ari'
                            ]
                            for x in pairwise
                        ]
                    )
                )
                if pairwise
                else None
            ),
        'pairwise_nmi_mean':
            (
                float(
                    np.mean(
                        [
                            x[
                                'nmi'
                            ]
                            for x in pairwise
                        ]
                    )
                )
                if pairwise
                else None
            ),
        'consensus_k':
            int(
                consensus_summary[
                    'k'
                ]
            ),
        'consensus_score':
            consensus_summary[
                'score'
            ],
        'consensus_within_seed_agreement':
            consensus_summary[
                'within_seed_agreement'
            ],
        'consensus_silhouette':
            consensus_summary[
                'silhouette'
            ],
        'mean_consensus_confidence':
            (
                float(
                    np.mean(
                        confidence
                    )
                )
                if len(
                    confidence
                )
                else None
            ),
        'mean_consensus_margin':
            (
                float(
                    np.mean(
                        margin
                    )
                )
                if len(
                    margin
                )
                else None
            ),
        'mean_confidence_across_seeds':
            (
                float(
                    np.mean(
                        [
                            r[
                                'mean_confidence'
                            ]
                            for r in run_rows
                            if r[
                                'mean_confidence'
                            ]
                            is not None
                        ]
                    )
                )
                if run_rows
                else None
            ),
        'mean_prob_margin_across_seeds':
            (
                float(
                    np.mean(
                        [
                            r[
                                'mean_probability_margin'
                            ]
                            for r in run_rows
                            if r[
                                'mean_probability_margin'
                            ]
                            is not None
                        ]
                    )
                )
                if run_rows
                else None
            ),
    }

    return {
        'runs':
            runs,
        'run_rows':
            run_rows,
        'pairwise':
            pairwise,
        'feature':
            feature,
        'labels':
            labels,
        'confidence':
            confidence,
        'margin':
            margin,
        'k_search':
            k_search,
        'consensus_summary':
            consensus_summary,
        'stability_summary':
            stability_summary,
    }


# ============================================================
# Segment consensus
# ============================================================

segment_ensemble=_ensemble_level(
    sh,
    MAX_SEGMENT_PROTOTYPES,
    'segment',
    SEGMENT_CONSENSUS_K_MIN,
    SEGMENT_CONSENSUS_K_MAX,
    5
)

seg_assign=segment_ensemble[
    'labels'
]

seg_conf=segment_ensemble[
    'confidence'
]

seg_margin=segment_ensemble[
    'margin'
]

if len(
    segdf
):
    segdf[
        'segment_prototype'
    ]=[
        f'SP{x:02d}'
        for x in seg_assign
    ]

    segdf[
        'segment_prototype_confidence'
    ]=seg_conf

    segdf[
        'segment_consensus_margin'
    ]=seg_margin

    segdf[
        'workflow_boundary_prob_before'
    ]=wprobs

    segdf[
        'workflow_contextual_boundary_prob_before'
    ]=workflow_contextual_probs

    segdf[
        'workflow_transition_boundary_score_before'
    ]=workflow_transition_scores

    segdf[
        'workflow_transition_raw_surprise_before'
    ]=workflow_transition_raw_surprise

    segdf.to_csv(
        OUTPUT_DIR
        / 'latent_segments.csv',
        index=False,
        encoding='utf-8-sig'
    )


# ============================================================
# Workflow consensus
# ============================================================

workflow_ensemble=_ensemble_level(
    workflow_embeddings,
    MAX_WORKFLOW_PROTOTYPES,
    'workflow',
    WORKFLOW_CONSENSUS_K_MIN,
    WORKFLOW_CONSENSUS_K_MAX,
    REPORT_MIN_INSTANCE_SUPPORT
)

wf_assign=workflow_ensemble[
    'labels'
]

wf_conf=workflow_ensemble[
    'confidence'
]

wf_margin=workflow_ensemble[
    'margin'
]

if len(
    workflow_df
):
    workflow_df[
        'workflow_prototype'
    ]=[
        f'WP{x:02d}'
        for x in wf_assign
    ]

    workflow_df[
        'workflow_prototype_confidence'
    ]=wf_conf

    workflow_df[
        'workflow_consensus_margin'
    ]=wf_margin

    seqs=[]

    for ids in workflow_members:
        seqs.append(
            ' -> '.join(
                segdf.loc[
                    ids,
                    'segment_prototype'
                ].tolist()
            )
        )

    workflow_df[
        'segment_prototype_sequence'
    ]=seqs

    workflow_df.to_csv(
        OUTPUT_DIR
        / 'latent_workflows.csv',
        index=False,
        encoding='utf-8-sig'
    )


# ============================================================
# Diagnostics
# ============================================================

def _prototype_usage_diagnostics(
    assignment,
    k
):
    assignment=np.asarray(
        assignment,
        dtype=int
    )

    if (
        len(
            assignment
        )==0
        or k<=0
    ):
        return {
            'n':
                0,
            'k':
                int(
                    k
                ),
            'used':
                0,
            'top1_share':
                None,
            'top2_share':
                None,
            'normalized_entropy':
                None,
            'counts':
                []
        }

    counts=np.bincount(
        assignment,
        minlength=k
    )

    shares=counts/max(
        counts.sum(),
        1
    )

    used=int(
        (
            counts>0
        ).sum()
    )

    nz=shares[
        shares>0
    ]

    if (
        k>1
        and len(
            nz
        )
    ):
        entropy=float(
            -(
                nz*np.log(
                    nz
                )
            ).sum()
            / math.log(
                k
            )
        )

    else:
        entropy=1.0

    top1=float(
        shares.max()
    )

    top2=(
        float(
            np.sort(
                shares
            )[
                -2:
            ].sum()
        )
        if k>=2
        else top1
    )

    return {
        'n':
            int(
                len(
                    assignment
                )
            ),
        'k':
            int(
                k
            ),
        'used':
            used,
        'top1_share':
            top1,
        'top2_share':
            top2,
        'normalized_entropy':
            entropy,
        'counts':
            counts.tolist()
    }


segment_consensus_k=int(
    segment_ensemble[
        'consensus_summary'
    ][
        'k'
    ]
)

workflow_consensus_k=int(
    workflow_ensemble[
        'consensus_summary'
    ][
        'k'
    ]
)

prototype_diagnostics={
    'segment':
        _prototype_usage_diagnostics(
            seg_assign,
            segment_consensus_k
        ),
    'workflow':
        _prototype_usage_diagnostics(
            wf_assign,
            workflow_consensus_k
        )
}

prototype_stability_diagnostics={
    'segment':
        segment_ensemble[
            'stability_summary'
        ],
    'workflow':
        workflow_ensemble[
            'stability_summary'
        ],
}

ensemble_run_df=pd.DataFrame(
    segment_ensemble[
        'run_rows'
    ]
    + workflow_ensemble[
        'run_rows'
    ]
)

ensemble_pairwise_df=pd.DataFrame(
    segment_ensemble[
        'pairwise'
    ]
    + workflow_ensemble[
        'pairwise'
    ]
)

consensus_k_search_df=pd.concat(
    [
        segment_ensemble[
            'k_search'
        ],
        workflow_ensemble[
            'k_search'
        ],
    ],
    ignore_index=True
)

ensemble_run_df.to_csv(
    DIAGNOSTICS_DIR
    / 'prototype_ensemble_runs.csv',
    index=False,
    encoding='utf-8-sig'
)

ensemble_pairwise_df.to_csv(
    DIAGNOSTICS_DIR
    / 'prototype_seed_pairwise.csv',
    index=False,
    encoding='utf-8-sig'
)

consensus_k_search_df.to_csv(
    DIAGNOSTICS_DIR
    / 'consensus_k_search.csv',
    index=False,
    encoding='utf-8-sig'
)

pd.DataFrame(
    {
        'segment_idx':
            segdf[
                'segment_idx'
            ].to_numpy()
            if len(
                segdf
            )
            else [],
        'consensus_cluster':
            seg_assign,
        'consensus_confidence':
            seg_conf,
        'consensus_margin':
            seg_margin,
    }
).to_csv(
    DIAGNOSTICS_DIR
    / 'segment_consensus_assignments.csv',
    index=False,
    encoding='utf-8-sig'
)

pd.DataFrame(
    {
        'workflow_idx':
            workflow_df[
                'workflow_idx'
            ].to_numpy()
            if len(
                workflow_df
            )
            else [],
        'workflow_id':
            workflow_df[
                'workflow_id'
            ].astype(
                str
            ).to_numpy()
            if len(
                workflow_df
            )
            else [],
        'consensus_cluster':
            wf_assign,
        'consensus_confidence':
            wf_conf,
        'consensus_margin':
            wf_margin,
    }
).to_csv(
    DIAGNOSTICS_DIR
    / 'workflow_consensus_assignments.csv',
    index=False,
    encoding='utf-8-sig'
)

# Save compact ensemble bundle.
torch.save(
    {
        'version':
            MODEL_VERSION,
        'seeds':
            [
                int(
                    x
                )
                for x in PROTOTYPE_ENSEMBLE_SEEDS
            ],
        'prototype_opt_steps':
            int(
                PROTOTYPE_OPT_STEPS
            ),
        'segment': {
            'consensus_k':
                segment_consensus_k,
            'consensus_summary':
                segment_ensemble[
                    'consensus_summary'
                ],
            'seed_states':
                [
                    {
                        'seed':
                            r[
                                'seed'
                            ],
                        'k':
                            r[
                                'k'
                            ],
                        'state_dict':
                            r[
                                'state_dict'
                            ],
                    }
                    for r in segment_ensemble[
                        'runs'
                    ]
                ],
        },
        'workflow': {
            'consensus_k':
                workflow_consensus_k,
            'consensus_summary':
                workflow_ensemble[
                    'consensus_summary'
                ],
            'seed_states':
                [
                    {
                        'seed':
                            r[
                                'seed'
                            ],
                        'k':
                            r[
                                'k'
                            ],
                        'state_dict':
                            r[
                                'state_dict'
                            ],
                    }
                    for r in workflow_ensemble[
                        'runs'
                    ]
                ],
        },
    },
    OUTPUT_DIR
    / 'prototype_consensus_bundle.pt'
)

print(
    'segment consensus:',
    prototype_stability_diagnostics[
        'segment'
    ]
)

print(
    'workflow consensus:',
    prototype_stability_diagnostics[
        'workflow'
    ]
)

print(
    'segment prototype diagnostics:',
    prototype_diagnostics[
        'segment'
    ]
)

print(
    'workflow prototype diagnostics:',
    prototype_diagnostics[
        'workflow'
    ]
)

# restore reproducible global seed
random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        SEED
    )


segment seed=41 step=001 loss=3.3106 k=32
segment seed=41 step=050 loss=3.3476 k=32
segment seed=41 step=100 loss=3.3922 k=32
segment seed=41 early-stop at 100
segment seed=42 step=001 loss=3.3820 k=32
segment seed=42 step=050 loss=3.4228 k=32
segment seed=42 step=100 loss=3.4463 k=32
segment seed=42 early-stop at 100
segment seed=43 step=001 loss=3.3935 k=32
segment seed=43 step=050 loss=3.3863 k=32
segment seed=43 step=100 loss=3.4148 k=32
segment seed=43 early-stop at 100
segment seed=44 step=001 loss=3.3200 k=32
segment seed=44 step=050 loss=3.3951 k=32
segment seed=44 step=100 loss=3.4215 k=32
segment seed=44 early-stop at 100
segment seed=45 step=001 loss=3.2716 k=32
segment seed=45 step=050 loss=3.3539 k=32
segment seed=45 step=100 loss=3.3899 k=32
segment seed=45 early-stop at 100
workflow seed=41 step=001 loss=2.6437 k=16
workflow seed=41 step=050 loss=2.7538 k=16
workflow seed=41 step=100 loss=2.7674 k=16
workflow seed=41 early-stop at 100
workflow seed=42 step=001 loss=2.683

## 9. Pattern summary / artifacts

In [10]:
# ============================================================
# Pattern summary / readable evidence
# ============================================================
def evidence_for_segment_ids(ids,max_items=8):
    vals=[]
    for sid in ids:
        v=safe_str(segdf.loc[int(sid),'evidence'])
        if v and v not in vals: vals.append(v)
        if len(vals)>=max_items: break
    return ' || '.join(vals)

if len(workflow_df):
    patterns=[]
    for proto,g in workflow_df.groupby('workflow_prototype'):
        rep=g.sort_values('workflow_prototype_confidence',ascending=False).iloc[0]
        patterns.append({
            'workflow_prototype':proto,
            'count':len(g),
            'mean_segment_count':float(g.segment_count.mean()),
            'representative_workflow_id':rep.workflow_id,
            'representative_segment_sequence':rep.segment_prototype_sequence,
            'representative_evidence':evidence_for_segment_ids(workflow_members[int(rep.workflow_idx)])
        })
    pattern_df=pd.DataFrame(patterns).sort_values(['count','workflow_prototype'],ascending=[False,True])
else:
    pattern_df=pd.DataFrame()

pattern_df.to_csv(OUTPUT_DIR/'workflow_patterns.csv',index=False,encoding='utf-8-sig')

with open(OUTPUT_DIR/'latent_workflows.jsonl','w',encoding='utf-8') as f:
    for _,wf in workflow_df.iterrows():
        ids=workflow_members[int(wf.workflow_idx)]
        obj={
            'workflow_id':wf.workflow_id,
            'stream_id':wf.stream_id,
            'workflow_prototype':wf.workflow_prototype,
            'workflow_prototype_confidence':float(wf.workflow_prototype_confidence),
            'start_time':str(wf.start_time),
            'end_time':str(wf.end_time),
            'segment_prototype_sequence':wf.segment_prototype_sequence,
            'segments':[
                {
                    'segment_id':segdf.loc[int(sid),'segment_id'],
                    'segment_prototype':segdf.loc[int(sid),'segment_prototype'],
                    'prototype_confidence':float(segdf.loc[int(sid),'segment_prototype_confidence']),
                    'event_count':int(segdf.loc[int(sid),'event_count']),
                    'evidence':segdf.loc[int(sid),'evidence'],
                } for sid in ids
            ]
        }
        f.write(json.dumps(obj,ensure_ascii=False)+'\n')

lines=[
    '# Deep Unsupervised Workflow Extraction v5.2',
    '',
    '- 특정 메뉴 whitelist: 사용하지 않음',
    '- 조회/저장/신규 keyword rule: 사용하지 않음',
    '- API endpoint 업무 rule: 사용하지 않음',
    '- 수동 step/workflow label: 사용하지 않음',
    '- Event Transformer: V4.2 self-supervised objective retained',
    '- Frozen multilingual semantic encoder: intfloat/multilingual-e5-small',
    '- Learned salience: disabled',
    '- VICReg: disabled',
    '- Prototype training: original V4.2 objective retained',
    '- Boundary: neural representation change distillation',
    '- Segment Transformer: self-supervised hierarchical model',
    '- Latent type: neural prototypes',
    '',
    f'- events: {len(df):,}',
    f'- latent segments: {len(segdf):,}',
    f'- latent workflows: {len(workflow_df):,}',
    ''
]
for _,r in pattern_df.head(30).iterrows():
    lines += [
        f"## Pattern {r.workflow_prototype}",
        f"- count: {int(r['count'])}",
        f"- mean segments: {r.mean_segment_count:.2f}",
        f"- representative sequence: `{r.representative_segment_sequence}`",
        f"- raw evidence: {r.representative_evidence}",
        ''
    ]
(OUTPUT_DIR/'workflow_summary_deep.md').write_text('\n'.join(lines),encoding='utf-8')

# Config / vocab artifacts
for name,v in vocabs.items():
    (OUTPUT_DIR/f'{name}_vocab.json').write_text(json.dumps({'id_to_token':v.itos,'token_to_id':v.stoi},ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'feature_stats.json').write_text(json.dumps({'numeric_mean':mu.tolist(),'numeric_std':sd.tolist(),'event_window':EVENT_WINDOW,'segment_window':SEGMENT_WINDOW,'max_text_bytes':MAX_TEXT_BYTES},indent=2),encoding='utf-8')

print('\\n=== DONE ===')
print('OUTPUT_DIR:',OUTPUT_DIR)
for fn in ['event_boundary_scores.csv','latent_segments.csv','latent_workflows.csv','workflow_patterns.csv','workflow_summary_deep.md']:
    print('-',OUTPUT_DIR/fn)


\n=== DONE ===
OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/data/urangi/artifacts/business_flow_deep_v6_2
- /content/drive/MyDrive/Colab Notebooks/data/urangi/artifacts/business_flow_deep_v6_2/event_boundary_scores.csv
- /content/drive/MyDrive/Colab Notebooks/data/urangi/artifacts/business_flow_deep_v6_2/latent_segments.csv
- /content/drive/MyDrive/Colab Notebooks/data/urangi/artifacts/business_flow_deep_v6_2/latent_workflows.csv
- /content/drive/MyDrive/Colab Notebooks/data/urangi/artifacts/business_flow_deep_v6_2/workflow_patterns.csv
- /content/drive/MyDrive/Colab Notebooks/data/urangi/artifacts/business_flow_deep_v6_2/workflow_summary_deep.md


## 9.5 urangi Evaluation v2 — internal evaluation only

V5.7에서는 모델을 더 바꾸기 전에 평가 체계를 교정합니다.

기존 exact/alias anchor 평가는 과거 버전과의 비교를 위해 `legacy`로 남겨둡니다.
새로운 Evaluation v2는 다음 원칙을 사용합니다.

- page title / page URL 기반 **current context anchor**
- menu/navigation click anchor를 별도 provenance로 분리
- 하나의 instance에 gold가 정확히 하나인 경우만 ARI/NMI/V-measure 계산
- multi-gold instance는 clustering metric에서 제외하고 boundary/co-occurrence 진단에 사용
- anchor sensitivity를 여러 규칙으로 동시에 보고
  한 규칙의 결과만 gold truth처럼 취급하지 않음

Evaluation v2 modes:

```text
legacy_loose
no_terminal_nav
context_ge1              ← primary
context_ge2
context_ge2_dwell3
```

또한 threshold 자체의 영향도 별도로 기록합니다.

```text
workflow_threshold_signal_summary.csv
workflow_cut_jaccard.csv
```

따라서 V5.7에서는 “workflow 개수”와 “cut 위치”를 동시에 확인합니다.

## 10. V6.2 고객 보고서 / Stable Consensus / Evaluation

이 단계에서는 기존 self-supervised / unsupervised 모델 결과를 **실제 화면에서 사용하기 좋은 구조**로 변환합니다.

### V4.2에서 바뀌는 점

1. **Session-safe instance**
   - session을 workflow 정답으로 사용하지 않습니다.
   - 다만 하나의 표시용 실제 실행 사례가 서로 다른 `session_id`를 넘지 않도록 최대 경계로만 사용합니다.
   - 하나의 latent workflow가 여러 session에 걸쳐 있으면 export 단계에서 여러 instance part로 나눕니다.

2. **Consecutive SP compression**
   - `SP09 → SP09 → SP09 → SP22`처럼 연속되는 동일 prototype은 화면용으로
     `SP09 (3회 반복) → SP22` 형태로 압축합니다.
   - 원래 latent 결과는 학습 단계에서 변경하지 않습니다.

3. **Instance step 명시**
   - `process_instance_steps.csv`를 추가합니다.
   - 개발자는 이 파일만으로 특정 실행 사례의 실제 step 순서를 복원할 수 있습니다.

4. **가독성 높은 표시명**
   - `glb-*`, `col_*`, 내부 코드, 행/열 dump, 오류 메시지, 단순 숫자값 등을 표시명 후보에서 우선 제외합니다.
   - 이 로직은 presentation layer이며 모델 학습에는 사용되지 않습니다.

5. **실제 edge graph**
   - `workflow_preview.html`은 단순 `A → B → C`가 아니라 `process_edges.csv`의 실제 topology를 사용합니다.
   - transition ratio 기준을 조절하면서 branch를 확인할 수 있습니다.

6. **입력값 masking 미적용**
   - QA 목적이므로 V4.2에서는 `input_data`를 추가 마스킹하지 않고 그대로 export합니다.
   - 단, 원본 로그 자체에 이미 `[REDACTED]`가 들어있다면 복원하지 않습니다.

### 최종 `main_results`

- `workflow_preview.html`
- `developer_handoff.md`
- `process_models.csv`
- `process_nodes.csv`
- `process_edges.csv`
- `process_instances.csv`
- `process_instance_steps.csv`
- `process_node_inputs.csv`

In [11]:

# ============================================================
# V6.2 Customer-facing stable workflow extraction + customer reporting
# ============================================================

import shutil
from datetime import datetime

V62_VERSION = "v6.2"

# 최종 전달 폴더는 매 실행마다 깨끗하게 다시 생성
if MAIN_RESULTS_DIR.exists():
    shutil.rmtree(MAIN_RESULTS_DIR)
MAIN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# A. Presentation helpers
# ============================================================

GENERIC_DISPLAY_TOKENS = {
    '',
    'click', 'change', 'paste', 'value_reflection',
    'api_transaction', 'api_transaction_error', 'api_transaction_timeout',
    'grid_cell_click', 'grid_context_enrichment',
    'popup_open', 'popup_close', 'popup_structure',
    'page_view', 'page_close', 'collector_boot', 'screen_change',
    'dom_event', 'exbuilder6',
}

TECHNICAL_VALUE_RE = re.compile(
    r'^(?:'
    r'glb-[A-Za-z0-9_-]+|'
    r'col_\d+|'
    r'SP\d+|WP\d+|S\d+|W\d+|'
    r'[A-Z][A-Z0-9_]{2,}|'
    r'[0-9a-f]{12,}'
    r')$',
    flags=re.I
)

ROW_CELL_RE = re.compile(
    r'\d+\s*행\s*\d+\s*열',
    flags=re.I
)

ERROR_TEXT_RE = re.compile(
    r'(ORA-\d+|오류|에러|필수\s*열|필수\s*입력|유효하지\s*않|'
    r'조회된\s*내역이\s*없|ERR-SRV|timeout)',
    flags=re.I
)

PURE_VALUE_RE = re.compile(
    r'^\s*(?:'
    r'[-+]?\d+(?:[.,]\d+)*|'
    r'\d{4}[-./년]\s*\d{1,2}.*|'
    r'[FMYN]|'
    r'true|false'
    r')\s*$',
    flags=re.I
)


def _safe_session(v):
    s = safe_str(v)
    return s if s else '__NO_SESSION__'


def _is_readable_label_candidate(x):
    s = safe_str(x)

    if not s:
        return False

    sl = s.lower()

    if sl in GENERIC_DISPLAY_TOKENS:
        return False

    if len(s) < 2 or len(s) > 55:
        return False

    if TECHNICAL_VALUE_RE.fullmatch(s):
        return False

    if ROW_CELL_RE.search(s):
        return False

    if ERROR_TEXT_RE.search(s):
        return False

    if PURE_VALUE_RE.fullmatch(s):
        return False

    if re.fullmatch(r'[\W_]+', s):
        return False

    if s.startswith('http://') or s.startswith('https://') or s.startswith('/'):
        return False

    return True


def _add_label_candidate(counter, value, weight):
    s = safe_str(value)

    if _is_readable_label_candidate(s):
        counter[s] += float(weight)


def _label_counter_for_events(ev):
    """
    화면 표시용 label 후보.
    associated_label을 가장 우선하고 element_text를 보조로 사용.
    input_data 자체는 node 이름 후보로 사용하지 않는다.
    """
    c = Counter()

    if 'associated_label' in ev.columns:
        for v in ev['associated_label'].tolist():
            _add_label_candidate(c, v, 5.0)

    if 'element_text' in ev.columns:
        for v in ev['element_text'].tolist():
            _add_label_candidate(c, v, 2.0)

    # 클릭된 메뉴/버튼 text가 의미 있는 경우 약간 가중
    if (
        'interaction_type' in ev.columns
        and 'element_text' in ev.columns
    ):
        for _, r in ev.iterrows():
            it = safe_str(r.get('interaction_type')).lower()

            if it in {
                'click',
                'grid_cell_click',
                'popup_open'
            }:
                _add_label_candidate(
                    c,
                    r.get('element_text'),
                    1.5
                )

    return c


def _pick_best_label(counter, fallback):
    if not counter:
        return fallback

    ranked = sorted(
        counter.items(),
        key=lambda kv: (
            kv[1],
            1 if re.search(r'[가-힣]', kv[0]) else 0,
            min(len(kv[0]), 30)
        ),
        reverse=True
    )

    return ranked[0][0] if ranked else fallback


def _segment_events(segment_idx):
    r = seg_by_idx.loc[int(segment_idx)]

    st = int(r['start_global_idx'])
    en = int(r['end_global_idx'])

    ev = df.iloc[
        st:en + 1
    ].copy()

    if 'stream_id' in ev.columns:
        ev = ev[
            ev['stream_id'].astype(str)
            == safe_str(r['stream_id'])
        ]

    ev = ev.sort_values(
        [
            'event_time_parsed',
            'global_event_idx'
        ],
        kind='stable'
    )

    return ev


def _source_redacted(raw_value):
    s = safe_str(raw_value).upper()

    return int(
        '[REDACTED]' in s
        or '[IDENTITY_CONTEXT]' in s
    )


def _input_kind(interaction_type):
    t = safe_str(interaction_type).lower()

    if t in {'change', 'paste'}:
        return 'direct_input'

    if t in {
        'click',
        'grid_cell_click'
    }:
        return 'selection'

    if t == 'value_reflection':
        return 'reflected_value'

    return 'observed_value'


# ============================================================
# B. Segment / workflow source maps
# ============================================================

seg_by_idx = segdf.set_index(
    'segment_idx',
    drop=False
)

workflow_member_map = {}

for pos, (_, wr) in enumerate(
    workflow_df.iterrows()
):
    widx = int(
        wr['workflow_idx']
    )

    if pos < len(workflow_members):
        workflow_member_map[widx] = np.asarray(
            workflow_members[pos],
            dtype=int
        )

    else:
        start_seg = int(
            wr['segment_start']
        )

        end_seg = int(
            wr['segment_end']
        )

        cand = segdf[
            (
                segdf['stream_id'].astype(str)
                == safe_str(wr['stream_id'])
            )
            & (
                segdf['segment_idx']
                >= start_seg
            )
            & (
                segdf['segment_idx']
                <= end_seg
            )
        ].sort_values(
            'start_global_idx'
        )

        workflow_member_map[widx] = (
            cand['segment_idx']
            .astype(int)
            .to_numpy()
        )


# ============================================================
# C. Segment를 session-safe fragment로 확장
# ============================================================

segment_fragments = {}

for seg_idx, sr in seg_by_idx.iterrows():
    ev = _segment_events(
        seg_idx
    )

    if len(ev) == 0:
        segment_fragments[
            int(seg_idx)
        ] = []
        continue

    # session_id가 없으면 하나의 fragment로 취급
    if 'session_id' not in ev.columns:
        ev = ev.copy()
        ev['_v42_session'] = '__NO_SESSION__'

    else:
        ev = ev.copy()
        ev['_v42_session'] = ev[
            'session_id'
        ].map(
            _safe_session
        )

    # segment 내부에서도 session 전환이 있으면 contiguous fragment로 나눔
    session_vals = ev[
        '_v42_session'
    ].tolist()

    run_ids = []
    run_no = -1
    prev = None

    for s in session_vals:
        if prev is None or s != prev:
            run_no += 1
        run_ids.append(run_no)
        prev = s

    ev['_v42_session_run'] = run_ids

    fragments = []

    for frag_no, (_, frag_ev) in enumerate(
        ev.groupby(
            '_v42_session_run',
            sort=False
        ),
        start=1
    ):
        frag_ev = frag_ev.copy()

        session_id = safe_str(
            frag_ev[
                '_v42_session'
            ].iloc[0]
        )

        label_counter = _label_counter_for_events(
            frag_ev
        )

        sp = (
            safe_str(
                sr.get(
                    'segment_prototype'
                )
            )
            or 'SP_UNASSIGNED'
        )

        display_label = _pick_best_label(
            label_counter,
            sp
        )

        inputs = []

        if 'input_data' in frag_ev.columns:
            for _, er in frag_ev.iterrows():
                raw_value = safe_str(
                    er.get(
                        'input_data'
                    )
                )

                if not raw_value:
                    continue

                label = (
                    safe_str(
                        er.get(
                            'associated_label'
                        )
                    )
                    or safe_str(
                        er.get(
                            'element_text'
                        )
                    )
                    or safe_str(
                        er.get(
                            'data_testid'
                        )
                    )
                    or '입력값'
                )

                # V5.2 QA 정책:
                # 추가 masking을 적용하지 않고 원본 input_data를 그대로 저장.
                inputs.append(
                    {
                        'event_id':
                            safe_str(
                                er.get(
                                    'event_id'
                                )
                            )
                            or safe_str(
                                er.get(
                                    'id'
                                )
                            ),
                        'event_time':
                            er.get(
                                'event_time_parsed',
                                er.get(
                                    'event_time'
                                )
                            ),
                        'interaction_type':
                            safe_str(
                                er.get(
                                    'interaction_type'
                                )
                            ),
                        'input_kind':
                            _input_kind(
                                er.get(
                                    'interaction_type'
                                )
                            ),
                        'input_label':
                            label,
                        'input_value':
                            raw_value,
                        'is_direct_user_input':
                            int(
                                _input_kind(
                                    er.get(
                                        'interaction_type'
                                    )
                                )
                                == 'direct_input'
                            ),
                        'is_source_redacted':
                            _source_redacted(
                                raw_value
                            ),
                    }
                )

        fragments.append(
            {
                'segment_idx':
                    int(seg_idx),
                'segment_id':
                    safe_str(
                        sr.get(
                            'segment_id'
                        )
                    ),
                'segment_prototype':
                    sp,
                'session_id':
                    session_id,
                'fragment_no':
                    frag_no,
                'start_time':
                    frag_ev[
                        'event_time_parsed'
                    ].min(),
                'end_time':
                    frag_ev[
                        'event_time_parsed'
                    ].max(),
                'event_count':
                    int(
                        len(
                            frag_ev
                        )
                    ),
                'prototype_confidence':
                    float(
                        sr.get(
                            'segment_prototype_confidence',
                            0
                        )
                        or 0
                    ),
                'display_label':
                    display_label,
                'label_counter':
                    label_counter,
                'evidence':
                    safe_str(
                        sr.get(
                            'evidence'
                        )
                    ),
                'inputs':
                    inputs,
            }
        )

    segment_fragments[
        int(seg_idx)
    ] = fragments


# ============================================================
# D. Latent workflow -> session-safe export instances
# ============================================================

export_instances_source = []

for _, wr in workflow_df.sort_values(
    'workflow_idx'
).iterrows():

    widx = int(
        wr['workflow_idx']
    )

    workflow_id = safe_str(
        wr['workflow_id']
    )

    wp = (
        safe_str(
            wr.get(
                'workflow_prototype'
            )
        )
        or 'WP_UNASSIGNED'
    )

    model_id = (
        f'{DATASET_NAME}:{wp}'
    )

    seg_ids = workflow_member_map.get(
        widx,
        np.array(
            [],
            dtype=int
        )
    )

    fragment_seq = []

    for seg_idx in seg_ids:
        fragment_seq.extend(
            segment_fragments.get(
                int(seg_idx),
                []
            )
        )

    if not fragment_seq:
        continue

    # session transition을 최대 경계로 사용
    parts = []
    current = []
    current_session = None

    for frag in fragment_seq:
        sid = frag[
            'session_id'
        ]

        if (
            current
            and sid != current_session
        ):
            parts.append(
                current
            )
            current = []

        if not current:
            current_session = sid

        current.append(
            frag
        )

    if current:
        parts.append(
            current
        )

    for part_no, part in enumerate(
        parts,
        start=1
    ):
        export_instances_source.append(
            {
                'source_workflow_idx':
                    widx,
                'source_workflow_id':
                    workflow_id,
                'workflow_prototype':
                    wp,
                'model_id':
                    model_id,
                'stream_id':
                    safe_str(
                        wr.get(
                            'stream_id'
                        )
                    ),
                'session_id':
                    part[0][
                        'session_id'
                    ],
                'session_part':
                    part_no,
                'fragments':
                    part,
                'source_workflow_prototype_confidence':
                    float(
                        wr.get(
                            'workflow_prototype_confidence',
                            0
                        )
                        or 0
                    ),
            }
        )


# ============================================================
# E. Consecutive identical SP compression
# ============================================================

def _compress_fragment_sequence(fragments):
    steps = []

    for frag in fragments:
        sp = frag[
            'segment_prototype'
        ]

        if (
            steps
            and steps[-1][
                'segment_prototype'
            ]
            == sp
        ):
            step = steps[-1]

            step[
                'fragments'
            ].append(
                frag
            )

            step[
                'repeat_count'
            ] += 1

            step[
                'event_count'
            ] += frag[
                'event_count'
            ]

            step[
                'end_time'
            ] = max(
                step[
                    'end_time'
                ],
                frag[
                    'end_time'
                ]
            )

            step[
                'prototype_confidences'
            ].append(
                frag[
                    'prototype_confidence'
                ]
            )

            step[
                'label_counter'
            ].update(
                frag[
                    'label_counter'
                ]
            )

            if frag[
                'evidence'
            ]:
                step[
                    'evidence_counter'
                ][
                    frag[
                        'evidence'
                    ]
                ] += 1

            step[
                'inputs'
            ].extend(
                frag[
                    'inputs'
                ]
            )

        else:
            ec = Counter()

            if frag[
                'evidence'
            ]:
                ec[
                    frag[
                        'evidence'
                    ]
                ] += 1

            steps.append(
                {
                    'segment_prototype':
                        sp,
                    'fragments':
                        [
                            frag
                        ],
                    'repeat_count':
                        1,
                    'start_time':
                        frag[
                            'start_time'
                        ],
                    'end_time':
                        frag[
                            'end_time'
                        ],
                    'event_count':
                        frag[
                            'event_count'
                        ],
                    'prototype_confidences':
                        [
                            frag[
                                'prototype_confidence'
                            ]
                        ],
                    'label_counter':
                        Counter(
                            frag[
                                'label_counter'
                            ]
                        ),
                    'evidence_counter':
                        ec,
                    'inputs':
                        list(
                            frag[
                                'inputs'
                            ]
                        ),
                }
            )

    return steps


# ============================================================
# F. Build process instances / steps / inputs / graph stats
# ============================================================

instance_rows = []
instance_step_rows = []
node_input_rows = []

node_stats = defaultdict(
    lambda: {
        'step_count':
            0,
        'source_segment_count':
            0,
        'instance_ids':
            set(),
        'positions':
            [],
        'event_counts':
            [],
        'repeat_counts':
            [],
        'confidences':
            [],
        'label_counter':
            Counter(),
        'evidence_counter':
            Counter(),
    }
)

edge_counts = Counter()
outgoing_counts = Counter()

input_seq = 0
instance_seq = 0
instance_step_seq = 0


for inst_src in export_instances_source:
    instance_seq += 1

    wp = inst_src[
        'workflow_prototype'
    ]

    model_id = inst_src[
        'model_id'
    ]

    source_workflow_id = inst_src[
        'source_workflow_id'
    ]

    instance_id = (
        f'{DATASET_NAME}:'
        f'{source_workflow_id}:'
        f'P{inst_src["session_part"]:02d}'
    )

    steps = _compress_fragment_sequence(
        inst_src[
            'fragments'
        ]
    )

    prev_node_id = None
    instance_input_count = 0
    source_segment_ids_all = []

    for step_order, step in enumerate(
        steps,
        start=1
    ):
        instance_step_seq += 1

        sp = step[
            'segment_prototype'
        ]

        node_id = (
            f'{model_id}:{sp}'
        )

        instance_step_id = (
            f'IS{instance_step_seq:08d}'
        )

        source_segment_ids = []

        for frag in step[
            'fragments'
        ]:
            sid = frag[
                'segment_id'
            ]

            if (
                sid
                and sid not in source_segment_ids
            ):
                source_segment_ids.append(
                    sid
                )

        source_segment_ids_all.extend(
            source_segment_ids
        )

        display_label = _pick_best_label(
            step[
                'label_counter'
            ],
            sp
        )

        representative_evidence = ''

        if step[
            'evidence_counter'
        ]:
            representative_evidence = (
                step[
                    'evidence_counter'
                ]
                .most_common(1)[0][0]
            )

        avg_seg_conf = (
            float(
                np.mean(
                    step[
                        'prototype_confidences'
                    ]
                )
            )
            if step[
                'prototype_confidences'
            ]
            else 0.0
        )

        # ----------------------------------------------------
        # instance step
        # ----------------------------------------------------
        instance_step_rows.append(
            {
                'instance_step_id':
                    instance_step_id,
                'instance_id':
                    instance_id,
                'model_id':
                    model_id,
                'source_workflow_id':
                    source_workflow_id,
                'session_id':
                    inst_src[
                        'session_id'
                    ],
                'step_order':
                    step_order,
                'node_id':
                    node_id,
                'segment_prototype':
                    sp,
                'display_label':
                    display_label,
                'repeat_count':
                    int(
                        step[
                            'repeat_count'
                        ]
                    ),
                'source_segment_ids':
                    '|'.join(
                        source_segment_ids
                    ),
                'start_time':
                    step[
                        'start_time'
                    ],
                'end_time':
                    step[
                        'end_time'
                    ],
                'event_count':
                    int(
                        step[
                            'event_count'
                        ]
                    ),
                'avg_segment_prototype_confidence':
                    avg_seg_conf,
                'input_count':
                    int(
                        len(
                            step[
                                'inputs'
                            ]
                        )
                    ),
                'representative_evidence':
                    representative_evidence,
            }
        )

        # ----------------------------------------------------
        # graph/node aggregate
        # ----------------------------------------------------
        stat = node_stats[
            (
                model_id,
                sp
            )
        ]

        stat[
            'step_count'
        ] += 1

        stat[
            'source_segment_count'
        ] += len(
            source_segment_ids
        )

        stat[
            'instance_ids'
        ].add(
            instance_id
        )

        stat[
            'positions'
        ].append(
            step_order
        )

        stat[
            'event_counts'
        ].append(
            step[
                'event_count'
            ]
        )

        stat[
            'repeat_counts'
        ].append(
            step[
                'repeat_count'
            ]
        )

        stat[
            'confidences'
        ].append(
            avg_seg_conf
        )

        stat[
            'label_counter'
        ].update(
            step[
                'label_counter'
            ]
        )

        if representative_evidence:
            stat[
                'evidence_counter'
            ][
                representative_evidence
            ] += 1

        # consecutive same SP는 이미 압축됐으므로 self-loop가 생기지 않음
        if prev_node_id is not None:
            edge_counts[
                (
                    model_id,
                    prev_node_id,
                    node_id
                )
            ] += 1

            outgoing_counts[
                (
                    model_id,
                    prev_node_id
                )
            ] += 1

        prev_node_id = node_id

        # ----------------------------------------------------
        # inputs - QA용 원본 값 유지
        # ----------------------------------------------------
        seen_in_step = set()

        for inp in step[
            'inputs'
        ]:
            dedup_key = (
                inp[
                    'event_id'
                ],
                inp[
                    'input_label'
                ],
                inp[
                    'input_value'
                ],
                inp[
                    'input_kind'
                ],
            )

            if dedup_key in seen_in_step:
                continue

            seen_in_step.add(
                dedup_key
            )

            input_seq += 1
            instance_input_count += 1

            node_input_rows.append(
                {
                    'input_id':
                        f'I{input_seq:08d}',
                    'instance_id':
                        instance_id,
                    'instance_step_id':
                        instance_step_id,
                    'model_id':
                        model_id,
                    'node_id':
                        node_id,
                    'source_workflow_id':
                        source_workflow_id,
                    'session_id':
                        inst_src[
                            'session_id'
                        ],
                    'step_order':
                        step_order,
                    'segment_prototype':
                        sp,
                    'source_segment_ids':
                        '|'.join(
                            source_segment_ids
                        ),
                    'event_id':
                        inp[
                            'event_id'
                        ],
                    'event_time':
                        inp[
                            'event_time'
                        ],
                    'interaction_type':
                        inp[
                            'interaction_type'
                        ],
                    'input_kind':
                        inp[
                            'input_kind'
                        ],
                    'input_label':
                        inp[
                            'input_label'
                        ],
                    'input_value':
                        inp[
                            'input_value'
                        ],
                    'is_direct_user_input':
                        inp[
                            'is_direct_user_input'
                        ],
                    'is_source_redacted':
                        inp[
                            'is_source_redacted'
                        ],
                }
            )

    start_time = min(
        frag[
            'start_time'
        ]
        for frag in inst_src[
            'fragments'
        ]
    )

    end_time = max(
        frag[
            'end_time'
        ]
        for frag in inst_src[
            'fragments'
        ]
    )

    instance_rows.append(
        {
            'instance_id':
                instance_id,
            'model_id':
                model_id,
            'source_workflow_id':
                source_workflow_id,
            'workflow_prototype':
                wp,
            'stream_id':
                inst_src[
                    'stream_id'
                ],
            'session_id':
                inst_src[
                    'session_id'
                ],
            'session_part':
                int(
                    inst_src[
                        'session_part'
                    ]
                ),
            'start_time':
                start_time,
            'end_time':
                end_time,
            'compressed_step_count':
                int(
                    len(
                        steps
                    )
                ),
            'source_segment_count':
                int(
                    len(
                        set(
                            source_segment_ids_all
                        )
                    )
                ),
            'source_workflow_prototype_confidence':
                inst_src[
                    'source_workflow_prototype_confidence'
                ],
            'input_count':
                int(
                    instance_input_count
                ),
        }
    )


process_instances = pd.DataFrame(
    instance_rows
)

process_instance_steps = pd.DataFrame(
    instance_step_rows
)

process_node_inputs = pd.DataFrame(
    node_input_rows
)


# ============================================================
# G. V6.2 Attach session.user_id to extracted process instances
#    and build user trajectory episodes.
# ============================================================

def _stable_user_display_id(
    raw_user_id
):
    raw=safe_str(
        raw_user_id
    )

    if not raw:
        return ''

    return (
        'USER-'
        + hashlib.sha1(
            raw.encode(
                'utf-8',
                errors='ignore'
            )
        ).hexdigest()[
            :8
        ].upper()
    )


if len(
    process_instances
):
    process_instances[
        'trajectory_user_id'
    ]=''

    if 'session_id' in process_instances.columns:
        process_instances[
            'trajectory_user_id'
        ]=(
            process_instances[
                'session_id'
            ]
            .map(
                safe_str
            )
            .map(
                session_user_map
            )
            .fillna(
                ''
            )
            .map(
                safe_str
            )
        )

    # session mapping이 없는 instance는 stream_id가 명시적 actor identity인 경우만 fallback.
    _fallback_mask=(
        process_instances[
            'trajectory_user_id'
        ]==''
    )

    if 'stream_id' in process_instances.columns:
        _stream_fallback=process_instances[
            'stream_id'
        ].map(
            safe_str
        )

        _stream_fallback=_stream_fallback.where(
            ~_stream_fallback.str.startswith(
                'session:',
                na=False
            ),
            ''
        )

        process_instances.loc[
            _fallback_mask,
            'trajectory_user_id'
        ]=_stream_fallback.loc[
            _fallback_mask
        ]

    process_instances[
        'trajectory_user_display_id'
    ]=process_instances[
        'trajectory_user_id'
    ].map(
        _stable_user_display_id
    )

else:
    process_instances[
        'trajectory_user_id'
    ]=[]

    process_instances[
        'trajectory_user_display_id'
    ]=[]


def _build_user_trajectory_instances(
    instances
):
    if (
        not USE_USER_TRAJECTORY
        or not len(
            instances
        )
    ):
        return pd.DataFrame(
            columns=list(
                instances.columns
            )
            + [
                'trajectory_id',
                'trajectory_order',
                'gap_from_prev_min',
            ]
        )

    work=instances[
        instances[
            'trajectory_user_id'
        ].map(
            safe_str
        )!=''
    ].copy()

    if not len(
        work
    ):
        return pd.DataFrame(
            columns=list(
                instances.columns
            )
            + [
                'trajectory_id',
                'trajectory_order',
                'gap_from_prev_min',
            ]
        )

    work[
        '_start'
    ]=pd.to_datetime(
        work[
            'start_time'
        ],
        errors='coerce'
    )

    work[
        '_end'
    ]=pd.to_datetime(
        work[
            'end_time'
        ],
        errors='coerce'
    )

    work=work.sort_values(
        [
            'trajectory_user_id',
            '_start',
            '_end',
            'instance_id',
        ],
        kind='stable'
    ).reset_index(
        drop=True
    )

    out=[]
    trajectory_seq=0

    for user_id,g in work.groupby(
        'trajectory_user_id',
        sort=False
    ):
        g=g.sort_values(
            [
                '_start',
                '_end',
                'instance_id',
            ],
            kind='stable'
        )

        current_id=None
        order=0
        prev_end=None

        for _,r in g.iterrows():
            start=r[
                '_start'
            ]

            end=r[
                '_end'
            ]

            if pd.isna(
                start
            ):
                start=end

            if pd.isna(
                end
            ):
                end=start

            if (
                prev_end is None
                or pd.isna(
                    prev_end
                )
                or pd.isna(
                    start
                )
            ):
                gap_min=None
                new_episode=True

            else:
                gap_min=max(
                    0.0,
                    (
                        start-prev_end
                    ).total_seconds()
                    / 60.0
                )

                date_changed=(
                    bool(
                        start.date()
                        != prev_end.date()
                    )
                    if (
                        USER_TRAJECTORY_BREAK_ON_DATE_CHANGE
                        and pd.notna(
                            start
                        )
                        and pd.notna(
                            prev_end
                        )
                    )
                    else False
                )

                new_episode=(
                    gap_min
                    > USER_TRAJECTORY_GAP_MINUTES
                    or date_changed
                )

            if (
                current_id is None
                or new_episode
            ):
                trajectory_seq+=1

                current_id=(
                    f'UT{trajectory_seq:06d}'
                )

                order=1

            else:
                order+=1

            row=r.drop(
                labels=[
                    '_start',
                    '_end',
                ]
            ).to_dict()

            row.update(
                {
                    'trajectory_id':
                        current_id,
                    'trajectory_order':
                        int(
                            order
                        ),
                    'gap_from_prev_min':
                        (
                            None
                            if gap_min is None
                            else float(
                                gap_min
                            )
                        ),
                }
            )

            out.append(
                row
            )

            if pd.notna(
                end
            ):
                prev_end=end

    return pd.DataFrame(
        out
    )


user_trajectory_instances=(
    _build_user_trajectory_instances(
        process_instances
    )
)


user_trajectory_summary_rows=[]

if len(
    user_trajectory_instances
):
    for trajectory_id,g in user_trajectory_instances.groupby(
        'trajectory_id',
        sort=False
    ):
        g=g.sort_values(
            'trajectory_order'
        )

        session_ids=[
            safe_str(
                x
            )
            for x in g[
                'session_id'
            ].tolist()
            if safe_str(
                x
            )
        ]

        model_ids=[
            safe_str(
                x
            )
            for x in g[
                'model_id'
            ].tolist()
            if safe_str(
                x
            )
        ]

        wps=[
            safe_str(
                x
            )
            for x in g[
                'workflow_prototype'
            ].tolist()
            if safe_str(
                x
            )
        ]

        user_trajectory_summary_rows.append(
            {
                'trajectory_id':
                    trajectory_id,
                'trajectory_user_id':
                    safe_str(
                        g[
                            'trajectory_user_id'
                        ].iloc[
                            0
                        ]
                    ),
                'trajectory_user_display_id':
                    safe_str(
                        g[
                            'trajectory_user_display_id'
                        ].iloc[
                            0
                        ]
                    ),
                'start_time':
                    pd.to_datetime(
                        g[
                            'start_time'
                        ],
                        errors='coerce'
                    ).min(),
                'end_time':
                    pd.to_datetime(
                        g[
                            'end_time'
                        ],
                        errors='coerce'
                    ).max(),
                'instance_count':
                    int(
                        len(
                            g
                        )
                    ),
                'session_count':
                    int(
                        len(
                            set(
                                session_ids
                            )
                        )
                    ),
                'is_cross_session':
                    int(
                        len(
                            set(
                                session_ids
                            )
                        )>1
                    ),
                'model_sequence':
                    ' → '.join(
                        model_ids
                    ),
                'workflow_prototype_sequence':
                    ' → '.join(
                        wps
                    ),
                'instance_ids':
                    ' | '.join(
                        g[
                            'instance_id'
                        ].map(
                            safe_str
                        ).tolist()
                    ),
            }
        )


user_trajectory_summary=pd.DataFrame(
    user_trajectory_summary_rows
)

print(
    "user trajectory users:",
    int(
        user_trajectory_instances[
            'trajectory_user_id'
        ].nunique()
    )
    if len(
        user_trajectory_instances
    )
    else 0
)

print(
    "user trajectory episodes:",
    len(
        user_trajectory_summary
    )
)

print(
    "cross-session trajectory episodes:",
    int(
        user_trajectory_summary[
            'is_cross_session'
        ].sum()
    )
    if len(
        user_trajectory_summary
    )
    else 0
)


# ============================================================
# H. Build process_nodes
# ============================================================

node_rows = []

for (
    model_id,
    sp
), stat in node_stats.items():

    node_id = (
        f'{model_id}:{sp}'
    )

    display_label = _pick_best_label(
        stat[
            'label_counter'
        ],
        sp
    )

    representative_evidence = ''

    if stat[
        'evidence_counter'
    ]:
        representative_evidence = (
            stat[
                'evidence_counter'
            ]
            .most_common(1)[0][0]
        )

    node_inputs = (
        process_node_inputs[
            process_node_inputs[
                'node_id'
            ]
            == node_id
        ]
        if len(
            process_node_inputs
        )
        else pd.DataFrame()
    )

    node_rows.append(
        {
            'node_id':
                node_id,
            'model_id':
                model_id,
            'segment_prototype':
                sp,
            'display_label':
                display_label,
            'support_step_count':
                int(
                    stat[
                        'step_count'
                    ]
                ),
            'support_segment_count':
                int(
                    stat[
                        'source_segment_count'
                    ]
                ),
            'support_instance_count':
                int(
                    len(
                        stat[
                            'instance_ids'
                        ]
                    )
                ),
            'avg_step_position':
                float(
                    np.mean(
                        stat[
                            'positions'
                        ]
                    )
                )
                if stat[
                    'positions'
                ]
                else 0.0,
            'avg_event_count':
                float(
                    np.mean(
                        stat[
                            'event_counts'
                        ]
                    )
                )
                if stat[
                    'event_counts'
                ]
                else 0.0,
            'avg_repeat_count':
                float(
                    np.mean(
                        stat[
                            'repeat_counts'
                        ]
                    )
                )
                if stat[
                    'repeat_counts'
                ]
                else 1.0,
            'avg_prototype_confidence':
                float(
                    np.mean(
                        stat[
                            'confidences'
                        ]
                    )
                )
                if stat[
                    'confidences'
                ]
                else 0.0,
            'input_count':
                int(
                    len(
                        node_inputs
                    )
                ),
            'direct_input_count':
                int(
                    node_inputs[
                        'is_direct_user_input'
                    ].sum()
                )
                if len(
                    node_inputs
                )
                else 0,
            'representative_evidence':
                representative_evidence,
        }
    )


process_nodes = pd.DataFrame(
    node_rows
)

if len(
    process_nodes
):
    process_nodes = (
        process_nodes
        .sort_values(
            [
                'model_id',
                'avg_step_position',
                'support_instance_count'
            ],
            ascending=[
                True,
                True,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )

    process_nodes[
        'sort_order'
    ] = (
        process_nodes
        .groupby(
            'model_id'
        )
        .cumcount()
        + 1
    )


# ============================================================
# H. Build process_edges from compressed instance sequence
# ============================================================

edge_rows = []

for edge_no, (
    (
        model_id,
        from_node_id,
        to_node_id
    ),
    cnt
) in enumerate(
    sorted(
        edge_counts.items()
    ),
    start=1
):

    denom = max(
        outgoing_counts[
            (
                model_id,
                from_node_id
            )
        ],
        1
    )

    edge_rows.append(
        {
            'edge_id':
                f'E{edge_no:08d}',
            'model_id':
                model_id,
            'from_node_id':
                from_node_id,
            'to_node_id':
                to_node_id,
            'transition_count':
                int(
                    cnt
                ),
            'transition_ratio':
                float(
                    cnt
                    / denom
                ),
            'is_repeat':
                int(
                    from_node_id
                    == to_node_id
                ),
        }
    )


process_edges = pd.DataFrame(
    edge_rows
)


# ============================================================
# I. Build process_models with cleaner display names
# ============================================================

model_rows = []

if len(
    process_instances
):
    for model_id, g in process_instances.groupby(
        'model_id',
        sort=False
    ):

        wp = safe_str(
            g[
                'workflow_prototype'
            ].iloc[0]
        )

        ng = (
            process_nodes[
                process_nodes[
                    'model_id'
                ]
                == model_id
            ]
            if len(
                process_nodes
            )
            else pd.DataFrame()
        )

        eg = (
            process_edges[
                process_edges[
                    'model_id'
                ]
                == model_id
            ]
            if len(
                process_edges
            )
            else pd.DataFrame()
        )

        label_candidates = []

        if len(
            ng
        ):
            ranked_nodes = ng.sort_values(
                [
                    'support_instance_count',
                    'support_step_count',
                    'avg_step_position'
                ],
                ascending=[
                    False,
                    False,
                    True
                ]
            )

            for label in ranked_nodes[
                'display_label'
            ].tolist():

                label = safe_str(
                    label
                )

                if (
                    _is_readable_label_candidate(
                        label
                    )
                    and label not in label_candidates
                ):
                    label_candidates.append(
                        label
                    )

                if len(
                    label_candidates
                ) >= 3:
                    break

        display_name = wp

        if label_candidates:
            display_name = (
                f'{wp} · '
                + ' / '.join(
                    label_candidates
                )
            )

        model_inputs = (
            process_node_inputs[
                process_node_inputs[
                    'model_id'
                ]
                == model_id
            ]
            if len(
                process_node_inputs
            )
            else pd.DataFrame()
        )

        model_steps = (
            process_instance_steps[
                process_instance_steps[
                    'model_id'
                ]
                == model_id
            ]
            if len(
                process_instance_steps
            )
            else pd.DataFrame()
        )

        model_rows.append(
            {
                'model_id':
                    model_id,
                'dataset_name':
                    DATASET_NAME,
                'workflow_prototype':
                    wp,
                'display_name':
                    display_name,
                'instance_count':
                    int(
                        len(
                            g
                        )
                    ),
                'node_count':
                    int(
                        len(
                            ng
                        )
                    ),
                'edge_count':
                    int(
                        len(
                            eg
                        )
                    ),
                'avg_compressed_step_count':
                    float(
                        g[
                            'compressed_step_count'
                        ].mean()
                    ),
                'avg_source_segment_count':
                    float(
                        g[
                            'source_segment_count'
                        ].mean()
                    ),
                'avg_prototype_confidence':
                    float(
                        g[
                            'source_workflow_prototype_confidence'
                        ].mean()
                    ),
                'input_count':
                    int(
                        len(
                            model_inputs
                        )
                    ),
                'direct_input_count':
                    int(
                        model_inputs[
                            'is_direct_user_input'
                        ].sum()
                    )
                    if len(
                        model_inputs
                    )
                    else 0,
                'first_seen':
                    pd.to_datetime(
                        g[
                            'start_time'
                        ],
                        errors='coerce'
                    ).min(),
                'last_seen':
                    pd.to_datetime(
                        g[
                            'end_time'
                        ],
                        errors='coerce'
                    ).max(),
            }
        )


process_models = pd.DataFrame(
    model_rows
)

if len(
    process_models
):
    process_models = (
        process_models
        .sort_values(
            [
                'instance_count',
                'input_count'
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# J. V6 Customer Reportability Scoring
# ============================================================

def _clip01(
    x
):
    try:
        x=float(
            x
        )

        if not np.isfinite(
            x
        ):
            return 0.0

        return float(
            min(
                1.0,
                max(
                    0.0,
                    x
                )
            )
        )

    except Exception:
        return 0.0


def _model_modal_path(
    model_id
):
    g=process_instance_steps[
        process_instance_steps[
            'model_id'
        ]==model_id
    ] if len(
        process_instance_steps
    ) else pd.DataFrame()

    if not len(
        g
    ):
        return (
            [],
            0.0
        )

    seqs=[]

    for _,ig in g.groupby(
        'instance_id',
        sort=False
    ):
        ig=ig.sort_values(
            'step_order'
        )

        seq=tuple(
            safe_str(
                x
            )
            for x in ig[
                'display_label'
            ].tolist()
            if safe_str(
                x
            )
        )

        if seq:
            seqs.append(
                seq
            )

    if not seqs:
        return (
            [],
            0.0
        )

    counter=Counter(
        seqs
    )

    modal,count=counter.most_common(
        1
    )[
        0
    ]

    consistency=(
        count
        / len(
            seqs
        )
    )

    return (
        list(
            modal
        ),
        float(
            consistency
        )
    )


def _structural_coherence_for_model(
    model_id
):
    ng=process_nodes[
        process_nodes[
            'model_id'
        ]==model_id
    ] if len(
        process_nodes
    ) else pd.DataFrame()

    if not len(
        ng
    ):
        return 0.0

    families=[]

    for r in ng.itertuples(
        index=False
    ):
        text=safe_str(
            getattr(
                r,
                'representative_evidence',
                ''
            )
        )

        paths=re.findall(
            r'/[A-Za-z0-9_.-]+(?:/[A-Za-z0-9_.-]+)+',
            text
        )

        weight=max(
            int(
                getattr(
                    r,
                    'support_step_count',
                    1
                )
            ),
            1
        )

        for p in paths:
            family=structural_api_family(
                p
            )

            if family:
                families.extend(
                    [
                        family
                    ]*min(
                        weight,
                        20
                    )
                )

    if not families:
        return 0.5

    counts=np.asarray(
        list(
            Counter(
                families
            ).values()
        ),
        dtype=float
    )

    p=counts/counts.sum()

    if len(
        p
    )<=1:
        return 1.0

    entropy=-float(
        (
            p*np.log(
                p
            )
        ).sum()
        / math.log(
            len(
                p
            )
        )
    )

    return _clip01(
        1.0-entropy
    )


def _evidence_richness_for_model(
    model_id
):
    ng=process_nodes[
        process_nodes[
            'model_id'
        ]==model_id
    ] if len(
        process_nodes
    ) else pd.DataFrame()

    if not len(
        ng
    ):
        return 0.0

    readable=0

    for r in ng.itertuples(
        index=False
    ):
        if (
            safe_str(
                getattr(
                    r,
                    'representative_evidence',
                    ''
                )
            )
            or _is_readable_label_candidate(
                safe_str(
                    getattr(
                        r,
                        'display_label',
                        ''
                    )
                )
            )
        ):
            readable+=1

    node_evidence=readable/max(
        len(
            ng
        ),
        1
    )

    inp=process_node_inputs[
        process_node_inputs[
            'model_id'
        ]==model_id
    ] if len(
        process_node_inputs
    ) else pd.DataFrame()

    input_presence=min(
        1.0,
        len(
            inp
        )/max(
            len(
                ng
            )*2,
            1
        )
    )

    return _clip01(
        .7*node_evidence
        + .3*input_presence
    )


def _customer_name(
    row,
    rank
):
    name=safe_str(
        row.get(
            'display_name',
            ''
        )
    )

    if '·' in name:
        name=name.split(
            '·',
            1
        )[
            1
        ].strip()

    labels=[
        x.strip()
        for x in name.split(
            '/'
        )
        if _is_readable_label_candidate(
            x.strip()
        )
    ]

    generic={
        '조회',
        '저장',
        '신규',
        '삭제',
        '닫기',
    }

    labels=[
        x
        for x in labels
        if x not in generic
    ]

    if labels:
        return ' · '.join(
            labels[
                :2
            ]
        )

    return f'업무 흐름 {rank:02d}'


workflow_consensus_stats=(
    workflow_df
    .groupby(
        'workflow_prototype'
    )
    .agg(
        workflow_consensus_confidence=(
            'workflow_prototype_confidence',
            'mean'
        ),
        workflow_consensus_margin=(
            'workflow_consensus_margin',
            'mean'
        ),
        latent_workflow_count=(
            'workflow_id',
            'count'
        ),
    )
    .reset_index()
    if len(
        workflow_df
    )
    else pd.DataFrame()
)

if (
    len(
        process_models
    )
    and len(
        workflow_consensus_stats
    )
):
    process_models=process_models.merge(
        workflow_consensus_stats,
        on='workflow_prototype',
        how='left'
    )

if len(
    process_models
):
    max_instances=max(
        int(
            process_models[
                'instance_count'
            ].max()
        ),
        1
    )

    report_rows=[]

    for _,r in process_models.iterrows():
        model_id=safe_str(
            r[
                'model_id'
            ]
        )

        modal_path,path_consistency=_model_modal_path(
            model_id
        )

        support_score=(
            math.log1p(
                int(
                    r[
                        'instance_count'
                    ]
                )
            )
            / math.log1p(
                max_instances
            )
        )

        stability_score=_clip01(
            r.get(
                'workflow_consensus_confidence',
                r.get(
                    'avg_prototype_confidence',
                    0.0
                )
            )
        )

        raw_margin=float(
            r.get(
                'workflow_consensus_margin',
                0.0
            )
            or 0.0
        )

        # consensus feature margin은 대체로 0~1.
        margin_score=_clip01(
            raw_margin*2.0
        )

        evidence_score=_evidence_richness_for_model(
            model_id
        )

        structural_score=_structural_coherence_for_model(
            model_id
        )

        reportability_score=(
            .20*support_score
            + .30*stability_score
            + .15*margin_score
            + .20*_clip01(
                path_consistency
            )
            + .10*evidence_score
            + .05*structural_score
        )

        report_rows.append(
            {
                'model_id':
                    model_id,
                'support_score':
                    float(
                        support_score
                    ),
                'stability_score':
                    float(
                        stability_score
                    ),
                'margin_score':
                    float(
                        margin_score
                    ),
                'path_consistency':
                    float(
                        path_consistency
                    ),
                'evidence_richness':
                    float(
                        evidence_score
                    ),
                'structural_coherence':
                    float(
                        structural_score
                    ),
                'reportability_score':
                    float(
                        reportability_score
                    ),
                'representative_path':
                    ' → '.join(
                        modal_path[
                            :12
                        ]
                    ),
            }
        )

    reportability_df=pd.DataFrame(
        report_rows
    )

    process_models=process_models.merge(
        reportability_df,
        on='model_id',
        how='left'
    )

    process_models=process_models.sort_values(
        [
            'reportability_score',
            'instance_count'
        ],
        ascending=[
            False,
            False
        ]
    ).reset_index(
        drop=True
    )

    process_models[
        'report_rank'
    ]=np.arange(
        1,
        len(
            process_models
        )+1
    )

    process_models[
        'customer_display_name'
    ]=[
        _customer_name(
            r,
            int(
                r[
                    'report_rank'
                ]
            )
        )
        for _,r in process_models.iterrows()
    ]

    primary_mask=(
        (
            process_models[
                'reportability_score'
            ]>=REPORT_SCORE_THRESHOLD
        )
        & (
            process_models[
                'instance_count'
            ]>=REPORT_MIN_INSTANCE_SUPPORT
        )
    )

    primary_count=int(
        primary_mask.sum()
    )

    target_count=min(
        len(
            process_models
        ),
        REPORT_MAX_WORKFLOWS,
        max(
            REPORT_TARGET_MIN_WORKFLOWS,
            primary_count
        )
    )

    process_models[
        'selected_for_customer_report'
    ]=0

    if target_count>0:
        process_models.loc[
            process_models.index[
                :target_count
            ],
            'selected_for_customer_report'
        ]=1

    process_models[
        'report_status'
    ]='Not selected'

    process_models.loc[
        primary_mask,
        'report_status'
    ]='Reportable'

    topup_mask=(
        (
            process_models[
                'selected_for_customer_report'
            ]==1
        )
        & (
            process_models[
                'report_status'
            ]!='Reportable'
        )
    )

    process_models.loc[
        topup_mask,
        'report_status'
    ]='Review candidate'

else:
    reportability_df=pd.DataFrame()

    for col,default in [
        (
            'report_rank',
            []
        ),
        (
            'customer_display_name',
            []
        ),
        (
            'selected_for_customer_report',
            []
        ),
        (
            'report_status',
            []
        ),
    ]:
        process_models[
            col
        ]=default


reportable_workflows=(
    process_models[
        process_models[
            'selected_for_customer_report'
        ]==1
    ].copy()
    if len(
        process_models
    )
    else pd.DataFrame()
)


# Representative step rows for customer/developer handoff.
reportable_step_rows=[]

if len(
    reportable_workflows
):
    for mr in reportable_workflows.itertuples(
        index=False
    ):
        path=[
            x.strip()
            for x in safe_str(
                mr.representative_path
            ).split(
                '→'
            )
            if x.strip()
        ]

        for step_order,label in enumerate(
            path,
            start=1
        ):
            reportable_step_rows.append(
                {
                    'report_rank':
                        int(
                            mr.report_rank
                        ),
                    'model_id':
                        mr.model_id,
                    'workflow_prototype':
                        mr.workflow_prototype,
                    'customer_display_name':
                        mr.customer_display_name,
                    'step_order':
                        int(
                            step_order
                        ),
                    'step_label':
                        label,
                }
            )

reportable_workflow_steps=pd.DataFrame(
    reportable_step_rows
)


# ============================================================
# K. V6.2 PoC Ground Truth 7-Scenario Comparison
# ============================================================
# IMPORTANT:
# - presentation / post-hoc evaluation only
# - training / threshold / prototype / consensus clustering에는 사용하지 않음
# - 고객에게 "사전 정의 시나리오와 모델 결과가 얼마나 닮았는가"를
#   사람이 읽기 쉬운 형태로 보여주기 위한 비교 계층

POC_SCENARIOS = [
    {
        'scenario_id': 'SC01',
        'owner': '국제교류',
        'business_area': '국제교류 기초·교류학생·기숙사 관리',
        'ground_truth_steps': [
            '국제교류학년도 학기',
            '교류기관관리',
            '교류학생관리(OUT)',
            '교류학생관리(IN)',
            '유학생기숙사호실관리',
            '유학생기숙사납부내역',
        ],
    },
    {
        'scenario_id': 'SC02',
        'owner': '교무/수업',
        'business_area': '학적·수업·성적 및 교원업적 관리',
        'ground_truth_steps': [
            '진급결과조회',
            '개설강좌관리',
            '수강조건별조회',
            '출석부관리',
            '성적입력마감조회',
            '교원자격증발급',
            '트랙이수관리',
            '평가항목관리',
            '이의신청관리',
        ],
    },
    {
        'scenario_id': 'SC03',
        'owner': '학생처/장학등록',
        'business_area': '학적·장학·등록 및 학생지원 관리',
        'ground_truth_steps': [
            '학적변동신청내역조회',
            '장학생 중 학적변동자명단',
            '등록자료 조건별 조회',
            '미등록장학생명단',
            '자치기구 회원조회',
            '운영현황',
            '학생별프로그램조회',
            '성과관리표',
            '출결정보조회',
            '예산사업내역서',
            '자금예산집행내역서',
        ],
    },
    {
        'scenario_id': 'SC04',
        'owner': '인사/노무/교원 관리',
        'business_area': '인사·복무·급여 및 평가 관리',
        'ground_truth_steps': [
            '교직원 개인별 현보직 명단',
            '교원현황',
            '개인별근태현황',
            '월근태현황',
            '개인별월별급여지급내역',
            '세금공제내역',
            '자료입력마감관리',
            '이의신청관리',
            '평가년도관리',
        ],
    },
    {
        'scenario_id': 'SC05',
        'owner': '예산',
        'business_area': '예산 조정 업무',
        'ground_truth_steps': [
            '부서 예산 조회',
            '조정 대상 확인',
            '산출근거 수정·저장',
            '예산현황 확인',
            '조정현황 확인',
        ],
    },
    {
        'scenario_id': 'SC06',
        'owner': '회계',
        'business_area': '회계 기준정보 관리',
        'ground_truth_steps': [
            '계정관리항목 조회·수정·저장',
            '기본세율 조회·수정·저장',
            '변경 결과 확인',
        ],
    },
    {
        'scenario_id': 'SC07',
        'owner': '회계/결산',
        'business_area': '결산자료 관리',
        'ground_truth_steps': [
            '결산 산출내역 조회·수정·저장',
            '투자유가증권 명세 조회·확인·수정',
            '결과 확인',
        ],
    },
]


def _scenario_norm(
    text
):
    text=safe_str(
        text
    ).lower()

    text=re.sub(
        r'[\s·ㆍ\-\_/()\[\]{}:,.]+',
        '',
        text
    )

    return text


def _customer_readable_step_label(
    row
):
    label=safe_str(
        row.get(
            'display_label'
        )
    ).strip()

    if not label:
        return ''

    # 고객 입장에서 의미가 없는 prototype ID 단독 표시만 숨긴다.
    # 조회/팝업/필드/달력 등 이름이 있는 하위 단계는 그대로 유지한다.
    if re.fullmatch(
        r'(?:SP|WP)\d+',
        label,
        flags=re.I
    ):
        return ''

    return label


def _scenario_step_matches(
    gt_step,
    observed_label
):
    """
    Ground Truth는 composite phrase를 포함할 수 있으므로
    normalization 후 exact/containment를 허용한다.

    예:
      '계정관리항목 조회·수정·저장'
      vs
      '계정관리항목'

    단, 너무 짧은 generic token의 containment는 허용하지 않는다.
    """
    g=_scenario_norm(
        gt_step
    )

    o=_scenario_norm(
        observed_label
    )

    if not g or not o:
        return False

    if g==o:
        return True

    if min(
        len(
            g
        ),
        len(
            o
        )
    )>=4:
        return (
            g in o
            or o in g
        )

    return False


def _scenario_lcs(
    gt_steps,
    observed_steps
):
    n=len(
        gt_steps
    )

    m=len(
        observed_steps
    )

    if not n or not m:
        return 0

    dp=np.zeros(
        (
            n+1,
            m+1
        ),
        dtype=np.int16
    )

    for i in range(
        1,
        n+1
    ):
        for j in range(
            1,
            m+1
        ):
            if _scenario_step_matches(
                gt_steps[
                    i-1
                ],
                observed_steps[
                    j-1
                ]
            ):
                dp[
                    i,
                    j
                ]=dp[
                    i-1,
                    j-1
                ]+1

            else:
                dp[
                    i,
                    j
                ]=max(
                    dp[
                        i-1,
                        j
                    ],
                    dp[
                        i,
                        j-1
                    ]
                )

    return int(
        dp[
            n,
            m
        ]
    )


def _scenario_instance_path(
    instance_id
):
    if not len(
        process_instance_steps
    ):
        return []

    g=process_instance_steps[
        process_instance_steps[
            'instance_id'
        ]==instance_id
    ].sort_values(
        'step_order'
    )

    labels=[]

    for _,r in g.iterrows():
        label=_customer_readable_step_label(
            r
        )

        if label:
            labels.append(
                label
            )

    return labels


def _scenario_score_path(
    scenario,
    labels
):
    gt=list(
        scenario[
            'ground_truth_steps'
        ]
    )

    matched=[]

    for step in gt:
        if any(
            _scenario_step_matches(
                step,
                label
            )
            for label in labels
        ):
            matched.append(
                step
            )

    coverage=(
        len(
            matched
        )
        / len(
            gt
        )
        if gt
        else 0.0
    )

    lcs=_scenario_lcs(
        gt,
        labels
    )

    order_ratio=(
        lcs
        / len(
            gt
        )
        if gt
        else 0.0
    )

    # Coverage를 우선하고, 같은 coverage에서는 기준 순서 재현을 우선한다.
    score=(
        .70*coverage
        + .30*order_ratio
    )

    missing=[
        step
        for step in gt
        if step not in matched
    ]

    return {
        'scenario_score':
            float(
                score
            ),
        'coverage_ratio':
            float(
                coverage
            ),
        'order_ratio':
            float(
                order_ratio
            ),
        'ground_truth_step_count':
            int(
                len(
                    gt
                )
            ),
        'matched_step_count':
            int(
                len(
                    matched
                )
            ),
        'ordered_match_count':
            int(
                lcs
            ),
        'matched_steps':
            matched,
        'missing_steps':
            missing,
    }


scenario_instance_rows=[]

if len(
    process_instances
):
    _instance_meta=(
        process_instances
        .drop_duplicates(
            'instance_id'
        )
        .set_index(
            'instance_id'
        )
        .to_dict(
            orient='index'
        )
    )

    for scenario in POC_SCENARIOS:
        for instance_id in process_instances[
            'instance_id'
        ].drop_duplicates(
        ).tolist():
            labels=_scenario_instance_path(
                instance_id
            )

            metrics=_scenario_score_path(
                scenario,
                labels
            )

            meta=_instance_meta.get(
                instance_id,
                {}
            )

            scenario_instance_rows.append(
                {
                    'scenario_id':
                        scenario[
                            'scenario_id'
                        ],
                    'owner':
                        scenario[
                            'owner'
                        ],
                    'business_area':
                        scenario[
                            'business_area'
                        ],
                    'instance_id':
                        instance_id,
                    'model_id':
                        safe_str(
                            meta.get(
                                'model_id'
                            )
                        ),
                    'workflow_prototype':
                        safe_str(
                            meta.get(
                                'workflow_prototype'
                            )
                        ),
                    'input_count':
                        int(
                            meta.get(
                                'input_count',
                                0
                            )
                            or 0
                        ),
                    'compressed_step_count':
                        int(
                            meta.get(
                                'compressed_step_count',
                                0
                            )
                            or 0
                        ),
                    'readable_path':
                        ' → '.join(
                            labels
                        ),
                    **metrics,
                }
            )


scenario_instance_matches=pd.DataFrame(
    scenario_instance_rows
)

scenario_summary_rows=[]

for scenario in POC_SCENARIOS:
    sid=scenario[
        'scenario_id'
    ]

    sg=(
        scenario_instance_matches[
            scenario_instance_matches[
                'scenario_id'
            ]==sid
        ].copy()
        if len(
            scenario_instance_matches
        )
        else pd.DataFrame()
    )

    if len(
        sg
    ):
        sg=sg.sort_values(
            [
                'scenario_score',
                'matched_step_count',
                'ordered_match_count',
                'input_count',
                'compressed_step_count'
            ],
            ascending=[
                False,
                False,
                False,
                False,
                False
            ]
        )

        best=sg.iloc[
            0
        ]

        best_score=float(
            best[
                'scenario_score'
            ]
        )

        if best[
            'matched_step_count'
        ]>0:
            best_model_id=safe_str(
                best[
                    'model_id'
                ]
            )
            best_instance_id=safe_str(
                best[
                    'instance_id'
                ]
            )
            best_wp=safe_str(
                best[
                    'workflow_prototype'
                ]
            )
        else:
            best_model_id=''
            best_instance_id=''
            best_wp=''

        scenario_summary_rows.append(
            {
                'scenario_id':
                    sid,
                'owner':
                    scenario[
                        'owner'
                    ],
                'business_area':
                    scenario[
                        'business_area'
                    ],
                'ground_truth_steps':
                    ' → '.join(
                        scenario[
                            'ground_truth_steps'
                        ]
                    ),
                'ground_truth_step_count':
                    int(
                        len(
                            scenario[
                                'ground_truth_steps'
                            ]
                        )
                    ),
                'best_model_id':
                    best_model_id,
                'best_workflow_prototype':
                    best_wp,
                'best_instance_id':
                    best_instance_id,
                'scenario_score':
                    best_score,
                'matched_step_count':
                    int(
                        best[
                            'matched_step_count'
                        ]
                    ),
                'ordered_match_count':
                    int(
                        best[
                            'ordered_match_count'
                        ]
                    ),
                'coverage_ratio':
                    float(
                        best[
                            'coverage_ratio'
                        ]
                    ),
                'order_ratio':
                    float(
                        best[
                            'order_ratio'
                        ]
                    ),
                'matched_steps':
                    ' | '.join(
                        best[
                            'matched_steps'
                        ]
                    ),
                'missing_steps':
                    ' | '.join(
                        best[
                            'missing_steps'
                        ]
                    ),
                'best_readable_path':
                    safe_str(
                        best[
                            'readable_path'
                        ]
                    ),
            }
        )

    else:
        scenario_summary_rows.append(
            {
                'scenario_id':
                    sid,
                'owner':
                    scenario[
                        'owner'
                    ],
                'business_area':
                    scenario[
                        'business_area'
                    ],
                'ground_truth_steps':
                    ' → '.join(
                        scenario[
                            'ground_truth_steps'
                        ]
                    ),
                'ground_truth_step_count':
                    int(
                        len(
                            scenario[
                                'ground_truth_steps'
                            ]
                        )
                    ),
                'best_model_id':
                    '',
                'best_workflow_prototype':
                    '',
                'best_instance_id':
                    '',
                'scenario_score':
                    0.0,
                'matched_step_count':
                    0,
                'ordered_match_count':
                    0,
                'coverage_ratio':
                    0.0,
                'order_ratio':
                    0.0,
                'matched_steps':
                    '',
                'missing_steps':
                    ' | '.join(
                        scenario[
                            'ground_truth_steps'
                        ]
                    ),
                'best_readable_path':
                    '',
            }
        )


scenario_comparison=pd.DataFrame(
    scenario_summary_rows
)


# ============================================================
# V6.2 User-level trajectory scenario discovery
# ============================================================

def _build_user_trajectory_windows():
    rows=[]

    if (
        not USE_USER_TRAJECTORY
        or not len(
            user_trajectory_instances
        )
    ):
        return pd.DataFrame()

    window_seq=0

    for trajectory_id,g in user_trajectory_instances.groupby(
        'trajectory_id',
        sort=False
    ):
        g=g.sort_values(
            'trajectory_order'
        ).reset_index(
            drop=True
        )

        n=len(
            g
        )

        if n==0:
            continue

        for start_idx in range(
            n
        ):
            labels=[]
            instance_ids=[]
            instance_paths=[]
            model_ids=[]
            wps=[]
            session_ids=[]

            window_start=pd.to_datetime(
                g.loc[
                    start_idx,
                    'start_time'
                ],
                errors='coerce'
            )

            for end_idx in range(
                start_idx,
                min(
                    n,
                    start_idx
                    + USER_TRAJECTORY_MAX_WINDOW_INSTANCES
                )
            ):
                row=g.loc[
                    end_idx
                ]

                end_time=pd.to_datetime(
                    row.get(
                        'end_time'
                    ),
                    errors='coerce'
                )

                duration_min=None

                if (
                    pd.notna(
                        window_start
                    )
                    and pd.notna(
                        end_time
                    )
                ):
                    duration_min=max(
                        0.0,
                        (
                            end_time-window_start
                        ).total_seconds()
                        / 60.0
                    )

                    if (
                        end_idx>start_idx
                        and duration_min
                        > USER_TRAJECTORY_MAX_WINDOW_MINUTES
                    ):
                        break

                iid=safe_str(
                    row.get(
                        'instance_id'
                    )
                )

                path=_scenario_instance_path(
                    iid
                )

                instance_ids.append(
                    iid
                )

                instance_paths.append(
                    ' → '.join(
                        path
                    )
                )

                labels.extend(
                    path
                )

                mid=safe_str(
                    row.get(
                        'model_id'
                    )
                )

                wp=safe_str(
                    row.get(
                        'workflow_prototype'
                    )
                )

                sid=safe_str(
                    row.get(
                        'session_id'
                    )
                )

                if mid:
                    model_ids.append(
                        mid
                    )

                if wp:
                    wps.append(
                        wp
                    )

                if sid:
                    session_ids.append(
                        sid
                    )

                if not labels:
                    continue

                window_seq+=1

                rows.append(
                    {
                        'window_id':
                            f'UTW{window_seq:08d}',
                        'trajectory_id':
                            trajectory_id,
                        'trajectory_user_id':
                            safe_str(
                                row.get(
                                    'trajectory_user_id'
                                )
                            ),
                        'trajectory_user_display_id':
                            safe_str(
                                row.get(
                                    'trajectory_user_display_id'
                                )
                            ),
                        'window_start_order':
                            int(
                                g.loc[
                                    start_idx,
                                    'trajectory_order'
                                ]
                            ),
                        'window_end_order':
                            int(
                                row.get(
                                    'trajectory_order'
                                )
                            ),
                        'instance_count':
                            int(
                                len(
                                    instance_ids
                                )
                            ),
                        'session_count':
                            int(
                                len(
                                    set(
                                        session_ids
                                    )
                                )
                            ),
                        'is_cross_session':
                            int(
                                len(
                                    set(
                                        session_ids
                                    )
                                )>1
                            ),
                        'duration_min':
                            (
                                None
                                if duration_min is None
                                else float(
                                    duration_min
                                )
                            ),
                        'first_instance_id':
                            instance_ids[
                                0
                            ],
                        'first_model_id':
                            (
                                model_ids[
                                    0
                                ]
                                if model_ids
                                else ''
                            ),
                        'first_workflow_prototype':
                            (
                                wps[
                                    0
                                ]
                                if wps
                                else ''
                            ),
                        'instance_ids':
                            ' | '.join(
                                instance_ids
                            ),
                        'session_ids':
                            ' | '.join(
                                dict.fromkeys(
                                    session_ids
                                )
                            ),
                        'model_ids':
                            ' | '.join(
                                dict.fromkeys(
                                    model_ids
                                )
                            ),
                        'workflow_prototypes':
                            ' | '.join(
                                dict.fromkeys(
                                    wps
                                )
                            ),
                        'readable_path':
                            ' → '.join(
                                labels
                            ),
                        'instance_paths':
                            ' || '.join(
                                instance_paths
                            ),
                    }
                )

    return pd.DataFrame(
        rows
    )


user_trajectory_windows=(
    _build_user_trajectory_windows()
)


trajectory_scenario_rows=[]

if len(
    user_trajectory_windows
):
    for scenario in POC_SCENARIOS:
        for _,window in user_trajectory_windows.iterrows():
            labels=[
                x.strip()
                for x in safe_str(
                    window[
                        'readable_path'
                    ]
                ).split(
                    '→'
                )
                if x.strip()
            ]

            metrics=_scenario_score_path(
                scenario,
                labels
            )

            if metrics[
                'matched_step_count'
            ]<=0:
                continue

            trajectory_scenario_rows.append(
                {
                    'scenario_id':
                        scenario[
                            'scenario_id'
                        ],
                    'owner':
                        scenario[
                            'owner'
                        ],
                    'business_area':
                        scenario[
                            'business_area'
                        ],
                    **window.to_dict(),
                    **metrics,
                }
            )


scenario_user_trajectory_matches=pd.DataFrame(
    trajectory_scenario_rows
)


# trajectory summary에 사람이 읽을 수 있는 전체 episode path도 추가
if len(
    user_trajectory_summary
):
    _traj_path_map={}

    for trajectory_id,g in user_trajectory_instances.groupby(
        'trajectory_id',
        sort=False
    ):
        g=g.sort_values(
            'trajectory_order'
        )

        _paths=[]

        for iid in g[
            'instance_id'
        ].tolist():
            p=_scenario_instance_path(
                iid
            )

            if p:
                _paths.extend(
                    p
                )

        _traj_path_map[
            trajectory_id
        ]=' → '.join(
            _paths
        )

    user_trajectory_summary[
        'readable_path'
    ]=user_trajectory_summary[
        'trajectory_id'
    ].map(
        _traj_path_map
    ).fillna(
        ''
    )


# 기존 single-instance 사후 비교값은 진단용으로 보존한다.
if len(
    scenario_comparison
):
    for _col in [
        'scenario_score',
        'matched_step_count',
        'ordered_match_count',
        'coverage_ratio',
        'order_ratio',
        'matched_steps',
        'missing_steps',
        'best_readable_path',
    ]:
        if _col in scenario_comparison.columns:
            scenario_comparison[
                f'instance_{_col}'
            ]=scenario_comparison[
                _col
            ]


trajectory_best_rows=[]

for scenario in POC_SCENARIOS:
    sid=scenario[
        'scenario_id'
    ]

    tg=(
        scenario_user_trajectory_matches[
            scenario_user_trajectory_matches[
                'scenario_id'
            ]==sid
        ].copy()
        if len(
            scenario_user_trajectory_matches
        )
        else pd.DataFrame()
    )

    if len(
        tg
    ):
        # 동일 사용자가 많은 overlapping window를 만들 수 있으므로
        # support user 수는 사용자별 최고 점수 하나만 사용한다.
        user_best=(
            tg.sort_values(
                [
                    'scenario_score',
                    'matched_step_count',
                    'ordered_match_count',
                ],
                ascending=[
                    False,
                    False,
                    False,
                ]
            )
            .groupby(
                'trajectory_user_id',
                as_index=False
            )
            .head(
                1
            )
        )

        supporting_user_count=int(
            (
                user_best[
                    'scenario_score'
                ]
                >= USER_TRAJECTORY_SUPPORT_SCORE
            ).sum()
        )

        trajectory_best=(
            tg.sort_values(
                [
                    'scenario_score',
                    'matched_step_count',
                    'ordered_match_count',
                    'instance_count',
                    'duration_min',
                ],
                ascending=[
                    False,
                    False,
                    False,
                    True,
                    True,
                ]
            )
            .iloc[
                0
            ]
        )

        trajectory_best_rows.append(
            {
                'scenario_id':
                    sid,
                'best_trajectory_window_id':
                    safe_str(
                        trajectory_best[
                            'window_id'
                        ]
                    ),
                'best_trajectory_id':
                    safe_str(
                        trajectory_best[
                            'trajectory_id'
                        ]
                    ),
                'best_trajectory_user_display_id':
                    safe_str(
                        trajectory_best[
                            'trajectory_user_display_id'
                        ]
                    ),
                'best_trajectory_first_instance_id':
                    safe_str(
                        trajectory_best[
                            'first_instance_id'
                        ]
                    ),
                'best_trajectory_first_model_id':
                    safe_str(
                        trajectory_best[
                            'first_model_id'
                        ]
                    ),
                'best_trajectory_first_workflow_prototype':
                    safe_str(
                        trajectory_best[
                            'first_workflow_prototype'
                        ]
                    ),
                'best_trajectory_instance_ids':
                    safe_str(
                        trajectory_best[
                            'instance_ids'
                        ]
                    ),
                'best_trajectory_session_ids':
                    safe_str(
                        trajectory_best[
                            'session_ids'
                        ]
                    ),
                'best_trajectory_model_ids':
                    safe_str(
                        trajectory_best[
                            'model_ids'
                        ]
                    ),
                'best_trajectory_workflow_prototypes':
                    safe_str(
                        trajectory_best[
                            'workflow_prototypes'
                        ]
                    ),
                'best_trajectory_instance_paths':
                    safe_str(
                        trajectory_best[
                            'instance_paths'
                        ]
                    ),
                'best_trajectory_readable_path':
                    safe_str(
                        trajectory_best[
                            'readable_path'
                        ]
                    ),
                'best_trajectory_instance_count':
                    int(
                        trajectory_best[
                            'instance_count'
                        ]
                    ),
                'best_trajectory_session_count':
                    int(
                        trajectory_best[
                            'session_count'
                        ]
                    ),
                'best_trajectory_is_cross_session':
                    int(
                        trajectory_best[
                            'is_cross_session'
                        ]
                    ),
                'best_trajectory_duration_min':
                    (
                        None
                        if pd.isna(
                            trajectory_best[
                                'duration_min'
                            ]
                        )
                        else float(
                            trajectory_best[
                                'duration_min'
                            ]
                        )
                    ),
                'trajectory_scenario_score':
                    float(
                        trajectory_best[
                            'scenario_score'
                        ]
                    ),
                'trajectory_matched_step_count':
                    int(
                        trajectory_best[
                            'matched_step_count'
                        ]
                    ),
                'trajectory_ordered_match_count':
                    int(
                        trajectory_best[
                            'ordered_match_count'
                        ]
                    ),
                'trajectory_coverage_ratio':
                    float(
                        trajectory_best[
                            'coverage_ratio'
                        ]
                    ),
                'trajectory_order_ratio':
                    float(
                        trajectory_best[
                            'order_ratio'
                        ]
                    ),
                'trajectory_matched_steps':
                    ' | '.join(
                        trajectory_best[
                            'matched_steps'
                        ]
                    ),
                'trajectory_missing_steps':
                    ' | '.join(
                        trajectory_best[
                            'missing_steps'
                        ]
                    ),
                'supporting_user_count':
                    supporting_user_count,
                'candidate_window_count':
                    int(
                        len(
                            tg
                        )
                    ),
            }
        )

    else:
        trajectory_best_rows.append(
            {
                'scenario_id':
                    sid,
                'best_trajectory_window_id':
                    '',
                'best_trajectory_id':
                    '',
                'best_trajectory_user_display_id':
                    '',
                'best_trajectory_first_instance_id':
                    '',
                'best_trajectory_first_model_id':
                    '',
                'best_trajectory_first_workflow_prototype':
                    '',
                'best_trajectory_instance_ids':
                    '',
                'best_trajectory_session_ids':
                    '',
                'best_trajectory_model_ids':
                    '',
                'best_trajectory_workflow_prototypes':
                    '',
                'best_trajectory_instance_paths':
                    '',
                'best_trajectory_readable_path':
                    '',
                'best_trajectory_instance_count':
                    0,
                'best_trajectory_session_count':
                    0,
                'best_trajectory_is_cross_session':
                    0,
                'best_trajectory_duration_min':
                    None,
                'trajectory_scenario_score':
                    0.0,
                'trajectory_matched_step_count':
                    0,
                'trajectory_ordered_match_count':
                    0,
                'trajectory_coverage_ratio':
                    0.0,
                'trajectory_order_ratio':
                    0.0,
                'trajectory_matched_steps':
                    '',
                'trajectory_missing_steps':
                    ' | '.join(
                        scenario[
                            'ground_truth_steps'
                        ]
                    ),
                'supporting_user_count':
                    0,
                'candidate_window_count':
                    0,
            }
        )


trajectory_best_df=pd.DataFrame(
    trajectory_best_rows
)

if len(
    scenario_comparison
):
    scenario_comparison=scenario_comparison.merge(
        trajectory_best_df,
        on='scenario_id',
        how='left'
    )

    # V6.2 고객/보고용 primary 비교는 user trajectory가 있으면 이를 사용.
    _has_trajectory=(
        scenario_comparison[
            'best_trajectory_window_id'
        ].fillna(
            ''
        )!=''
    )

    scenario_comparison[
        'comparison_source'
    ]=np.where(
        _has_trajectory,
        'user_trajectory',
        'single_instance'
    )

    for _target,_traj,_inst in [
        (
            'scenario_score',
            'trajectory_scenario_score',
            'instance_scenario_score'
        ),
        (
            'matched_step_count',
            'trajectory_matched_step_count',
            'instance_matched_step_count'
        ),
        (
            'ordered_match_count',
            'trajectory_ordered_match_count',
            'instance_ordered_match_count'
        ),
        (
            'coverage_ratio',
            'trajectory_coverage_ratio',
            'instance_coverage_ratio'
        ),
        (
            'order_ratio',
            'trajectory_order_ratio',
            'instance_order_ratio'
        ),
        (
            'matched_steps',
            'trajectory_matched_steps',
            'instance_matched_steps'
        ),
        (
            'missing_steps',
            'trajectory_missing_steps',
            'instance_missing_steps'
        ),
        (
            'best_readable_path',
            'best_trajectory_readable_path',
            'instance_best_readable_path'
        ),
    ]:
        scenario_comparison[
            _target
        ]=np.where(
            _has_trajectory,
            scenario_comparison[
                _traj
            ],
            scenario_comparison[
                _inst
            ]
        )


# V6.2 user trajectory artifacts
# raw sessions.user_id는 grouping에만 사용하고 export에는 직접 노출하지 않는다.
# 고객/진단 파일에는 stable hash 기반 trajectory_user_display_id만 남긴다.

def _drop_raw_trajectory_user_id(
    frame
):
    if frame is None:
        return pd.DataFrame()

    out=frame.copy()

    if 'trajectory_user_id' in out.columns:
        out=out.drop(
            columns=[
                'trajectory_user_id'
            ]
        )

    return out


user_trajectory_instances_export=(
    _drop_raw_trajectory_user_id(
        user_trajectory_instances
    )
)

user_trajectory_windows_export=(
    _drop_raw_trajectory_user_id(
        user_trajectory_windows
    )
)

scenario_user_trajectory_matches_export=(
    _drop_raw_trajectory_user_id(
        scenario_user_trajectory_matches
    )
)

user_trajectory_summary_export=(
    _drop_raw_trajectory_user_id(
        user_trajectory_summary
    )
)

user_trajectory_instances_export.to_csv(
    DIAGNOSTICS_DIR
    / 'user_trajectory_instances.csv',
    index=False,
    encoding='utf-8-sig'
)

user_trajectory_windows_export.to_csv(
    DIAGNOSTICS_DIR
    / 'user_trajectory_windows.csv',
    index=False,
    encoding='utf-8-sig'
)

scenario_user_trajectory_matches_export.to_csv(
    DIAGNOSTICS_DIR
    / 'poc_scenario_user_trajectory_matches.csv',
    index=False,
    encoding='utf-8-sig'
)

user_trajectory_summary_export.to_csv(
    MAIN_RESULTS_DIR
    / 'user_trajectory_summary.csv',
    index=False,
    encoding='utf-8-sig'
)


scenario_comparison.to_csv(
    MAIN_RESULTS_DIR
    / 'poc_scenario_comparison.csv',
    index=False,
    encoding='utf-8-sig'
)

scenario_instance_matches.to_csv(
    DIAGNOSTICS_DIR
    / 'poc_scenario_instance_matches.csv',
    index=False,
    encoding='utf-8-sig'
)

print(
    '\\n=== V6.2 POC 7 SCENARIO COMPARISON ==='
)

print(
    scenario_comparison[
        [
            'scenario_id',
            'owner',
            'comparison_source',
            'best_workflow_prototype',
            'best_trajectory_workflow_prototypes',
            'supporting_user_count',
            'matched_step_count',
            'ground_truth_step_count',
            'ordered_match_count',
            'coverage_ratio',
            'order_ratio',
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# K2. Customer Workflow Report
# ============================================================

def _html_pct(
    value,
    digits=0
):
    try:
        x=float(
            value
        )
        if not np.isfinite(
            x
        ):
            return '-'
        return f'{x*100:.{digits}f}%'
    except Exception:
        return '-'


def _customer_model_inputs(
    model_id,
    max_labels=8
):
    if not len(
        process_node_inputs
    ):
        return []

    g=process_node_inputs[
        process_node_inputs[
            'model_id'
        ]==model_id
    ]

    if not len(
        g
    ):
        return []

    labels=[]

    for label,count in (
        g[
            'input_label'
        ]
        .fillna(
            ''
        )
        .astype(
            str
        )
        .value_counts()
        .items()
    ):
        label=label.strip()

        if (
            label
            and label not in labels
        ):
            labels.append(
                (
                    label,
                    int(
                        count
                    )
                )
            )

        if len(
            labels
        )>=max_labels:
            break

    return labels


def _customer_examples(
    model_id,
    max_examples=3
):
    if not len(
        process_instances
    ):
        return []

    g=process_instances[
        process_instances[
            'model_id'
        ]==model_id
    ].copy()

    if not len(
        g
    ):
        return []

    g=g.sort_values(
        [
            'input_count',
            'compressed_step_count',
            'source_segment_count'
        ],
        ascending=[
            False,
            False,
            False
        ]
    )

    return g.head(
        max_examples
    ).to_dict(
        orient='records'
    )


def build_customer_workflow_report():
    selected=reportable_workflows.copy()

    period_start=(
        pd.to_datetime(
            df[
                'event_time_parsed'
            ],
            errors='coerce'
        ).min()
        if len(
            df
        )
        else None
    )

    period_end=(
        pd.to_datetime(
            df[
                'event_time_parsed'
            ],
            errors='coerce'
        ).max()
        if len(
            df
        )
        else None
    )

    reportable_count=int(
        (
            selected[
                'report_status'
            ]=='Reportable'
        ).sum()
    ) if len(
        selected
    ) else 0

    cards=[]

    for r in selected.itertuples(
        index=False
    ):
        status_class=(
            'good'
            if r.report_status=='Reportable'
            else 'review'
        )

        path=[
            x.strip()
            for x in safe_str(
                r.representative_path
            ).split(
                '→'
            )
            if x.strip()
        ]

        path_html=''.join(
            (
                f'<div class="step">{html_lib.escape(label)}</div>'
                + (
                    '<div class="arrow">↓</div>'
                    if i<len(
                        path
                    )-1
                    else ''
                )
            )
            for i,label in enumerate(
                path[
                    :10
                ]
            )
        )

        if not path_html:
            path_html='<div class="empty">대표 경로를 생성하기에 충분한 step 정보가 없습니다.</div>'

        inputs=_customer_model_inputs(
            r.model_id
        )

        inputs_html=''.join(
            (
                '<span class="input-chip">'
                f'{html_lib.escape(label)} '
                f'<b>{count}</b>'
                '</span>'
            )
            for label,count in inputs
        )

        if not inputs_html:
            inputs_html='<span class="muted">주요 입력 항목 없음</span>'

        examples=_customer_examples(
            r.model_id
        )

        ex_html=''.join(
            (
                '<tr>'
                f'<td>{html_lib.escape(safe_str(x.get("instance_id")))}</td>'
                f'<td>{html_lib.escape(safe_str(x.get("start_time")))}</td>'
                f'<td>{html_lib.escape(safe_str(x.get("end_time")))}</td>'
                f'<td class="num">{html_lib.escape(safe_str(x.get("compressed_step_count")))}</td>'
                f'<td class="num">{html_lib.escape(safe_str(x.get("input_count")))}</td>'
                '</tr>'
            )
            for x in examples
        )

        cards.append(
            f"""
<section class="workflow-card">
  <div class="workflow-head">
    <div>
      <div class="eyebrow">업무 흐름 {int(r.report_rank):02d} · {html_lib.escape(safe_str(r.workflow_prototype))}</div>
      <h2>{html_lib.escape(safe_str(r.customer_display_name))}</h2>
    </div>
    <span class="status {status_class}">{html_lib.escape(safe_str(r.report_status))}</span>
  </div>

  <div class="metrics">
    <div><b>{int(r.instance_count):,}</b><span>실행 사례</span></div>
    <div><b>{int(r.node_count):,}</b><span>대표 단계 후보</span></div>
    <div><b>{float(r.reportability_score):.3f}</b><span>Reportability</span></div>
    <div><b>{_html_pct(r.stability_score)}</b><span>Consensus 안정성</span></div>
    <div><b>{_html_pct(r.path_consistency)}</b><span>대표 경로 일관성</span></div>
  </div>

  <div class="two-col">
    <div>
      <h3>대표 업무 흐름</h3>
      <div class="path-box">{path_html}</div>
    </div>
    <div>
      <h3>주요 입력 항목</h3>
      <div class="input-chips">{inputs_html}</div>
      <div class="score-note">
        <b>분석 근거</b><br>
        Seed consensus confidence {_html_pct(r.stability_score)} ·
        path consistency {_html_pct(r.path_consistency)} ·
        structural coherence {_html_pct(r.structural_coherence)}
      </div>
    </div>
  </div>

  <h3>대표 실행 사례</h3>
  <div class="scroll">
  <table>
    <thead>
      <tr>
        <th>실행 ID</th>
        <th>시작</th>
        <th>종료</th>
        <th class="num">단계 수</th>
        <th class="num">입력 수</th>
      </tr>
    </thead>
    <tbody>{ex_html}</tbody>
  </table>
  </div>
</section>
"""
        )

    if not cards:
        cards=[
            """
<section class="workflow-card">
<div class="empty">보고 대상으로 선택된 업무 흐름이 없습니다.</div>
</section>
"""
        ]

    scenario_table_rows=[]

    if len(
        scenario_comparison
    ):
        for r in scenario_comparison.itertuples(
            index=False
        ):
            coverage=(
                f'{float(r.coverage_ratio)*100:.0f}%'
            )

            order=(
                f'{float(r.order_ratio)*100:.0f}%'
            )

            best=(
                safe_str(
                    getattr(
                        r,
                        'best_trajectory_workflow_prototypes',
                        ''
                    )
                )
                or safe_str(
                    r.best_workflow_prototype
                )
                or '미검출'
            )

            missing=(
                safe_str(
                    r.missing_steps
                )
                or '없음'
            )

            user_evidence=(
                safe_str(
                    getattr(
                        r,
                        'best_trajectory_user_display_id',
                        ''
                    )
                )
                or '-'
            )

            support_users=int(
                getattr(
                    r,
                    'supporting_user_count',
                    0
                )
                or 0
            )

            session_count=int(
                getattr(
                    r,
                    'best_trajectory_session_count',
                    0
                )
                or 0
            )

            instance_count=int(
                getattr(
                    r,
                    'best_trajectory_instance_count',
                    0
                )
                or 0
            )

            scenario_table_rows.append(
                '<tr>'
                f'<td>{html_lib.escape(safe_str(r.owner))}</td>'
                f'<td>{html_lib.escape(safe_str(r.business_area))}</td>'
                f'<td>{html_lib.escape(best)}</td>'
                f'<td>{html_lib.escape(user_evidence)}</td>'
                f'<td class="num">{support_users}</td>'
                f'<td class="num">{instance_count} / {session_count}</td>'
                f'<td class="num">{int(r.matched_step_count)} / {int(r.ground_truth_step_count)}</td>'
                f'<td class="num">{coverage}</td>'
                f'<td class="num">{order}</td>'
                f'<td>{html_lib.escape(missing)}</td>'
                '</tr>'
            )

    scenario_table_html=''.join(
        scenario_table_rows
    )


    report=f"""<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>업무 프로세스 분석 보고서 · {html_lib.escape(MODEL_VERSION)}</title>
<style>
*{{box-sizing:border-box}}
body{{margin:0;background:#f3f5f8;color:#1f2937;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif}}
.wrap{{max-width:1450px;margin:0 auto;padding:32px}}
.hero,.workflow-card,.method{{background:#fff;border:1px solid #e1e6ed;border-radius:18px}}
.hero{{padding:28px;margin-bottom:20px}}
.hero h1{{margin:0 0 8px;font-size:28px}}
.hero p{{margin:0;color:#667085;line-height:1.65}}
.summary{{display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:12px;margin:20px 0}}
.summary>div,.metrics>div{{background:#fff;border:1px solid #e1e6ed;border-radius:14px;padding:15px}}
.summary b,.metrics b{{display:block;font-size:24px;margin-bottom:5px}}
.summary span,.metrics span{{font-size:12px;color:#737b87}}
.workflow-card{{padding:24px;margin-bottom:18px}}
.workflow-head{{display:flex;justify-content:space-between;gap:20px;align-items:flex-start}}
.eyebrow{{font-size:12px;font-weight:800;color:#667085}}
.workflow-card h2{{font-size:22px;margin:5px 0 14px}}
.workflow-card h3{{font-size:14px;margin:18px 0 10px}}
.status{{padding:6px 10px;border-radius:999px;font-size:11px;font-weight:800;white-space:nowrap}}
.status.good{{background:#e8f6ee;color:#267348}}
.status.review{{background:#fff4d9;color:#805b0b}}
.metrics{{display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:10px}}
.metrics b{{font-size:20px}}
.two-col{{display:grid;grid-template-columns:1.15fr .85fr;gap:18px}}
.path-box{{border:1px solid #edf0f3;border-radius:14px;padding:14px;background:#fafbfc}}
.step{{padding:9px 12px;background:white;border:1px solid #dfe4ea;border-radius:10px;font-size:13px;font-weight:700}}
.arrow{{text-align:center;color:#98a2b3;font-size:18px;line-height:24px}}
.input-chips{{display:flex;flex-wrap:wrap;gap:7px}}
.input-chip{{padding:6px 9px;background:#f2f4f7;border-radius:999px;font-size:12px}}
.score-note{{margin-top:14px;padding:12px;background:#f8fafc;border-radius:12px;font-size:12px;line-height:1.65;color:#667085}}
.scroll{{overflow:auto}}
table{{border-collapse:collapse;width:100%;font-size:12px}}
th,td{{padding:9px;border-bottom:1px solid #edf0f3;text-align:left;white-space:nowrap}}
th{{background:#fafbfc;color:#667085}}
.num{{text-align:right}}
.method{{padding:20px;font-size:12px;line-height:1.7;color:#667085}}
.scenario-panel{{background:#fff;border:1px solid #e1e6ed;border-radius:18px;padding:22px;margin:20px 0}}
.scenario-panel h2{{margin:0 0 8px;font-size:18px;color:#1f2937}}
.scenario-panel .note{{color:#667085;font-size:12px;line-height:1.6;margin-bottom:12px}}
.empty{{padding:18px;color:#7b818a;background:#fafafa;border-radius:10px}}
.muted{{color:#7b818a;font-size:12px}}
@media(max-width:900px){{
 .wrap{{padding:14px}}
 .summary,.metrics{{grid-template-columns:1fr 1fr}}
 .two-col{{grid-template-columns:1fr}}
}}
</style>
</head>
<body>
<div class="wrap">

<section class="hero">
<h1>업무 프로세스 분석 보고서</h1>
<p>
원시 이벤트 로그를 기반으로 반복적으로 관찰되는 업무 흐름을 비지도/자기지도 방식으로 추출한 결과입니다.
업무명은 화면/입력/단계 근거에서 자동 생성한 표시명이며 모델 학습 label이 아닙니다. V6.2의 사용자 업무 궤적은 sessions.user_id를 학습 feature가 아닌 사후 grouping key로만 사용합니다.
</p>
</section>

<section class="summary">
<div><b>{len(df):,}</b><span>분석 이벤트</span></div>
<div><b>{len(workflow_df):,}</b><span>Latent workflow</span></div>
<div><b>{len(process_instances):,}</b><span>실행 사례</span></div>
<div><b>{len(process_models):,}</b><span>발견 업무 후보</span></div>
<div><b>{len(selected):,}</b><span>보고 대상 업무 흐름</span></div>
</section>

<section class="method">
<b>분석 기간</b> {html_lib.escape(safe_str(period_start))} ~ {html_lib.escape(safe_str(period_end))}<br>
<b>보고서 선정 기준</b> instance support, multi-seed consensus stability, consensus margin,
대표 경로 일관성, evidence richness, structural coherence를 함께 사용합니다.<br>
<b>상태</b> Reportable은 내부 품질 기준을 통과한 후보이며,
Review candidate는 고객 목표인 최소 {REPORT_TARGET_MIN_WORKFLOWS}개 흐름을 검토할 수 있도록
상위 후보에서 추가 포함한 항목입니다.
</section>

<section class="scenario-panel">
<h2>PoC Ground Truth 7개 시나리오 사후 비교</h2>
<div class="note">
아래 Ground Truth는 모델 학습에 사용하지 않고, 모델 결과가 생성된 뒤 sessions.user_id로 연결한 동일 사용자의 업무 궤적과 사후 비교한 결과입니다.
세부 클릭/조회/팝업 단계는 유지하며, 사람이 해석할 수 없는 SPxx 단독 단계만 경로 표시에서 제외합니다.
</div>
<div class="scroll">
<table>
<thead>
<tr>
<th>담당</th>
<th>업무</th>
<th>관련 모델 패턴</th>
<th>대표 사용자 궤적</th>
<th class="num">관찰 사용자</th>
<th class="num">Instance / Session</th>
<th class="num">기준 단계 확인</th>
<th class="num">단계 검출</th>
<th class="num">기준 순서 재현</th>
<th>미확인 기준 단계</th>
</tr>
</thead>
<tbody>{scenario_table_html}</tbody>
</table>
</div>
</section>

{''.join(cards)}

<section class="method">
<b>해석 주의</b><br>
이 보고서는 자동 발견된 latent workflow를 고객 검토가 가능한 형태로 정리한 PoC 산출물입니다.
Gold 업무명은 clustering에 사용하지 않았습니다.
실제 운영 반영 전에는 담당자 검토를 통해 표시명과 업무 경계의 최종 확인을 권장합니다.
</section>

</div>
</body>
</html>
"""

    path=MAIN_RESULTS_DIR/'customer_workflow_report.html'

    path.write_text(
        report,
        encoding='utf-8'
    )

    return path


customer_workflow_report_path=build_customer_workflow_report()

print(
    'customer workflow report:',
    customer_workflow_report_path
)

print(
    'selected workflow count:',
    len(
        reportable_workflows
    )
)

print(
    reportable_workflows[
        [
            'report_rank',
            'workflow_prototype',
            'customer_display_name',
            'instance_count',
            'reportability_score',
            'report_status',
        ]
    ].to_string(
        index=False
    )
    if len(
        reportable_workflows
    )
    else 'no selected workflow'
)


# ============================================================
# L. Post-hoc Gold Evaluation — urangi six known processes
# ============================================================
# IMPORTANT:
# - 학습 / boundary / clustering / prototype assignment에는 절대 사용하지 않음.
# - 이미 추출된 process instance / step / input 결과에 대해서만 평가.
# - exact/alias phrase anchor가 등장한 unique instance를 집계하는 근사 평가.
# - true event boundary ground truth가 아니므로 "accuracy"가 아니라
#   separation / purity diagnostic으로 해석한다.

GOLD_PROCESSES = [
    {
        'gold_id':
            'G01',
        'gold_path':
            '학사행정>국제교류관리>기초관리>국제교류학년도 학기',
        'leaf_name':
            '국제교류학년도학기',
        'aliases': [
            '국제교류학년도학기',
            '국제교류학년도 학기',
        ],
    },
    {
        'gold_id':
            'G02',
        'gold_path':
            '학사행정>국제교류관리>기초관리>교류기관관리',
        'leaf_name':
            '교류기관관리',
        'aliases': [
            '교류기관관리',
        ],
    },
    {
        'gold_id':
            'G03',
        'gold_path':
            '학사행정>국제교류관리>교류학생관리>교류학생관리(out)',
        'leaf_name':
            '교류학생관리(OUT)',
        'aliases': [
            '교류학생관리(out)',
            '교류학생(out)',
        ],
    },
    {
        'gold_id':
            'G04',
        'gold_path':
            '학사행정>국제교류관리>교류학생관리>교류학생관리(in)',
        'leaf_name':
            '교류학생관리(IN)',
        'aliases': [
            '교류학생관리(in)',
            '교류학생(in)',
        ],
    },
    {
        'gold_id':
            'G05',
        'gold_path':
            '학사행정>국제교류관리유학생기숙사관리>유학생기숙사호실관리',
        'leaf_name':
            '유학생기숙사호실관리',
        'aliases': [
            '유학생기숙사호실관리',
        ],
    },
    {
        'gold_id':
            'G06',
        'gold_path':
            '학사행정>국제교류관리유학생기숙사관리>유학생기숙사납부내역',
        'leaf_name':
            '유학생기숙사납부내역',
        'aliases': [
            '유학생기숙사납부내역',
        ],
    },
]


def _gold_norm(
    x
):
    """
    평가용 normalization.
    업무 rule을 학습에 넣지 않고, anchor phrase 비교에만 사용.
    """
    s = safe_str(
        x
    ).lower()

    s = re.sub(
        r'\s+',
        '',
        s
    )

    return s


def _concat_unique_text(
    values
):
    seen = []
    seen_set = set()

    for v in values:
        s = safe_str(
            v
        )

        if (
            s
            and s not in seen_set
        ):
            seen.append(
                s
            )
            seen_set.add(
                s
            )

    return ' | '.join(
        seen
    )


def build_gold_eval_corpus():
    base_cols = [
        'instance_id',
        'model_id',
        'workflow_prototype',
    ]

    base = process_instances[
        base_cols
    ].drop_duplicates(
        'instance_id'
    ).copy()

    if len(
        process_instance_steps
    ):
        step_text = (
            process_instance_steps
            .assign(
                _eval_text=lambda x: (
                    x[
                        'display_label'
                    ].map(
                        safe_str
                    )
                    + ' | '
                    + x[
                        'representative_evidence'
                    ].map(
                        safe_str
                    )
                )
            )
            .groupby(
                'instance_id',
                sort=False
            )[
                '_eval_text'
            ]
            .agg(
                _concat_unique_text
            )
            .rename(
                'step_eval_text'
            )
            .reset_index()
        )

        base = base.merge(
            step_text,
            on='instance_id',
            how='left'
        )

    else:
        base[
            'step_eval_text'
        ] = ''

    if len(
        process_node_inputs
    ):
        input_text = (
            process_node_inputs
            .assign(
                _eval_text=lambda x: (
                    x[
                        'input_label'
                    ].map(
                        safe_str
                    )
                    + ' | '
                    + x[
                        'input_value'
                    ].map(
                        safe_str
                    )
                )
            )
            .groupby(
                'instance_id',
                sort=False
            )[
                '_eval_text'
            ]
            .agg(
                _concat_unique_text
            )
            .rename(
                'input_eval_text'
            )
            .reset_index()
        )

        base = base.merge(
            input_text,
            on='instance_id',
            how='left'
        )

    else:
        base[
            'input_eval_text'
        ] = ''

    base[
        'eval_text'
    ] = (
        base[
            'step_eval_text'
        ].fillna(
            ''
        )
        + ' | '
        + base[
            'input_eval_text'
        ].fillna(
            ''
        )
    )

    base[
        'eval_text_norm'
    ] = base[
        'eval_text'
    ].map(
        _gold_norm
    )

    return base


def evaluate_known_processes():
    empty_metrics = {
        'enabled':
            False,
        'evaluation_only':
            True,
        'reason':
            'gold evaluation skipped',
    }

    if not ENABLE_GOLD_EVAL:
        return (
            pd.DataFrame(),
            pd.DataFrame(),
            empty_metrics
        )

    if safe_str(
        DATASET_NAME
    ).lower() != safe_str(
        GOLD_EVAL_DATASET
    ).lower():
        empty_metrics[
            'reason'
        ] = (
            'dataset does not match GOLD_EVAL_DATASET'
        )

        return (
            pd.DataFrame(),
            pd.DataFrame(),
            empty_metrics
        )

    corpus = build_gold_eval_corpus()

    model_order = (
        process_models[
            'workflow_prototype'
        ].tolist()
        if len(
            process_models
        )
        else sorted(
            corpus[
                'workflow_prototype'
            ].dropna().unique().tolist()
        )
    )

    hit_rows = []

    for gold in GOLD_PROCESSES:
        aliases = [
            _gold_norm(
                a
            )
            for a in gold[
                'aliases'
            ]
        ]

        hit_mask = corpus[
            'eval_text_norm'
        ].map(
            lambda text: any(
                a in text
                for a in aliases
                if a
            )
        )

        matched = corpus[
            hit_mask
        ][
            [
                'instance_id',
                'model_id',
                'workflow_prototype'
            ]
        ].drop_duplicates(
            'instance_id'
        )

        for row in matched.itertuples(
            index=False
        ):
            hit_rows.append(
                {
                    'gold_id':
                        gold[
                            'gold_id'
                        ],
                    'gold_path':
                        gold[
                            'gold_path'
                        ],
                    'leaf_name':
                        gold[
                            'leaf_name'
                        ],
                    'instance_id':
                        row.instance_id,
                    'model_id':
                        row.model_id,
                    'workflow_prototype':
                        row.workflow_prototype,
                }
            )

    hits = pd.DataFrame(
        hit_rows
    )

    pair_rows=[]
    support_map={}
    for gold in GOLD_PROCESSES:
        gid=gold['gold_id']
        support_map[gid]=set(hits[hits['gold_id']==gid]['instance_id'].tolist()) if len(hits) else set()
    for i in range(len(GOLD_PROCESSES)):
        for j in range(i+1,len(GOLD_PROCESSES)):
            ga,gb=GOLD_PROCESSES[i],GOLD_PROCESSES[j]
            sa,sb=support_map[ga['gold_id']],support_map[gb['gold_id']]
            inter=len(sa&sb); union=len(sa|sb); min_support=min(len(sa),len(sb))
            pair_rows.append({
                'gold_id_a':ga['gold_id'],'leaf_name_a':ga['leaf_name'],
                'gold_id_b':gb['gold_id'],'leaf_name_b':gb['leaf_name'],
                'support_a':len(sa),'support_b':len(sb),
                'same_instance_count':inter,
                'cooccurrence_rate_min_support':inter/min_support if min_support else np.nan,
                'jaccard':inter/union if union else np.nan,
            })
    pair_df=pd.DataFrame(pair_rows)

    if len(
        hits
    ):
        confusion = (
            hits
            .groupby(
                [
                    'gold_id',
                    'gold_path',
                    'leaf_name',
                    'workflow_prototype'
                ]
            )[
                'instance_id'
            ]
            .nunique()
            .unstack(
                fill_value=0
            )
        )

        for wp in model_order:
            if wp not in confusion.columns:
                confusion[
                    wp
                ] = 0

        extra_cols = [
            c
            for c in confusion.columns
            if c not in model_order
        ]

        confusion = confusion[
            model_order
            + extra_cols
        ]

        confusion = confusion.reset_index()

    else:
        confusion = pd.DataFrame(
            columns=[
                'gold_id',
                'gold_path',
                'leaf_name',
                *model_order
            ]
        )

    summary_rows = []

    numeric_cols = [
        c
        for c in confusion.columns
        if c not in {
            'gold_id',
            'gold_path',
            'leaf_name'
        }
    ]

    for gold in GOLD_PROCESSES:
        row = confusion[
            confusion[
                'gold_id'
            ]
            == gold[
                'gold_id'
            ]
        ]

        if len(
            row
        ):
            counts = (
                row[
                    numeric_cols
                ]
                .iloc[
                    0
                ]
                .astype(
                    int
                )
            )

        else:
            counts = pd.Series(
                0,
                index=numeric_cols,
                dtype=int
            )

        total = int(
            counts.sum()
        )

        if total > 0:
            dominant_wp = str(
                counts.idxmax()
            )

            dominant_count = int(
                counts.max()
            )

            purity = (
                dominant_count
                / total
            )

        else:
            dominant_wp = ''

            dominant_count = 0

            purity = float(
                'nan'
            )

        summary_rows.append(
            {
                'gold_id':
                    gold[
                        'gold_id'
                    ],
                'gold_path':
                    gold[
                        'gold_path'
                    ],
                'leaf_name':
                    gold[
                        'leaf_name'
                    ],
                'anchor_instance_count':
                    total,
                'dominant_workflow_prototype':
                    dominant_wp,
                'dominant_count':
                    dominant_count,
                'purity':
                    purity,
                'contamination':
                    (
                        1.0
                        - purity
                        if np.isfinite(
                            purity
                        )
                        else np.nan
                    ),
            }
        )

    summary = pd.DataFrame(
        summary_rows
    )

    valid_dominants = summary[
        summary[
            'dominant_workflow_prototype'
        ]
        != ''
    ][
        'dominant_workflow_prototype'
    ]

    unique_dominant = int(
        valid_dominants.nunique()
    )

    mean_purity = float(
        summary[
            'purity'
        ].dropna().mean()
    ) if summary[
        'purity'
    ].notna().any() else None

    # --------------------------------------------------------
    # Hungarian 1:1 separation score
    # --------------------------------------------------------
    hungarian_score = None
    matched_count = 0
    total_anchor_assignments = 0
    assignment_pairs = []

    if (
        len(
            confusion
        )
        and numeric_cols
    ):
        matrix = confusion[
            numeric_cols
        ].to_numpy(
            dtype=float
        )

        total_anchor_assignments = int(
            matrix.sum()
        )

        try:
            from scipy.optimize import linear_sum_assignment

            row_ind, col_ind = linear_sum_assignment(
                -matrix
            )

            matched_count = int(
                matrix[
                    row_ind,
                    col_ind
                ].sum()
            )

            if total_anchor_assignments > 0:
                hungarian_score = (
                    matched_count
                    / total_anchor_assignments
                )

            for r, c in zip(
                row_ind,
                col_ind
            ):
                assignment_pairs.append(
                    {
                        'gold_id':
                            str(
                                confusion.iloc[
                                    r
                                ][
                                    'gold_id'
                                ]
                            ),
                        'leaf_name':
                            str(
                                confusion.iloc[
                                    r
                                ][
                                    'leaf_name'
                                ]
                            ),
                        'workflow_prototype':
                            str(
                                numeric_cols[
                                    c
                                ]
                            ),
                        'count':
                            int(
                                matrix[
                                    r,
                                    c
                                ]
                            ),
                    }
                )

        except Exception as e:
            print(
                'Hungarian evaluation skipped:',
                repr(
                    e
                )
            )

    pair_rates=pair_df['cooccurrence_rate_min_support'].dropna() if len(pair_df) else pd.Series(dtype=float)
    max_pair_rate=float(pair_rates.max()) if len(pair_rates) else None
    mean_pair_rate=float(pair_rates.mean()) if len(pair_rates) else None

    metrics = {
        'enabled':
            True,
        'evaluation_only':
            True,
        'used_for_training':
            False,
        'method':
            (
                'post-hoc exact/alias phrase matching over exported '
                'instance step + input text; counts are unique instances'
            ),
        'gold_process_count':
            int(
                len(
                    GOLD_PROCESSES
                )
            ),
        'gold_processes': [
            {
                'gold_id':
                    g[
                        'gold_id'
                    ],
                'gold_path':
                    g[
                        'gold_path'
                    ],
                'leaf_name':
                    g[
                        'leaf_name'
                    ],
            }
            for g in GOLD_PROCESSES
        ],
        'unique_dominant_workflow_count':
            unique_dominant,
        'max_unique_dominant_workflow_count':
            int(
                len(
                    GOLD_PROCESSES
                )
            ),
        'dominant_collision_count':
            int(
                len(
                    GOLD_PROCESSES
                )
                - unique_dominant
            ),
        'mean_anchor_purity':
            mean_purity,
        'hungarian_1to1_separation_score':
            (
                float(
                    hungarian_score
                )
                if hungarian_score is not None
                else None
            ),
        'hungarian_matched_anchor_assignments':
            int(
                matched_count
            ),
        'total_anchor_assignments':
            int(
                total_anchor_assignments
            ),
        'hungarian_assignment_pairs':
            assignment_pairs,
        'per_process': (
            summary
            .replace(
                {
                    np.nan:
                        None
                }
            )
            .to_dict(
                orient='records'
            )
        ),
        'gold_pair_cooccurrence':pair_df.replace({np.nan:None}).to_dict(orient='records'),
        'max_pair_cooccurrence_rate_min_support':max_pair_rate,
        'mean_pair_cooccurrence_rate_min_support':mean_pair_rate,
        'interpretation_note':
            (
                'Anchor-based diagnostic only. This is not event-level '
                'ground-truth accuracy because exact start/end boundaries '
                'for the six real processes are not labeled.'
            ),
    }

    return (
        confusion,
        summary,
        metrics
    )


gold_confusion_df, gold_summary_df, gold_eval_metrics = (
    evaluate_known_processes()
)

gold_pair_df=pd.DataFrame(gold_eval_metrics.get('gold_pair_cooccurrence',[]))

if gold_eval_metrics.get(
    'enabled'
):
    gold_confusion_df.to_csv(
        DIAGNOSTICS_DIR
        / 'gold_anchor_confusion.csv',
        index=False,
        encoding='utf-8-sig'
    )

    gold_summary_df.to_csv(
        DIAGNOSTICS_DIR
        / 'gold_anchor_summary.csv',
        index=False,
        encoding='utf-8-sig'
    )

    gold_pair_df.to_csv(
        DIAGNOSTICS_DIR / 'gold_pair_cooccurrence.csv',
        index=False,encoding='utf-8-sig'
    )

    print(
        '\n=== V6.2 LEGACY GOLD EVALUATION ==='
    )

    print(
        gold_summary_df[
            [
                'leaf_name',
                'anchor_instance_count',
                'dominant_workflow_prototype',
                'purity'
            ]
        ].to_string(
            index=False
        )
    )

    print(
        'unique dominant workflows:',
        gold_eval_metrics[
            'unique_dominant_workflow_count'
        ],
        '/',
        gold_eval_metrics[
            'max_unique_dominant_workflow_count'
        ]
    )

    print(
        'mean anchor purity:',
        gold_eval_metrics[
            'mean_anchor_purity'
        ]
    )

    print(
        'Hungarian 1:1 separation:',
        gold_eval_metrics[
            'hungarian_1to1_separation_score'
        ]
    )

    print('max gold-pair same-instance rate:',gold_eval_metrics['max_pair_cooccurrence_rate_min_support'])
    if len(gold_pair_df):
        print(gold_pair_df.sort_values('cooccurrence_rate_min_support',ascending=False).to_string(index=False))


# ============================================================
# K. V5.7 Evaluation v2 — provenance-aware / sensitivity-aware
# ============================================================

def _gold_aliases_norm():
    return {
        g[
            'gold_id'
        ]:[
            _gold_norm(
                a
            )
            for a in g[
                'aliases'
            ]
            if _gold_norm(
                a
            )
        ]
        for g in GOLD_PROCESSES
    }


def _text_has_alias(
    text,
    aliases
):
    t=_gold_norm(
        text
    )

    return any(
        a in t
        for a in aliases
        if a
    )


def _instance_raw_events():
    """
    process_instance_steps의 source_segment_ids를 통해
    실제 raw event를 instance 단위로 다시 모은다.
    """
    seg_id_to_idx={
        safe_str(
            r.segment_id
        ):
            int(
                r.segment_idx
            )
        for r in segdf[
            [
                'segment_id',
                'segment_idx'
            ]
        ].itertuples(
            index=False
        )
    }

    result={}

    if not len(
        process_instance_steps
    ):
        return result

    for instance_id,g in process_instance_steps.groupby(
        'instance_id',
        sort=False
    ):
        seg_ids=[]

        for raw in g[
            'source_segment_ids'
        ].tolist():
            for sid in safe_str(
                raw
            ).split(
                '|'
            ):
                sid=sid.strip()

                if (
                    sid
                    and sid not in seg_ids
                ):
                    seg_ids.append(
                        sid
                    )

        frames=[]

        for sid in seg_ids:
            if sid not in seg_id_to_idx:
                continue

            frames.append(
                _segment_events(
                    seg_id_to_idx[
                        sid
                    ]
                )
            )

        if frames:
            ev=pd.concat(
                frames,
                ignore_index=True
            )

            if 'global_event_idx' in ev.columns:
                ev=ev.drop_duplicates(
                    'global_event_idx'
                )

            ev=ev.sort_values(
                [
                    'event_time_parsed',
                    'global_event_idx'
                ],
                kind='stable'
            ).reset_index(
                drop=True
            )

        else:
            ev=pd.DataFrame()

        result[
            safe_str(
                instance_id
            )
        ]=ev

    return result


def _build_eval_v2_anchor_stats():
    aliases_map=_gold_aliases_norm()
    instance_events=_instance_raw_events()

    rows=[]

    for instance_id,ev in instance_events.items():
        if len(
            ev
        )==0:
            continue

        last_global_idx=(
            int(
                ev[
                    'global_event_idx'
                ].iloc[
                    -1
                ]
            )
            if 'global_event_idx' in ev.columns
            else None
        )

        for gold in GOLD_PROCESSES:
            gid=gold[
                'gold_id'
            ]

            aliases=aliases_map[
                gid
            ]

            context_times=[]
            context_event_ids=[]
            navigation_event_ids=[]
            any_event_ids=[]
            no_terminal_nav_event_ids=[]

            for _,er in ev.iterrows():
                interaction=safe_str(
                    er.get(
                        'interaction_type'
                    )
                ).lower()

                page_title=safe_str(
                    er.get(
                        'page_title'
                    )
                )

                page_url=safe_str(
                    er.get(
                        'page_url'
                    )
                )

                element_text=safe_str(
                    er.get(
                        'element_text'
                    )
                )

                associated_label=safe_str(
                    er.get(
                        'associated_label'
                    )
                )

                global_idx=(
                    int(
                        er.get(
                            'global_event_idx'
                        )
                    )
                    if pd.notna(
                        er.get(
                            'global_event_idx'
                        )
                    )
                    else None
                )

                event_key=(
                    safe_str(
                        er.get(
                            'event_id'
                        )
                    )
                    or safe_str(
                        er.get(
                            'id'
                        )
                    )
                    or safe_str(
                        global_idx
                    )
                )

                # V6: current-page context는 page title/URL뿐 아니라
                # non-navigation event의 label/text에서도 수집한다.
                # 메뉴 클릭 자체는 navigation provenance로 분리한다.
                non_navigation_text_hit=(
                    interaction!='click'
                    and (
                        _text_has_alias(
                            element_text,
                            aliases
                        )
                        or _text_has_alias(
                            associated_label,
                            aliases
                        )
                    )
                )

                context_hit=(
                    _text_has_alias(
                        page_title,
                        aliases
                    )
                    or _text_has_alias(
                        page_url,
                        aliases
                    )
                    or non_navigation_text_hit
                )

                nav_hit=(
                    interaction=='click'
                    and (
                        _text_has_alias(
                            element_text,
                            aliases
                        )
                        or _text_has_alias(
                            associated_label,
                            aliases
                        )
                    )
                    and not (
                        _text_has_alias(
                            page_title,
                            aliases
                        )
                        or _text_has_alias(
                            page_url,
                            aliases
                        )
                    )
                )

                any_hit=(
                    context_hit
                    or _text_has_alias(
                        element_text,
                        aliases
                    )
                    or _text_has_alias(
                        associated_label,
                        aliases
                    )
                )

                terminal_nav=(
                    nav_hit
                    and last_global_idx is not None
                    and global_idx==last_global_idx
                )

                if context_hit:
                    context_event_ids.append(
                        event_key
                    )

                    ts=pd.to_datetime(
                        er.get(
                            'event_time_parsed'
                        ),
                        errors='coerce'
                    )

                    if pd.notna(
                        ts
                    ):
                        context_times.append(
                            ts
                        )

                if nav_hit:
                    navigation_event_ids.append(
                        event_key
                    )

                if any_hit:
                    any_event_ids.append(
                        event_key
                    )

                    if not terminal_nav:
                        no_terminal_nav_event_ids.append(
                            event_key
                        )

            if len(
                context_times
            )>=2:
                dwell=(
                    max(
                        context_times
                    )
                    - min(
                        context_times
                    )
                ).total_seconds()

            else:
                dwell=0.0

            rows.append(
                {
                    'instance_id':
                        instance_id,
                    'gold_id':
                        gid,
                    'leaf_name':
                        gold[
                            'leaf_name'
                        ],
                    'context_hit_count':
                        int(
                            len(
                                set(
                                    context_event_ids
                                )
                            )
                        ),
                    'navigation_hit_count':
                        int(
                            len(
                                set(
                                    navigation_event_ids
                                )
                            )
                        ),
                    'raw_any_hit_count':
                        int(
                            len(
                                set(
                                    any_event_ids
                                )
                            )
                        ),
                    'no_terminal_nav_hit_count':
                        int(
                            len(
                                set(
                                    no_terminal_nav_event_ids
                                )
                            )
                        ),
                    'context_dwell_sec':
                        float(
                            dwell
                        ),
                }
            )

    return pd.DataFrame(
        rows
    )


def _legacy_label_sets():
    corpus=build_gold_eval_corpus()
    aliases_map=_gold_aliases_norm()
    result={}

    for r in corpus.itertuples(
        index=False
    ):
        labels=set()

        text=safe_str(
            r.eval_text_norm
        )

        for gold in GOLD_PROCESSES:
            gid=gold[
                'gold_id'
            ]

            if any(
                a in text
                for a in aliases_map[
                    gid
                ]
                if a
            ):
                labels.add(
                    gid
                )

        result[
            safe_str(
                r.instance_id
            )
        ]=labels

    return result


def _label_sets_from_stats(
    stats,
    mode
):
    result={
        safe_str(
            x
        ):
            set()
        for x in process_instances[
            'instance_id'
        ].tolist()
    }

    if mode=='legacy_loose':
        legacy=_legacy_label_sets()

        for k,v in legacy.items():
            result[
                k
            ]=set(
                v
            )

        return result

    for r in stats.itertuples(
        index=False
    ):
        accept=False

        if mode=='no_terminal_nav':
            accept=(
                r.no_terminal_nav_hit_count
                >=1
            )

        elif mode=='context_ge1':
            accept=(
                r.context_hit_count
                >=1
            )

        elif mode=='context_ge2':
            accept=(
                r.context_hit_count
                >=2
            )

        elif mode=='context_ge2_dwell3':
            accept=(
                r.context_hit_count
                >= EVAL_V2_PERSISTENT_MIN_HITS
                and r.context_dwell_sec
                >= EVAL_V2_PERSISTENT_MIN_DWELL_SEC
            )

        else:
            raise ValueError(
                mode
            )

        if accept:
            result.setdefault(
                safe_str(
                    r.instance_id
                ),
                set()
            ).add(
                safe_str(
                    r.gold_id
                )
            )

    return result


def _pair_stats_from_label_sets(
    label_sets,
    mode
):
    support={
        g[
            'gold_id'
        ]:
            set()
        for g in GOLD_PROCESSES
    }

    for instance_id,labels in label_sets.items():
        for gid in labels:
            if gid in support:
                support[
                    gid
                ].add(
                    instance_id
                )

    rows=[]

    for i in range(
        len(
            GOLD_PROCESSES
        )
    ):
        for j in range(
            i+1,
            len(
                GOLD_PROCESSES
            )
        ):
            a=GOLD_PROCESSES[
                i
            ]

            b=GOLD_PROCESSES[
                j
            ]

            sa=support[
                a[
                    'gold_id'
                ]
            ]

            sb=support[
                b[
                    'gold_id'
                ]
            ]

            inter=len(
                sa&sb
            )

            min_support=min(
                len(
                    sa
                ),
                len(
                    sb
                )
            )

            union=len(
                sa|sb
            )

            rows.append(
                {
                    'mode':
                        mode,
                    'gold_id_a':
                        a[
                            'gold_id'
                        ],
                    'leaf_name_a':
                        a[
                            'leaf_name'
                        ],
                    'gold_id_b':
                        b[
                            'gold_id'
                        ],
                    'leaf_name_b':
                        b[
                            'leaf_name'
                        ],
                    'support_a':
                        int(
                            len(
                                sa
                            )
                        ),
                    'support_b':
                        int(
                            len(
                                sb
                            )
                        ),
                    'same_instance_count':
                        int(
                            inter
                        ),
                    'cooccurrence_rate_min_support':
                        (
                            inter/min_support
                            if min_support
                            else np.nan
                        ),
                    'jaccard':
                        (
                            inter/union
                            if union
                            else np.nan
                        ),
                }
            )

    return pd.DataFrame(
        rows
    )


def _cluster_metrics_for_mode(
    label_sets,
    mode
):
    pred_map=(
        process_instances[
            [
                'instance_id',
                'workflow_prototype'
            ]
        ]
        .drop_duplicates(
            'instance_id'
        )
        .set_index(
            'instance_id'
        )[
            'workflow_prototype'
        ]
        .to_dict()
    )

    canonical=[]
    ambiguous=0
    labeled=0
    instance_rows=[]

    for instance_id,pred in pred_map.items():
        labels=set(
            label_sets.get(
                safe_str(
                    instance_id
                ),
                set()
            )
        )

        if labels:
            labeled+=1

        if len(
            labels
        )>1:
            ambiguous+=1

        if len(
            labels
        )==1:
            gold_id=next(
                iter(
                    labels
                )
            )

            canonical.append(
                (
                    safe_str(
                        instance_id
                    ),
                    gold_id,
                    safe_str(
                        pred
                    )
                )
            )

        instance_rows.append(
            {
                'mode':
                    mode,
                'instance_id':
                    safe_str(
                        instance_id
                    ),
                'workflow_prototype':
                    safe_str(
                        pred
                    ),
                'gold_ids':
                    '|'.join(
                        sorted(
                            labels
                        )
                    ),
                'gold_count':
                    int(
                        len(
                            labels
                        )
                    ),
                'is_canonical_single_gold':
                    int(
                        len(
                            labels
                        )==1
                    ),
            }
        )

    if canonical:
        y_true=[
            x[
                1
            ]
            for x in canonical
        ]

        y_pred=[
            x[
                2
            ]
            for x in canonical
        ]

        ari=float(
            adjusted_rand_score(
                y_true,
                y_pred
            )
        )

        nmi=float(
            normalized_mutual_info_score(
                y_true,
                y_pred
            )
        )

        hom=float(
            homogeneity_score(
                y_true,
                y_pred
            )
        )

        comp=float(
            completeness_score(
                y_true,
                y_pred
            )
        )

        vmeasure=float(
            v_measure_score(
                y_true,
                y_pred
            )
        )

        canonical_df=pd.DataFrame(
            canonical,
            columns=[
                'instance_id',
                'gold_id',
                'workflow_prototype'
            ]
        )

        purity_by_gold=[]

        for gid,g in canonical_df.groupby(
            'gold_id'
        ):
            vc=g[
                'workflow_prototype'
            ].value_counts()

            purity_by_gold.append(
                float(
                    vc.iloc[
                        0
                    ]
                    / vc.sum()
                )
            )

        macro_purity=(
            float(
                np.mean(
                    purity_by_gold
                )
            )
            if purity_by_gold
            else None
        )

        represented_gold=int(
            canonical_df[
                'gold_id'
            ].nunique()
        )

    else:
        ari=None
        nmi=None
        hom=None
        comp=None
        vmeasure=None
        macro_purity=None
        represented_gold=0

    pair_df=_pair_stats_from_label_sets(
        label_sets,
        mode
    )

    rates=(
        pair_df[
            'cooccurrence_rate_min_support'
        ].dropna()
        if len(
            pair_df
        )
        else pd.Series(
            dtype=float
        )
    )

    metrics={
        'mode':
            mode,
        'total_process_instances':
            int(
                len(
                    pred_map
                )
            ),
        'labeled_instance_count':
            int(
                labeled
            ),
        'canonical_single_gold_count':
            int(
                len(
                    canonical
                )
            ),
        'ambiguous_multi_gold_count':
            int(
                ambiguous
            ),
        'unlabeled_instance_count':
            int(
                len(
                    pred_map
                )
                - labeled
            ),
        'represented_gold_processes':
            represented_gold,
        'ari':
            ari,
        'nmi':
            nmi,
        'homogeneity':
            hom,
        'completeness':
            comp,
        'v_measure':
            vmeasure,
        'macro_purity_reference':
            macro_purity,
        'max_pair_cooccurrence':
            (
                float(
                    rates.max()
                )
                if len(
                    rates
                )
                else None
            ),
        'mean_pair_cooccurrence':
            (
                float(
                    rates.mean()
                )
                if len(
                    rates
                )
                else None
            ),
        'median_pair_cooccurrence':
            (
                float(
                    rates.median()
                )
                if len(
                    rates
                )
                else None
            ),
        'p90_pair_cooccurrence':
            (
                float(
                    rates.quantile(
                        .90
                    )
                )
                if len(
                    rates
                )
                else None
            ),
    }

    return (
        metrics,
        pd.DataFrame(
            instance_rows
        ),
        pair_df
    )


evaluation_v2_anchor_provenance=(
    _build_eval_v2_anchor_stats()
)

evaluation_v2_modes=[
    'legacy_loose',
    'no_terminal_nav',
    'context_ge1',
    'context_ge2',
    'context_ge2_dwell3',
]

evaluation_v2_summary_rows=[]
evaluation_v2_instance_frames=[]
evaluation_v2_pair_frames=[]
evaluation_v2_mode_metrics={}

for mode in evaluation_v2_modes:
    label_sets=_label_sets_from_stats(
        evaluation_v2_anchor_provenance,
        mode
    )

    metrics,instance_df,pair_df=(
        _cluster_metrics_for_mode(
            label_sets,
            mode
        )
    )

    evaluation_v2_mode_metrics[
        mode
    ]=metrics

    evaluation_v2_summary_rows.append(
        metrics
    )

    evaluation_v2_instance_frames.append(
        instance_df
    )

    evaluation_v2_pair_frames.append(
        pair_df
    )


evaluation_v2_summary=pd.DataFrame(
    evaluation_v2_summary_rows
)

evaluation_v2_instance_labels=pd.concat(
    evaluation_v2_instance_frames,
    ignore_index=True
) if evaluation_v2_instance_frames else pd.DataFrame()

evaluation_v2_pair_cooccurrence=pd.concat(
    evaluation_v2_pair_frames,
    ignore_index=True
) if evaluation_v2_pair_frames else pd.DataFrame()

evaluation_v2_primary=(
    evaluation_v2_mode_metrics.get(
        EVAL_V2_PRIMARY_MODE,
        {}
    )
)

evaluation_v2_metrics={
    'enabled':
        True,
    'evaluation_only':
        True,
    'used_for_training':
        False,
    'primary_mode':
        EVAL_V2_PRIMARY_MODE,
    'primary':
        evaluation_v2_primary,
    'modes':
        evaluation_v2_mode_metrics,
    'canonical_definition':
        (
            'ARI/NMI/V-measure are computed only on instances '
            'with exactly one accepted gold label under each mode. '
            'Multi-gold instances are excluded from clustering metrics '
            'and retained for boundary/co-occurrence diagnostics.'
        ),
    'anchor_provenance':
        (
            'page_title/page_url context hits are separated from '
            'navigation click hits. context modes exclude navigation '
            'anchors from canonical gold assignment.'
        ),
}

evaluation_v2_anchor_provenance.to_csv(
    DIAGNOSTICS_DIR
    / 'evaluation_v2_anchor_provenance.csv',
    index=False,
    encoding='utf-8-sig'
)

evaluation_v2_summary.to_csv(
    DIAGNOSTICS_DIR
    / 'evaluation_v2_summary.csv',
    index=False,
    encoding='utf-8-sig'
)

evaluation_v2_instance_labels.to_csv(
    DIAGNOSTICS_DIR
    / 'evaluation_v2_instance_labels.csv',
    index=False,
    encoding='utf-8-sig'
)

evaluation_v2_pair_cooccurrence.to_csv(
    DIAGNOSTICS_DIR
    / 'evaluation_v2_pair_cooccurrence.csv',
    index=False,
    encoding='utf-8-sig'
)

print(
    '\\n=== V5.7 EVALUATION V2 ==='
)

print(
    evaluation_v2_summary.to_string(
        index=False
    )
)


# K2. Customer-facing Workflow Quality Report — Evaluation v2
# ============================================================

def _report_pct(v, digits=1):
    try:
        if v is None:
            return '-'
        x=float(v)
        if not np.isfinite(x):
            return '-'
        return f"{x*100:.{digits}f}%"
    except Exception:
        return '-'


def _report_num(v, digits=3):
    try:
        if v is None:
            return '-'
        x=float(v)
        if not np.isfinite(x):
            return '-'
        return f"{x:.{digits}f}"
    except Exception:
        return '-'


def _version_sort_key(v):
    m=re.search(
        r'(\d+)\.(\d+)',
        safe_str(v)
    )

    return (
        int(m.group(1)),
        int(m.group(2))
    ) if m else (0,0)


def _load_legacy_version_rows():
    rows=[]
    artifacts_root=BASE_DIR/'artifacts'

    if artifacts_root.exists():
        for p in artifacts_root.glob(
            'business_flow_deep_v*/run_manifest.json'
        ):
            try:
                if p.parent==OUTPUT_DIR:
                    continue

                data=json.loads(
                    p.read_text(
                        encoding='utf-8'
                    )
                )

                gold=data.get(
                    'gold_evaluation',
                    {}
                ) or {}

                rows.append({
                    'version':
                        data.get(
                            'version',
                            p.parent.name
                        ),
                    'process_models':
                        data.get(
                            'process_models'
                        ),
                    'process_nodes':
                        data.get(
                            'process_nodes'
                        ),
                    'unique_dominant':
                        gold.get(
                            'unique_dominant_workflow_count'
                        ),
                    'mean_purity':
                        gold.get(
                            'mean_anchor_purity'
                        ),
                    'separation':
                        gold.get(
                            'hungarian_1to1_separation_score'
                        ),
                    'max_pair':
                        gold.get(
                            'max_pair_cooccurrence_rate_min_support'
                        ),
                })

            except Exception:
                pass

    rows.append({
        'version':
            MODEL_VERSION,
        'process_models':
            int(
                len(
                    process_models
                )
            ),
        'process_nodes':
            int(
                len(
                    process_nodes
                )
            ),
        'unique_dominant':
            gold_eval_metrics.get(
                'unique_dominant_workflow_count'
            ),
        'mean_purity':
            gold_eval_metrics.get(
                'mean_anchor_purity'
            ),
        'separation':
            gold_eval_metrics.get(
                'hungarian_1to1_separation_score'
            ),
        'max_pair':
            gold_eval_metrics.get(
                'max_pair_cooccurrence_rate_min_support'
            ),
    })

    dedup={}

    for r in rows:
        dedup[
            safe_str(
                r[
                    'version'
                ]
            )
        ]=r

    return sorted(
        dedup.values(),
        key=lambda r:_version_sort_key(
            r[
                'version'
            ]
        )
    )


def build_workflow_quality_report():
    primary=evaluation_v2_primary

    cards=[
        (
            '추출 프로세스',
            str(
                len(
                    process_models
                )
            ),
            'calibrated threshold 적용 후 발견된 process model 수'
        ),
        (
            f'ARI · {EVAL_V2_PRIMARY_MODE}',
            _report_num(
                primary.get(
                    'ari'
                )
            ),
            'single-gold canonical subset의 Adjusted Rand Index'
        ),
        (
            'V-measure',
            _report_num(
                primary.get(
                    'v_measure'
                )
            ),
            'homogeneity와 completeness의 조화평균'
        ),
        (
            'NMI',
            _report_num(
                primary.get(
                    'nmi'
                )
            ),
            '기준 업무와 추출 prototype의 normalized mutual information'
        ),
        (
            'Canonical 평가 사례',
            str(
                primary.get(
                    'canonical_single_gold_count',
                    '-'
                )
            ),
            'gold label이 정확히 하나인 instance만 clustering 지표에 사용'
        ),
        (
            'Ambiguous 사례',
            str(
                primary.get(
                    'ambiguous_multi_gold_count',
                    '-'
                )
            ),
            '둘 이상의 gold anchor가 잡혀 clustering 지표에서 제외된 instance'
        ),
    ]

    card_html=''.join(
        '<div class="card">'
        f'<div class="card-title">{html_lib.escape(a)}</div>'
        f'<div class="card-value">{html_lib.escape(b)}</div>'
        f'<div class="card-note">{html_lib.escape(c)}</div>'
        '</div>'
        for a,b,c in cards
    )

    # Evaluation sensitivity
    sensitivity_rows=''

    for r in evaluation_v2_summary.itertuples(
        index=False
    ):
        is_primary=(
            safe_str(
                r.mode
            )
            == EVAL_V2_PRIMARY_MODE
        )

        cls=(
            ' class="current"'
            if is_primary
            else ''
        )

        sensitivity_rows+=(
            f'<tr{cls}>'
            f'<td>{html_lib.escape(safe_str(r.mode))}</td>'
            f'<td class="num">{int(r.canonical_single_gold_count)}</td>'
            f'<td class="num">{int(r.ambiguous_multi_gold_count)}</td>'
            f'<td class="num">{html_lib.escape(_report_num(r.ari))}</td>'
            f'<td class="num">{html_lib.escape(_report_num(r.nmi))}</td>'
            f'<td class="num">{html_lib.escape(_report_num(r.homogeneity))}</td>'
            f'<td class="num">{html_lib.escape(_report_num(r.completeness))}</td>'
            f'<td class="num">{html_lib.escape(_report_num(r.v_measure))}</td>'
            f'<td class="num">{html_lib.escape(_report_pct(r.macro_purity_reference))}</td>'
            f'<td class="num">{html_lib.escape(_report_pct(r.max_pair_cooccurrence))}</td>'
            '</tr>'
        )

    # Threshold signal summary
    threshold_rows=''

    for r in workflow_threshold_signal_summary.itertuples(
        index=False
    ):
        current=(
            r.signal=='combined'
            and r.threshold_method=='calibrated'
        )

        cls=(
            ' class="current"'
            if current
            else ''
        )

        threshold_rows+=(
            f'<tr{cls}>'
            f'<td>{html_lib.escape(safe_str(r.signal))}</td>'
            f'<td>{html_lib.escape(safe_str(r.threshold_method))}</td>'
            f'<td class="num">{int(r.workflow_count)}</td>'
            f'<td class="num">{int(r.cut_count)}</td>'
            f'<td class="num">{html_lib.escape(_report_num(r.score_median))}</td>'
            f'<td class="num">{html_lib.escape(_report_num(r.score_p90))}</td>'
            '</tr>'
        )

    # Cut Jaccard
    jaccard_rows=''

    for r in workflow_cut_jaccard.itertuples(
        index=False
    ):
        jaccard_rows+=(
            '<tr>'
            f'<td>{html_lib.escape(safe_str(r.comparison_type))}</td>'
            f'<td>{html_lib.escape(safe_str(r.signal_a))}</td>'
            f'<td>{html_lib.escape(safe_str(r.signal_b))}</td>'
            f'<td class="num">{int(r.cuts_a)}</td>'
            f'<td class="num">{int(r.cuts_b)}</td>'
            f'<td class="num">{int(r.shared_cuts)}</td>'
            f'<td class="num">{html_lib.escape(_report_num(r.jaccard))}</td>'
            '</tr>'
        )

    # Prototype stability
    stability_rows=''

    for level,diag in prototype_stability_diagnostics.items():
        stability_rows+=(
            '<tr>'
            f'<td>{html_lib.escape(level)}</td>'
            f'<td class="num">{html_lib.escape(_report_num(diag.get("pairwise_ari_mean")))}</td>'
            f'<td class="num">{html_lib.escape(_report_num(diag.get("pairwise_ari_min")))}</td>'
            f'<td class="num">{html_lib.escape(_report_num(diag.get("pairwise_nmi_mean")))}</td>'
            f'<td class="num">{html_lib.escape(_report_num(diag.get("mean_confidence_across_seeds")))}</td>'
            f'<td class="num">{html_lib.escape(_report_num(diag.get("mean_prob_margin_across_seeds")))}</td>'
            '</tr>'
        )

    # Legacy historical comparison
    legacy_rows=''

    for r in _load_legacy_version_rows():
        current=(
            safe_str(
                r[
                    'version'
                ]
            )
            == MODEL_VERSION
        )

        cls=(
            ' class="current"'
            if current
            else ''
        )

        legacy_rows+=(
            f'<tr{cls}>'
            f'<td>{html_lib.escape(safe_str(r.get("version")))}</td>'
            f'<td class="num">{html_lib.escape(safe_str(r.get("process_models","-")))}</td>'
            f'<td class="num">{html_lib.escape(safe_str(r.get("process_nodes","-")))}</td>'
            f'<td class="num">{html_lib.escape(safe_str(r.get("unique_dominant","-")))}</td>'
            f'<td class="num">{html_lib.escape(_report_pct(r.get("mean_purity")))}</td>'
            f'<td class="num">{html_lib.escape(_report_pct(r.get("separation")))}</td>'
            f'<td class="num">{html_lib.escape(_report_pct(r.get("max_pair")))}</td>'
            '</tr>'
        )

    report=f"""<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>업무 프로세스 내부 품질 리포트 {html_lib.escape(MODEL_VERSION)}</title>
<style>
*{{box-sizing:border-box}}
body{{margin:0;background:#f5f7fa;color:#20242a;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif}}
.wrap{{max-width:1550px;margin:0 auto;padding:28px}}
.hero,.panel,.card{{background:#fff;border:1px solid #e2e6ec;border-radius:16px}}
.hero{{padding:24px;margin-bottom:18px}}
.hero h1{{margin:0 0 8px;font-size:24px}}
.hero p,.note,.footer{{color:#68707c;line-height:1.65}}
.cards{{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px;margin:18px 0}}
.card{{padding:16px}}
.card-title{{font-size:12px;font-weight:700;color:#747c87}}
.card-value{{font-size:27px;font-weight:800;margin:7px 0}}
.card-note{{font-size:12px;line-height:1.5;color:#7b838e}}
.panel{{padding:20px;margin-bottom:18px}}
.panel h2{{font-size:18px;margin:0 0 6px}}
.note{{font-size:12px;margin-bottom:14px}}
.scroll{{overflow:auto}}
table{{border-collapse:collapse;width:100%;font-size:13px}}
th,td{{padding:10px 9px;border-bottom:1px solid #edf0f3;text-align:left;white-space:nowrap}}
th{{background:#fafbfc;font-size:12px;color:#5f6772}}
.num{{text-align:right}}
.current td{{font-weight:700;background:#f1f5ff}}
.badge{{display:inline-block;padding:3px 7px;border-radius:999px;background:#eef3ff;color:#3757a4;font-size:11px;font-weight:700}}
.footer{{font-size:12px;padding:5px 2px 20px}}
@media(max-width:800px){{
 .wrap{{padding:14px}}
 .cards{{grid-template-columns:1fr}}
}}
</style>
</head>
<body>
<div class="wrap">

<section class="hero">
<h1>업무 프로세스 내부 품질 리포트 · {html_lib.escape(MODEL_VERSION)}</h1>
<p>
V6.2은 calibrated segmentation과 multi-seed consensus prototype을 사용해 고객 보고 가능한 업무 흐름을 안정적으로 추출하는 버전입니다.
Gold 업무명은 학습과 threshold 결정에 사용하지 않습니다.
<span class="badge">Primary evaluation: {html_lib.escape(EVAL_V2_PRIMARY_MODE)}</span>
</p>
</section>

<section class="cards">{card_html}</section>

<section class="panel">
<h2>Evaluation v2 · anchor sensitivity</h2>
<div class="note">
ARI/NMI/V-measure는 gold label이 정확히 하나인 canonical instance에만 계산합니다.
여러 gold가 동시에 잡힌 instance는 clustering 지표에서 제외하고 boundary/co-occurrence 진단에 남깁니다.
현재 primary mode는 강조 표시됩니다.
</div>
<div class="scroll">
<table>
<thead><tr>
<th>Anchor mode</th>
<th class="num">Canonical N</th>
<th class="num">Ambiguous N</th>
<th class="num">ARI</th>
<th class="num">NMI</th>
<th class="num">Homogeneity</th>
<th class="num">Completeness</th>
<th class="num">V-measure</th>
<th class="num">Purity 참고</th>
<th class="num">Max pair 병합</th>
</tr></thead>
<tbody>{sensitivity_rows}</tbody>
</table>
</div>
</section>

<section class="panel">
<h2>Boundary threshold calibration</h2>
<div class="note">
같은 boundary signal에 대해 기존 GMM midpoint와 V5.7 calibrated threshold가
workflow 개수와 cut 위치를 어떻게 바꾸는지 비교합니다.
</div>
<div class="scroll">
<table>
<thead><tr>
<th>Signal</th>
<th>Threshold</th>
<th class="num">Workflow 수</th>
<th class="num">Cut 수</th>
<th class="num">Score median</th>
<th class="num">Score p90</th>
</tr></thead>
<tbody>{threshold_rows}</tbody>
</table>
</div>
</section>

<section class="panel">
<h2>Cut location Jaccard</h2>
<div class="note">
workflow 개수만 보지 않고 실제 cut 위치가 얼마나 겹치는지 확인합니다.
</div>
<div class="scroll">
<table>
<thead><tr>
<th>비교 유형</th>
<th>Signal A</th>
<th>Signal B</th>
<th class="num">Cuts A</th>
<th class="num">Cuts B</th>
<th class="num">공유 Cut</th>
<th class="num">Jaccard</th>
</tr></thead>
<tbody>{jaccard_rows}</tbody>
</table>
</div>
</section>

<section class="panel">
<h2>Prototype seed stability</h2>
<div class="note">
Prototype은 collapse 여부가 아니라 seed에 따라 assignment가 얼마나 유지되는지,
그리고 top1-top2 probability margin이 얼마나 확보되는지를 평가합니다.
</div>
<div class="scroll">
<table>
<thead><tr>
<th>Level</th>
<th class="num">Pairwise ARI mean</th>
<th class="num">Pairwise ARI min</th>
<th class="num">Pairwise NMI mean</th>
<th class="num">Mean confidence</th>
<th class="num">Mean top1-top2 margin</th>
</tr></thead>
<tbody>{stability_rows}</tbody>
</table>
</div>
</section>

<section class="panel">
<h2>Legacy version comparison · 참고용</h2>
<div class="note">
V5.3~V5.6의 기존 purity/Hungarian 지표는 Evaluation v2와 정의가 다르므로 직접적인 성능 비교값이 아닙니다.
과거 추세 확인용으로만 남깁니다.
</div>
<div class="scroll">
<table>
<thead><tr>
<th>버전</th>
<th class="num">추출 프로세스</th>
<th class="num">노드</th>
<th class="num">대표 WP</th>
<th class="num">평균 집중도</th>
<th class="num">Legacy 1:1 분리</th>
<th class="num">Legacy 최대 병합</th>
</tr></thead>
<tbody>{legacy_rows}</tbody>
</table>
</div>
</section>

<div class="footer">
Evaluation v2의 context mode는 navigation click과 현재 페이지 context를 분리합니다.
ARI/NMI/V-measure는 canonical single-gold subset에서만 계산됩니다.
tasks.csv는 business gold label로 사용하지 않습니다.
</div>

</div>
</body>
</html>
"""

    path=DIAGNOSTICS_DIR/'workflow_quality_report.html'

    path.write_text(
        report,
        encoding='utf-8'
    )

    return path


workflow_quality_report_path=(
    build_workflow_quality_report()
)

print(
    'customer quality report:',
    workflow_quality_report_path
)


# L. Export CSV
# ============================================================

csv_outputs = {
    'process_models.csv':
        process_models,
    'process_nodes.csv':
        process_nodes,
    'process_edges.csv':
        process_edges,
    'process_instances.csv':
        process_instances,
    'process_instance_steps.csv':
        process_instance_steps,
    'process_node_inputs.csv':
        process_node_inputs,
    'reportable_workflows.csv':
        reportable_workflows,
    'reportable_workflow_steps.csv':
        reportable_workflow_steps,
    'poc_scenario_comparison.csv':
        scenario_comparison,
    'workflow_reportability.csv':
        (
            process_models[
                [
                    c
                    for c in [
                        'model_id',
                        'workflow_prototype',
                        'customer_display_name',
                        'instance_count',
                        'reportability_score',
                        'support_score',
                        'stability_score',
                        'margin_score',
                        'path_consistency',
                        'evidence_richness',
                        'structural_coherence',
                        'report_status',
                        'selected_for_customer_report',
                        'representative_path',
                    ]
                    if c in process_models.columns
                ]
            ].copy()
            if len(
                process_models
            )
            else pd.DataFrame()
        ),
}

for filename, frame in csv_outputs.items():
    frame.to_csv(
        MAIN_RESULTS_DIR
        / filename,
        index=False,
        encoding='utf-8-sig'
    )


# ============================================================
# M. Developer handoff Markdown
# ============================================================

developer_handoff_md = f"""
# 업무 프로세스 가시화 개발자 전달 명세 — V6.2

## 1. 목적

본 산출물은 딥러닝 모델이 추출한 latent workflow를 개발자가 화면에 시각화할 수 있도록 전달하기 위한 중간 명세입니다.

현재 단계에서는 DB schema를 확정하지 않습니다.
아래 CSV 구조와 HTML preview를 기준으로 실제 서비스 DB/화면 구현 가능 여부를 먼저 검토합니다.

### V6.2 변경 — Stable Consensus Workflow + Customer Report

V6.2은 **V5.7 calibrated segmentation을 유지**하고,
single-seed prototype 대신 multi-seed consensus를 최종 process/node assignment에 사용합니다.

- 유지: frozen multilingual semantic encoder (`intfloat/multilingual-e5-small`)
- 유지: semantic weight `{SEMANTIC_WEIGHT:.2f}`
- 유지: Structural-aware Event Boundary
- 유지: Segment Transformer / workflow mean pooling
- 유지: Transition Contrastive Workflow Boundary
- 유지: V5.7 calibrated event/workflow threshold
- 변경: prototype full-batch 20 step → 최대 `{PROTOTYPE_OPT_STEPS}` optimization step + early stopping
- 변경: single-seed SP/WP → `{len(PROTOTYPE_ENSEMBLE_SEEDS)}`-seed consensus SP/WP
- 추가: reportability score 및 고객 보고 대상 workflow 선정
- 추가: `customer_workflow_report.html`
- 추가: `reportable_workflows.csv` / `reportable_workflow_steps.csv`
- Gold 업무명은 training / threshold / consensus clustering / reportability score에 사용하지 않음

### urangi Gold Evaluation

알려진 6개 실제 업무 프로세스는 **학습에 사용하지 않습니다.**
최종 추출 결과에 대해서만 post-hoc anchor evaluation을 수행합니다.

평가 산출물:

```text
diagnostics/gold_anchor_confusion.csv
diagnostics/gold_anchor_summary.csv
diagnostics/gold_pair_cooccurrence.csv
diagnostics/evaluation_v2_anchor_provenance.csv
diagnostics/evaluation_v2_summary.csv
diagnostics/evaluation_v2_instance_labels.csv
diagnostics/evaluation_v2_pair_cooccurrence.csv
diagnostics/event_threshold_diagnostics.csv
diagnostics/workflow_threshold_diagnostics.csv
diagnostics/workflow_threshold_signal_summary.csv
diagnostics/workflow_cut_jaccard.csv
diagnostics/prototype_seed_stability.csv
diagnostics/workflow_quality_report.html
diagnostics/workflow_transition_training.csv
diagnostics/workflow_boundary_scores.csv
diagnostics/prototype_ensemble_runs.csv
diagnostics/prototype_seed_pairwise.csv
diagnostics/consensus_k_search.csv
diagnostics/workflow_consensus_assignments.csv
```

V6.2 고객 화면 / 사용자 궤적 변경:

- `sessions.session_id → sessions.user_id`를 이용해 동일 사용자의 process instance를 시간순으로 연결
- `user_id`는 event embedding / prototype clustering feature로 사용하지 않음
- 동일 사용자의 여러 session을 generic temporal episode/window로 연결해 상위 시나리오 후보 생성
- Ground Truth 비교는 single instance뿐 아니라 user trajectory를 primary presentation으로 사용
- 고객 HTML에는 raw user_id 대신 stable hash 기반 `USER-XXXXXXXX` 표시
- 7개 PoC Ground Truth Scenario는 모델 학습 이후 post-hoc 비교에만 사용
- compressed path를 Aggregate process graph보다 먼저 표시
- 조회/팝업/필드/달력 등 사람이 읽을 수 있는 하위 단계는 유지
- `SPxx` / `WPxx`처럼 표시명이 prototype ID뿐인 단계만 HTML path에서 숨김
- Ground Truth와 일치하는 단계는 `✓ Ground Truth`로 강조
- 기준 단계 확인 수 / 기준 순서 재현 / 미확인 기준 단계를 함께 표시
- 전체 `events.csv` row를 제한 없이 사용

고객 전달용 주요 산출물:

```text
main_results/customer_workflow_report.html
main_results/reportable_workflows.csv
main_results/reportable_workflow_steps.csv
main_results/workflow_reportability.csv
```

평가 지표는 workflow separation을 보기 위한 근사 diagnostic이며,
event-level start/end ground truth accuracy로 해석하지 않습니다.


V5.7부터는 기존 anchor 평가는 `legacy` 비교용으로 유지하고,
공식 비교는 `Evaluation v2`를 사용합니다.

- navigation click과 current-page context를 분리
- multi-gold instance는 ARI/NMI에서 제외
- anchor sensitivity를 여러 모드로 동시에 공개
- purity/Hungarian은 참고값으로만 사용

---

## 2. 모델 결과 개요

- dataset: `{DATASET_NAME}`
- events: {len(df):,}
- latent segments: {len(segdf):,}
- latent workflows: {len(workflow_df):,}
- session-safe process instances: {len(process_instances):,}
- process models: {len(process_models):,}
- process nodes: {len(process_nodes):,}
- process edges: {len(process_edges):,}
- compressed instance steps: {len(process_instance_steps):,}
- process node inputs: {len(process_node_inputs):,}
- customer report selected workflows: {len(reportable_workflows):,}
- strict reportable workflows: {int((reportable_workflows['report_status']=='Reportable').sum()) if len(reportable_workflows) else 0:,}

---

## 3. V6.2의 중요 변경

### Session-safe instance

session을 business workflow 정답으로 사용하지 않습니다.

다만 하나의 실제 실행 사례가 서로 다른 session을 넘으면 화면 해석이 어려워지므로,
export 단계에서 `session_id`를 **최대 경계**로만 사용합니다.

즉:

```text
한 session 안에서 여러 workflow 가능
서로 다른 session을 하나의 표시용 instance로 연결하지 않음
```

### Consecutive prototype compression

모델이 만든 원본 sequence:

```text
SP09 → SP09 → SP09 → SP22
```

화면/DB 전달용 sequence:

```text
SP09 (repeat_count=3) → SP22
```

모델 학습 결과를 변경하는 것이 아니라 presentation/export 단계에서만 압축합니다.

### 입력값

QA/테스트 목적이므로 V6.2에서는 `input_data`를 추가 masking하지 않습니다.

- 원본 값이 있으면 그대로 `input_value`로 전달
- 원본 로그가 이미 `[REDACTED]` 또는 `[IDENTITY_CONTEXT]`이면 그 문자열을 그대로 유지
- `is_source_redacted`는 원본이 이미 가려져 있었는지를 표시

---

## 4. 전달 파일

### `process_models.csv`

프로세스 목록의 최상위 단위.

주요 컬럼:
- `model_id`
- `workflow_prototype`
- `display_name`
- `instance_count`
- `node_count`
- `edge_count`
- `avg_compressed_step_count`
- `avg_source_segment_count`
- `input_count`
- `first_seen`
- `last_seen`

### `process_nodes.csv`

프로세스 맵의 node.

주요 컬럼:
- `node_id`
- `model_id`
- `segment_prototype`
- `display_label`
- `sort_order`
- `support_step_count`
- `support_segment_count`
- `support_instance_count`
- `avg_step_position`
- `avg_repeat_count`
- `input_count`
- `representative_evidence`

`display_label`은 presentation layer에서 생성한 이름이며 학습 라벨이 아닙니다.

HTML preview의 업무 프로세스 명칭도 같은 원칙으로,
`WPxx · 추출 업무 프로세스 NN` 형식의 중립적 표시명을 사용합니다.
이 변경은 `workflow_prototype`, `model_id`, clustering 결과, CSV의 원본 `display_name`을 변경하지 않습니다.

### `process_edges.csv`

실제 압축된 instance sequence에서 관찰된 node transition.

주요 컬럼:
- `from_node_id`
- `to_node_id`
- `transition_count`
- `transition_ratio`

V6.2 preview는 이 edge를 실제 graph topology로 사용합니다.

### `process_instances.csv`

session-safe 실제 실행 사례.

주요 컬럼:
- `instance_id`
- `model_id`
- `source_workflow_id`
- `session_id`
- `session_part`
- `start_time`
- `end_time`
- `compressed_step_count`
- `source_segment_count`
- `input_count`

### `process_instance_steps.csv`

**특정 실제 실행의 전체 step sequence**를 복원하기 위한 파일.

주요 컬럼:
- `instance_step_id`
- `instance_id`
- `step_order`
- `node_id`
- `segment_prototype`
- `display_label`
- `repeat_count`
- `source_segment_ids`
- `start_time`
- `end_time`
- `event_count`
- `input_count`

화면에서 특정 실행을 선택했을 때 이 파일을 `instance_id + step_order` 순으로 조회하면 됩니다.

### `process_node_inputs.csv`

각 실행 step에 연결된 실제 입력/선택 값.

주요 컬럼:
- `instance_id`
- `instance_step_id`
- `node_id`
- `step_order`
- `input_kind`
- `input_label`
- `input_value`
- `is_direct_user_input`
- `is_source_redacted`

---

## 5. 데이터 관계

```text
process_models
    │
    ├── process_nodes
    │       │
    │       └── process_edges
    │
    └── process_instances
            │
            └── process_instance_steps
                    │
                    └── process_node_inputs
```

권장 연결:
- `process_models.model_id` → nodes / edges / instances
- `process_instances.instance_id` → instance_steps
- `process_instance_steps.instance_step_id` → node_inputs
- `process_nodes.node_id` → edges / instance_steps / node_inputs

---

## 6. 권장 화면

### A. Process model 선택
`process_models`

### B. Aggregate process graph
`process_nodes + process_edges`

- node = 업무 step prototype
- edge = 실제 관찰 transition
- transition ratio가 낮은 edge는 화면에서 필터링 가능
- branch는 여러 outgoing edge로 표현

### C. 실제 실행 사례 선택
`process_instances`

### D. 선택한 실행의 정확한 path
`process_instance_steps`

예:

```text
학생검색
  ↓
프로그램선택
  ↓
학생정보입력 (반복 3회)
  ↓
저장
```

### E. 당시 입력값
`process_node_inputs`

`instance_step_id` 기준으로 조회하여 해당 step 아래에 표시합니다.

---

## 7. 개발자 검토 포인트

1. 현재 6개 CSV 구조로 기존 DB에 적재 가능한지
2. `process_instance_steps`를 별도 테이블로 두는 것이 적절한지
3. 문자열 ID를 그대로 사용할지 내부 numeric PK를 둘지
4. `input_value`를 row 단위로 유지할지 JSON으로 묶을지
5. 관리자용 process/node 표시명 수정 컬럼이 필요한지
6. aggregate graph와 actual instance path를 화면에서 어떻게 구분할지
7. transition ratio threshold UI가 필요한지
8. graph library 선택
9. workflow prototype 버전 관리 필요 여부

---

## 8. 중요한 해석 원칙

- `WPxx` / `SPxx`는 모델이 발견한 latent prototype
- session은 workflow 정답이 아니라 export instance의 최대 경계
- 연속 SP compression은 presentation layer 처리
- `display_label`은 presentation layer 처리
- input 값은 workflow 학습 라벨로 사용하지 않음
"""

(
    MAIN_RESULTS_DIR
    / 'developer_handoff.md'
).write_text(
    developer_handoff_md.strip()
    + '\n',
    encoding='utf-8'
)


# ============================================================
# L. Preview payload
# ============================================================

def _json_safe_records(frame):
    if frame is None or not len(frame):
        return []

    temp = frame.copy()

    for col in temp.columns:
        if pd.api.types.is_datetime64_any_dtype(
            temp[
                col
            ]
        ):
            temp[
                col
            ] = temp[
                col
            ].astype(
                str
            )

    temp = temp.where(
        pd.notna(
            temp
        ),
        None
    )

    return temp.to_dict(
        orient='records'
    )


preview_model_ids = (
    process_models
    .head(
        PREVIEW_MAX_MODELS
    )[
        'model_id'
    ]
    .tolist()
    if len(
        process_models
    )
    else []
)

# V6.2: 7개 PoC scenario의 대표 user trajectory에 포함되는 model은
# reportability rank와 관계없이 preview에 포함한다.
if len(
    scenario_comparison
):
    _trajectory_model_ids=[]

    for raw in scenario_comparison[
        'best_trajectory_model_ids'
    ].fillna(
        ''
    ).tolist():
        for mid in safe_str(
            raw
        ).split(
            '|'
        ):
            mid=mid.strip()

            if (
                mid
                and mid not in _trajectory_model_ids
            ):
                _trajectory_model_ids.append(
                    mid
                )

    for mid in _trajectory_model_ids:
        if (
            mid in process_models[
                'model_id'
            ].tolist()
            and mid not in preview_model_ids
        ):
            preview_model_ids.append(
                mid
            )

preview_models = (
    process_models[
        process_models[
            'model_id'
        ].isin(
            preview_model_ids
        )
    ].copy()
)


# ============================================================
# HTML presentation-only workflow names
# ============================================================
# 모델이 자동 생성한 semantic display_name은 내부 CSV에 그대로 보존합니다.
# 다만 HTML preview에서는 과도한 의미 해석을 피하기 위해
# 중립적인 이름으로 표시합니다.
#
# 예:
#   WP01 · 조회 / 이전년 / 국제교류학년도학기
#       ↓
#   WP01 · 추출 업무 프로세스 01
#
# 번호는 HTML 목록에 보이는 순서 기준입니다.
if len(
    preview_models
):
    preview_models = preview_models.reset_index(
        drop=True
    )

    preview_models[
        'display_name'
    ] = [
        (
            f"{wp} · "
            f"추출 업무 프로세스 {i:02d}"
        )
        for i, wp in enumerate(
            preview_models[
                'workflow_prototype'
            ].tolist(),
            start=1
        )
    ]

preview_nodes = (
    process_nodes[
        process_nodes[
            'model_id'
        ].isin(
            preview_model_ids
        )
    ].copy()
)

preview_edges = (
    process_edges[
        process_edges[
            'model_id'
        ].isin(
            preview_model_ids
        )
    ].copy()
)

preview_instances_parts = []

for model_id in preview_model_ids:
    g = (
        process_instances[
            process_instances[
                'model_id'
            ]
            == model_id
        ]
        .sort_values(
            [
                'input_count',
                'compressed_step_count',
                'start_time'
            ],
            ascending=[
                False,
                False,
                True
            ]
        )
        .head(
            PREVIEW_INSTANCES_PER_MODEL
        )
    )

    # V6.2: single-instance best뿐 아니라 user trajectory의 모든 instance를
    # input/step 수와 관계없이 HTML preview에 반드시 포함한다.
    _scenario_best_ids=set()

    if len(
        scenario_comparison
    ):
        _scenario_best_ids.update(
            scenario_comparison.loc[
                scenario_comparison[
                    'best_model_id'
                ]==model_id,
                'best_instance_id'
            ].dropna(
            ).astype(
                str
            ).tolist()
        )

        for _,sr in scenario_comparison.iterrows():
            raw_ids=safe_str(
                sr.get(
                    'best_trajectory_instance_ids'
                )
            )

            if not raw_ids:
                continue

            for iid in raw_ids.split(
                '|'
            ):
                iid=iid.strip()

                if not iid:
                    continue

                _row=process_instances[
                    process_instances[
                        'instance_id'
                    ].astype(
                        str
                    )==iid
                ]

                if (
                    len(
                        _row
                    )
                    and safe_str(
                        _row[
                            'model_id'
                        ].iloc[
                            0
                        ]
                    )==model_id
                ):
                    _scenario_best_ids.add(
                        iid
                    )

    if _scenario_best_ids:
        _extra = process_instances[
            (
                process_instances[
                    'model_id'
                ]==model_id
            )
            & (
                process_instances[
                    'instance_id'
                ].astype(
                    str
                ).isin(
                    _scenario_best_ids
                )
            )
        ]

        g=pd.concat(
            [
                g,
                _extra
            ],
            ignore_index=True
        ).drop_duplicates(
            'instance_id'
        )

    preview_instances_parts.append(
        g
    )

preview_instances = (
    pd.concat(
        preview_instances_parts,
        ignore_index=True
    )
    if preview_instances_parts
    else pd.DataFrame(
        columns=process_instances.columns
    )
)

preview_instance_ids = set(
    preview_instances[
        'instance_id'
    ].tolist()
) if len(
    preview_instances
) else set()

preview_steps = (
    process_instance_steps[
        process_instance_steps[
            'instance_id'
        ].isin(
            preview_instance_ids
        )
    ].copy()
)

preview_inputs = (
    process_node_inputs[
        process_node_inputs[
            'instance_id'
        ].isin(
            preview_instance_ids
        )
    ].copy()
)

if len(
    preview_inputs
):
    limited_inputs = []

    for iid, g in preview_inputs.groupby(
        'instance_id',
        sort=False
    ):
        limited_inputs.append(
            g
            .sort_values(
                [
                    'step_order',
                    'event_time'
                ]
            )
            .head(
                PREVIEW_MAX_INPUTS_PER_INSTANCE
            )
        )

    preview_inputs = pd.concat(
        limited_inputs,
        ignore_index=True
    )


preview_payload = {
    'models':
        _json_safe_records(
            preview_models
        ),
    'nodes':
        _json_safe_records(
            preview_nodes
        ),
    'edges':
        _json_safe_records(
            preview_edges
        ),
    'instances':
        _json_safe_records(
            preview_instances
        ),
    'steps':
        _json_safe_records(
            preview_steps
        ),
    'inputs':
        _json_safe_records(
            preview_inputs
        ),
    'scenarios':
        POC_SCENARIOS,
    'scenario_matches':
        _json_safe_records(
            scenario_comparison
        ),
}

preview_json = (
    json.dumps(
        preview_payload,
        ensure_ascii=False,
        default=str
    )
    .replace(
        '</',
        '<\\/'
    )
)


# ============================================================
# M. Actual-edge SVG workflow preview
# ============================================================

preview_html = f"""<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>업무 프로세스 미리보기 V6.2</title>
<style>
*{{box-sizing:border-box}}
body{{margin:0;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;background:#f6f7f9;color:#202124}}
.wrap{{max-width:1600px;margin:0 auto;padding:24px}}
.toolbar{{display:flex;gap:14px;flex-wrap:wrap;align-items:end;margin-bottom:16px}}
.field{{display:flex;flex-direction:column;gap:6px;min-width:250px}}
label{{font-size:12px;color:#6b7280;font-weight:700}}
select{{min-height:42px;padding:8px 12px;border:1px solid #d6d9df;border-radius:10px;background:white;font-size:14px}}
.summary{{display:flex;gap:8px;flex-wrap:wrap;margin:8px 0 16px}}
.badge{{background:white;border:1px solid #e2e5ea;border-radius:999px;padding:6px 10px;font-size:12px}}
.panel{{background:white;border:1px solid #e2e5ea;border-radius:14px;padding:16px;margin-bottom:16px}}
.panel h3{{margin:0 0 12px;font-size:16px}}
.graph-scroll{{overflow:auto;width:100%;border:1px solid #eef0f3;border-radius:12px;background:#fbfbfc}}
#graphSvg{{display:block;min-width:900px}}
.edge{{fill:none;stroke:#adb4bf;stroke-width:1.5;opacity:.72}}
.edge.instance{{stroke:#5865d8;stroke-width:3;opacity:.95}}
.node-rect{{fill:white;stroke:#cfd5dd;stroke-width:1.2;rx:12}}
.node-rect.instance{{stroke:#5865d8;stroke-width:2.5}}
.node-rect.selected{{stroke:#202124;stroke-width:3}}
.node-label{{font-size:13px;font-weight:700;fill:#202124;pointer-events:none}}
.node-meta{{font-size:10px;fill:#6b7280;pointer-events:none}}
.node-group{{cursor:pointer}}
.path{{display:flex;align-items:center;gap:8px;overflow-x:auto;padding:4px 2px 10px}}
.path-step{{min-width:150px;border:1px solid #dfe3e8;background:#fff;border-radius:12px;padding:10px;cursor:pointer}}
.path-step.active{{outline:2px solid #5865d8}}
.path-step.gt-match{{border-color:#54a273;background:#f0faf4}}
.gt-check{{display:inline-block;margin-left:5px;color:#267348;font-size:11px;font-weight:800}}
.scenario-grid{{display:grid;grid-template-columns:1fr;gap:12px}}
.scenario-box{{border:1px solid #e3e7ed;border-radius:12px;padding:14px;background:#fafbfc}}
.scenario-title{{font-size:12px;color:#6b7280;font-weight:800;margin-bottom:8px}}
.gt-path{{display:flex;align-items:center;gap:7px;overflow-x:auto;padding-bottom:5px}}
.gt-step{{white-space:nowrap;background:white;border:1px solid #cfd5dd;border-radius:10px;padding:8px 10px;font-size:12px;font-weight:700}}
.gt-arrow{{color:#9aa0a6}}
.scenario-metrics{{display:flex;flex-wrap:wrap;gap:7px;margin-top:10px}}
.scenario-metric{{border-radius:999px;background:#eef3ff;color:#3757a4;padding:5px 9px;font-size:11px;font-weight:700}}
.scenario-missing{{margin-top:9px;font-size:12px;color:#805b0b;line-height:1.5}}
.trajectory-title{{font-size:12px;color:#3757a4;font-weight:800;margin:14px 0 8px}}
.trajectory-path{{display:flex;align-items:center;gap:7px;overflow-x:auto;padding-bottom:7px}}
.trajectory-step{{white-space:nowrap;background:#fff;border:1px solid #dfe4ea;border-radius:10px;padding:8px 10px;font-size:12px;font-weight:700}}
.trajectory-step.gt-match{{border-color:#54a273;background:#f0faf4}}
.instance-divider{{white-space:nowrap;color:#7a8290;font-size:11px;font-weight:800;padding:0 3px}}
.trajectory-meta{{margin-top:9px;font-size:12px;color:#667085;line-height:1.6}}
.path-arrow{{color:#9aa0a6;font-size:20px}}
.input-list{{display:grid;grid-template-columns:minmax(120px,220px) minmax(160px,1fr) 110px;gap:0;border-top:1px solid #eee}}
.input-list>div{{padding:9px 8px;border-bottom:1px solid #eee;font-size:13px;overflow-wrap:anywhere}}
.muted{{color:#7b818a}}
.empty{{padding:18px;color:#7b818a;background:#fafafa;border-radius:10px}}
.evidence{{font-size:12px;line-height:1.65;color:#60656d;margin-top:12px;padding-top:12px;border-top:1px solid #eee}}
.note{{font-size:12px;color:#6b7280;line-height:1.6}}
@media(max-width:700px){{
  .wrap{{padding:14px}}
  .field{{min-width:100%}}
  .input-list{{grid-template-columns:1fr}}
  .input-list>div{{border-bottom:0}}
  .input-list>div:nth-child(3n){{border-bottom:1px solid #eee}}
}}
</style>
</head>
<body>
<div class="wrap">

  <div class="toolbar">
    <div class="field">
      <label for="scenarioSelect">PoC 시나리오</label>
      <select id="scenarioSelect"></select>
    </div>

    <div class="field">
      <label for="modelSelect">업무 프로세스 모델</label>
      <select id="modelSelect"></select>
    </div>

    <div class="field">
      <label for="instanceSelect">실제 실행 사례</label>
      <select id="instanceSelect"></select>
    </div>

    <div class="field">
      <label>고객 보고서</label>
      <a class="report-link" href="customer_workflow_report.html">고객용 업무 보고서 열기</a>
    </div>

    <div class="field">
      <label>품질 비교</label>
      <a class="report-link" href="../diagnostics/workflow_quality_report.html">업무 추출 품질 리포트 열기</a>
    </div>

    <div class="field">
      <label for="edgeThreshold">Graph edge 최소 transition ratio</label>
      <select id="edgeThreshold">
        <option value="0">0% — 전체</option>
        <option value="0.02">2%</option>
        <option value="0.05" selected>5% — 권장</option>
        <option value="0.10">10%</option>
        <option value="0.20">20%</option>
      </select>
    </div>
  </div>

  <div id="summary" class="summary"></div>

  <div class="panel">
    <h3>PoC Ground Truth와 사용자 업무 궤적 비교</h3>
    <div class="note">
      Ground Truth는 모델 학습에 사용하지 않고 결과 생성 후 사후 비교에만 사용합니다.
      시나리오를 선택하면 sessions.user_id로 연결한 동일 사용자의 대표 업무 궤적과 가장 관련 있는 실행 사례가 자동 선택됩니다.
    </div>
    <div id="scenarioComparison" class="scenario-grid"></div>
  </div>

  <div class="panel">
    <h3>모델이 추출한 실행 경로 (compressed path)</h3>
    <div class="note">
      조회·팝업·필드 선택 등 이름이 있는 하위 단계는 그대로 유지합니다.
      <code>SP03</code>처럼 prototype ID만 있어 고객이 해석할 수 없는 단계만 화면에서 숨깁니다.
    </div>
    <div id="instancePath" class="path"></div>
  </div>

  <div class="panel">
    <h3>Aggregate process graph</h3>
    <div class="note">
      실제 <code>process_edges</code>의 topology를 사용합니다.
      선택한 실행 사례의 path는 진한 선으로 강조됩니다.
    </div>
    <div class="graph-scroll">
      <svg id="graphSvg" role="img" aria-label="업무 프로세스 그래프"></svg>
    </div>
  </div>

  <div id="detail" class="panel"></div>

</div>

<script>
const DATA={preview_json};

const scenarioSelect=document.getElementById('scenarioSelect');
const scenarioComparison=document.getElementById('scenarioComparison');
const modelSelect=document.getElementById('modelSelect');
const instanceSelect=document.getElementById('instanceSelect');
const edgeThreshold=document.getElementById('edgeThreshold');
const summary=document.getElementById('summary');
const svg=document.getElementById('graphSvg');
const instancePath=document.getElementById('instancePath');
const detail=document.getElementById('detail');

let selectedNodeId=null;
let selectedInstanceStepId=null;

function txt(v){{
  return v===null||v===undefined?'':String(v);
}}

function byModel(arr,id){{
  return arr.filter(x=>x.model_id===id);
}}

function byInstance(arr,id){{
  return arr.filter(x=>x.instance_id===id);
}}

function esc(s){{
  return txt(s)
    .replaceAll('&','&amp;')
    .replaceAll('<','&lt;')
    .replaceAll('>','&gt;');
}}

function normScenario(v){{
  return txt(v)
    .toLowerCase()
    .replace(/[\s·ㆍ\-_/()\[\]{{}}:,.]+/g,'');
}}

function readableStep(s){{
  const label=txt(s?.display_label).trim();

  if(!label) return false;

  if(/^(SP|WP)\d+$/i.test(label)) return false;

  return true;
}}

function scenarioStepMatches(gt,label){{
  const a=normScenario(gt);
  const b=normScenario(label);

  if(!a||!b) return false;
  if(a===b) return true;

  if(Math.min(a.length,b.length)>=4){{
    return a.includes(b)||b.includes(a);
  }}

  return false;
}}

function currentScenario(){{
  return DATA.scenarios.find(
    x=>x.scenario_id===scenarioSelect.value
  )||null;
}}

function scenarioMetricsForSteps(scenario,steps){{
  const visible=steps
    .filter(readableStep)
    .map(s=>txt(s.display_label).trim());

  const gt=scenario?.ground_truth_steps||[];

  const matched=gt.filter(g=>
    visible.some(label=>scenarioStepMatches(g,label))
  );

  // LCS with fuzzy scenario-step matching.
  const n=gt.length;
  const m=visible.length;
  const dp=Array.from(
    {{length:n+1}},
    ()=>Array(m+1).fill(0)
  );

  for(let i=1;i<=n;i++){{
    for(let j=1;j<=m;j++){{
      if(scenarioStepMatches(gt[i-1],visible[j-1])){{
        dp[i][j]=dp[i-1][j-1]+1;
      }}else{{
        dp[i][j]=Math.max(dp[i-1][j],dp[i][j-1]);
      }}
    }}
  }}

  const missing=gt.filter(g=>!matched.includes(g));

  return {{
    matched,
    missing,
    matchedCount:matched.length,
    gtCount:gt.length,
    orderedCount:dp[n][m],
  }};
}}

function loadScenarios(){{
  scenarioSelect.innerHTML='';

  DATA.scenarios.forEach(s=>{{
    const o=document.createElement('option');
    o.value=s.scenario_id;
    o.textContent=`${{s.owner}} · ${{s.business_area}}`;
    scenarioSelect.appendChild(o);
  }});
}}

let preferredInstanceId=null;

function applyScenarioSelection(){{
  const sid=scenarioSelect.value;

  const match=DATA.scenario_matches.find(
    x=>x.scenario_id===sid
  );

  if(!match){{
    renderAll();
    return;
  }}

  const preferredModel=
    match.best_trajectory_first_model_id
    || match.best_model_id
    || '';

  const preferredInstance=
    match.best_trajectory_first_instance_id
    || match.best_instance_id
    || null;

  if(
    preferredModel
    && DATA.models.some(m=>m.model_id===preferredModel)
  ){{
    preferredInstanceId=preferredInstance;
    modelSelect.value=preferredModel;
    renderModel();
  }}else{{
    renderAll();
  }}
}}

function renderScenarioComparison(){{
  const scenario=currentScenario();

  if(!scenario){{
    scenarioComparison.innerHTML=
      '<div class="empty">PoC 시나리오를 선택하세요.</div>';
    return;
  }}

  const match=DATA.scenario_matches.find(
    x=>x.scenario_id===scenario.scenario_id
  )||null;

  const iid=instanceSelect.value;

  const currentSteps=byInstance(DATA.steps,iid)
    .sort((a,b)=>Number(a.step_order)-Number(b.step_order));

  const fallbackMetrics=scenarioMetricsForSteps(
    scenario,
    currentSteps
  );

  const gtHtml=(scenario.ground_truth_steps||[])
    .map((s,i)=>
      `${{i?'<span class="gt-arrow">→</span>':''}}`
      +`<span class="gt-step">${{esc(s)}}</span>`
    )
    .join('');

  let trajectoryHtml='';
  let matchedCount=fallbackMetrics.matchedCount;
  let orderedCount=fallbackMetrics.orderedCount;
  let gtCount=fallbackMetrics.gtCount;
  let missing=fallbackMetrics.missing;
  let trajectoryMeta='현재 선택한 단일 실행 사례 기준';

  if(
    match
    && txt(match.best_trajectory_instance_paths).trim()
  ){{
    const instancePaths=txt(
      match.best_trajectory_instance_paths
    )
      .split(' || ')
      .filter(Boolean);

    const chunks=[];

    instancePaths.forEach((path,instanceIdx)=>{{
      if(instanceIdx){{
        chunks.push(
          `<span class="instance-divider">| 다음 실행 |</span>`
        );
      }}

      const labels=path
        .split(' → ')
        .map(x=>x.trim())
        .filter(Boolean);

      labels.forEach((label,i)=>{{
        if(i){{
          chunks.push(
            '<span class="gt-arrow">→</span>'
          );
        }}

        const gtMatch=(scenario.ground_truth_steps||[])
          .some(g=>scenarioStepMatches(g,label));

        chunks.push(
          `<span class="trajectory-step ${{gtMatch?'gt-match':''}}">`
          +`${{esc(label)}}`
          +`${{gtMatch?'<span class="gt-check"> ✓</span>':''}}`
          +`</span>`
        );
      }});
    }});

    trajectoryHtml=chunks.join('');

    matchedCount=Number(
      match.matched_step_count||0
    );

    orderedCount=Number(
      match.ordered_match_count||0
    );

    gtCount=Number(
      match.ground_truth_step_count
      || scenario.ground_truth_steps.length
      || 0
    );

    missing=txt(
      match.missing_steps
    )
      .split(' | ')
      .filter(Boolean);

    const userId=txt(
      match.best_trajectory_user_display_id
    )||'-';

    const instanceCount=Number(
      match.best_trajectory_instance_count||0
    );

    const sessionCount=Number(
      match.best_trajectory_session_count||0
    );

    const supportUsers=Number(
      match.supporting_user_count||0
    );

    const models=txt(
      match.best_trajectory_workflow_prototypes
    )||'-';

    trajectoryMeta=
      `대표 사용자 ${{esc(userId)}} · `
      +`${{instanceCount}}개 instance / ${{sessionCount}}개 session · `
      +`관련 패턴 ${{esc(models)}} · `
      +`동일 기준 이상 관찰 사용자 ${{supportUsers}}명`;
  }}else{{
    trajectoryHtml=currentSteps
      .filter(readableStep)
      .map((s,i)=>{{
        const label=txt(s.display_label).trim();
        const gtMatch=(scenario.ground_truth_steps||[])
          .some(g=>scenarioStepMatches(g,label));

        return (
          `${{i?'<span class="gt-arrow">→</span>':''}}`
          +`<span class="trajectory-step ${{gtMatch?'gt-match':''}}">`
          +`${{esc(label)}}`
          +`${{gtMatch?'<span class="gt-check"> ✓</span>':''}}`
          +`</span>`
        );
      }})
      .join('');
  }}

  const missingText=missing.length
    ? missing.join(' · ')
    : '없음';

  scenarioComparison.innerHTML=
    `<div class="scenario-box">`
    +`<div class="scenario-title">기준 흐름 (Ground Truth)</div>`
    +`<div class="gt-path">${{gtHtml}}</div>`
    +`<div class="trajectory-title">동일 사용자 기반 대표 업무 궤적</div>`
    +`<div class="trajectory-path">${{trajectoryHtml||'<span class="muted">매칭 가능한 사용자 궤적 없음</span>'}}</div>`
    +`<div class="trajectory-meta">${{trajectoryMeta}}</div>`
    +`<div class="scenario-metrics">`
    +`<span class="scenario-metric">기준 단계 ${{matchedCount}} / ${{gtCount}} 확인</span>`
    +`<span class="scenario-metric">기준 순서 재현 ${{orderedCount}} / ${{gtCount}}</span>`
    +`</div>`
    +`<div class="scenario-missing"><b>미확인 기준 단계:</b> ${{esc(missingText)}}</div>`
    +`</div>`;
}}


function loadModels(){{
  modelSelect.innerHTML='';

  DATA.models.forEach(m=>{{
    const o=document.createElement('option');
    o.value=m.model_id;
    o.textContent=m.display_name||m.workflow_prototype;
    modelSelect.appendChild(o);
  }});

  if(DATA.models.length){{
    modelSelect.selectedIndex=0;
    renderModel();
  }}else{{
    detail.innerHTML='<div class="empty">표시할 workflow가 없습니다.</div>';
  }}
}}

function renderModel(){{
  const mid=modelSelect.value;
  const m=DATA.models.find(x=>x.model_id===mid);
  const instances=byModel(DATA.instances,mid);

  summary.innerHTML='';

  [
    ['패턴',m?.workflow_prototype],
    ['실행 수',m?.instance_count],
    ['노드',m?.node_count],
    ['edge',m?.edge_count],
    ['입력값',m?.input_count],
    ['평균 compressed steps',
      m?.avg_compressed_step_count!==undefined
        ? Number(m.avg_compressed_step_count).toFixed(2)
        : ''
    ],
  ].forEach(([k,v])=>{{
    const b=document.createElement('span');
    b.className='badge';
    b.textContent=`${{k}}: ${{txt(v)}}`;
    summary.appendChild(b);
  }});

  instanceSelect.innerHTML='';

  instances.forEach(x=>{{
    const o=document.createElement('option');
    o.value=x.instance_id;
    o.textContent=
      `${{x.source_workflow_id}} / ${{txt(x.session_id)}}`
      + ` · ${{txt(x.start_time)}}`
      + ` · steps ${{x.compressed_step_count}}`
      + ` · inputs ${{x.input_count}}`;
    instanceSelect.appendChild(o);
  }});

  if(!instances.length){{
    const o=document.createElement('option');
    o.value='';
    o.textContent='미리보기 실행 사례 없음';
    instanceSelect.appendChild(o);
  }}

  if(
    preferredInstanceId
    && instances.some(x=>x.instance_id===preferredInstanceId)
  ){{
    instanceSelect.value=preferredInstanceId;
  }}

  preferredInstanceId=null;

  selectedNodeId=null;
  selectedInstanceStepId=null;

  renderAll();
}}

function renderAll(){{
  renderScenarioComparison();
  renderInstancePath();
  renderGraph();
  renderDetail();
}}

function renderGraph(){{
  const mid=modelSelect.value;

  const nodes=byModel(DATA.nodes,mid)
    .sort((a,b)=>
      Number(a.avg_step_position||0)
      -Number(b.avg_step_position||0)
    );

  const threshold=Number(edgeThreshold.value||0);

  const edges=byModel(DATA.edges,mid)
    .filter(e=>Number(e.transition_ratio||0)>=threshold);

  const iid=instanceSelect.value;

  const steps=byInstance(DATA.steps,iid)
    .sort((a,b)=>Number(a.step_order)-Number(b.step_order));

  const instanceEdgeSet=new Set();

  for(let i=0;i<steps.length-1;i++){{
    instanceEdgeSet.add(
      `${{steps[i].node_id}}|||${{steps[i+1].node_id}}`
    );
  }}

  const instanceNodeSet=new Set(
    steps.map(s=>s.node_id)
  );

  const W=210;
  const H=74;
  const XGAP=250;
  const YGAP=110;
  const PAD=40;

  // avg_step_position을 1 단위 layer로 quantize
  const layers=new Map();

  nodes.forEach(n=>{{
    const layer=Math.max(
      1,
      Math.round(Number(n.avg_step_position||1))
    );

    if(!layers.has(layer)) layers.set(layer,[]);
    layers.get(layer).push(n);
  }});

  const layerKeys=[...layers.keys()].sort((a,b)=>a-b);

  const coords=new Map();

  let maxRows=1;

  layerKeys.forEach((layer,li)=>{{
    const arr=layers.get(layer)
      .sort((a,b)=>
        Number(b.support_instance_count||0)
        -Number(a.support_instance_count||0)
      );

    maxRows=Math.max(maxRows,arr.length);

    arr.forEach((n,ri)=>{{
      coords.set(
        n.node_id,
        {{
          x:PAD+li*XGAP,
          y:PAD+ri*YGAP
        }}
      );
    }});
  }});

  const width=Math.max(
    900,
    PAD*2+Math.max(1,layerKeys.length)*XGAP
  );

  const height=Math.max(
    260,
    PAD*2+maxRows*YGAP
  );

  svg.setAttribute('viewBox',`0 0 ${{width}} ${{height}}`);
  svg.setAttribute('width',width);
  svg.setAttribute('height',height);

  svg.innerHTML='';

  // arrow marker
  const defs=document.createElementNS(
    'http://www.w3.org/2000/svg',
    'defs'
  );

  defs.innerHTML=`
    <marker id="arrow" markerWidth="8" markerHeight="8"
      refX="7" refY="4" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L8,4 L0,8 z" fill="#adb4bf"></path>
    </marker>
    <marker id="arrowInstance" markerWidth="8" markerHeight="8"
      refX="7" refY="4" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L8,4 L0,8 z" fill="#5865d8"></path>
    </marker>
  `;

  svg.appendChild(defs);

  edges.forEach(e=>{{
    const a=coords.get(e.from_node_id);
    const b=coords.get(e.to_node_id);

    if(!a||!b) return;

    const key=`${{e.from_node_id}}|||${{e.to_node_id}}`;
    const inInstance=instanceEdgeSet.has(key);

    const x1=a.x+W;
    const y1=a.y+H/2;
    const x2=b.x;
    const y2=b.y+H/2;
    const dx=Math.max(40,(x2-x1)*0.55);

    const path=document.createElementNS(
      'http://www.w3.org/2000/svg',
      'path'
    );

    path.setAttribute(
      'd',
      `M ${{x1}} ${{y1}} C ${{x1+dx}} ${{y1}}, ${{x2-dx}} ${{y2}}, ${{x2}} ${{y2}}`
    );

    path.setAttribute(
      'class',
      inInstance?'edge instance':'edge'
    );

    path.setAttribute(
      'marker-end',
      inInstance
        ? 'url(#arrowInstance)'
        : 'url(#arrow)'
    );

    const title=document.createElementNS(
      'http://www.w3.org/2000/svg',
      'title'
    );

    title.textContent=
      `count=${{e.transition_count}}, ratio=${{Number(e.transition_ratio).toFixed(3)}}`;

    path.appendChild(title);
    svg.appendChild(path);
  }});

  nodes.forEach(n=>{{
    const c=coords.get(n.node_id);
    if(!c) return;

    const g=document.createElementNS(
      'http://www.w3.org/2000/svg',
      'g'
    );

    g.setAttribute('class','node-group');

    const rect=document.createElementNS(
      'http://www.w3.org/2000/svg',
      'rect'
    );

    let cls='node-rect';

    if(instanceNodeSet.has(n.node_id)) cls+=' instance';
    if(selectedNodeId===n.node_id) cls+=' selected';

    rect.setAttribute('class',cls);
    rect.setAttribute('x',c.x);
    rect.setAttribute('y',c.y);
    rect.setAttribute('width',W);
    rect.setAttribute('height',H);

    g.appendChild(rect);

    const t1=document.createElementNS(
      'http://www.w3.org/2000/svg',
      'text'
    );

    t1.setAttribute('class','node-label');
    t1.setAttribute('x',c.x+12);
    t1.setAttribute('y',c.y+25);

    const label=txt(
      n.display_label||n.segment_prototype
    );

    t1.textContent=
      label.length>25
        ? label.slice(0,24)+'…'
        : label;

    g.appendChild(t1);

    const t2=document.createElementNS(
      'http://www.w3.org/2000/svg',
      'text'
    );

    t2.setAttribute('class','node-meta');
    t2.setAttribute('x',c.x+12);
    t2.setAttribute('y',c.y+48);

    t2.textContent=
      `${{n.segment_prototype}} · instances ${{n.support_instance_count}}`;

    g.appendChild(t2);

    const t3=document.createElementNS(
      'http://www.w3.org/2000/svg',
      'text'
    );

    t3.setAttribute('class','node-meta');
    t3.setAttribute('x',c.x+12);
    t3.setAttribute('y',c.y+63);

    t3.textContent=
      `avg pos ${{Number(n.avg_step_position||0).toFixed(1)}} · inputs ${{n.input_count}}`;

    g.appendChild(t3);

    g.addEventListener('click',()=>{{
      selectedNodeId=n.node_id;
      selectedInstanceStepId=null;
      renderAll();
    }});

    svg.appendChild(g);
  }});
}}

function renderInstancePath(){{
  const iid=instanceSelect.value;

  const steps=byInstance(DATA.steps,iid)
    .sort((a,b)=>Number(a.step_order)-Number(b.step_order))
    .filter(readableStep);

  const scenario=currentScenario();
  const gt=scenario?.ground_truth_steps||[];

  instancePath.innerHTML='';

  if(!steps.length){{
    instancePath.innerHTML=
      '<div class="empty">사람이 읽을 수 있는 표시명이 있는 step이 없습니다.</div>';
    return;
  }}

  steps.forEach((s,i)=>{{
    if(i){{
      const a=document.createElement('span');
      a.className='path-arrow';
      a.textContent='→';
      instancePath.appendChild(a);
    }}

    const b=document.createElement('button');
    b.type='button';
    b.className='path-step';

    if(
      selectedInstanceStepId===s.instance_step_id
    ){{
      b.classList.add('active');
    }}

    const label=txt(
      s.display_label
    ).trim();

    const gtMatch=gt.some(
      g=>scenarioStepMatches(g,label)
    );

    if(gtMatch){{
      b.classList.add('gt-match');
    }}

    b.innerHTML=
      `<strong>STEP ${{i+1}}</strong>`
      +`${{gtMatch?'<span class="gt-check">✓ Ground Truth</span>':''}}`
      +`<br>`
      +`${{esc(label)}}<br>`
      +`<span class="muted">${{s.segment_prototype}}`
      +`${{Number(s.repeat_count)>1?` · 반복 ${{s.repeat_count}}회`:''}}`
      +`</span>`;

    b.addEventListener('click',()=>{{
      selectedInstanceStepId=s.instance_step_id;
      selectedNodeId=s.node_id;
      renderAll();
    }});

    instancePath.appendChild(b);
  }});
}}

function renderDetail(){{
  const iid=instanceSelect.value;

  let step=null;

  if(selectedInstanceStepId){{
    step=DATA.steps.find(
      x=>x.instance_step_id===selectedInstanceStepId
    );
  }}

  if(!step && selectedNodeId){{
    step=byInstance(DATA.steps,iid)
      .find(x=>x.node_id===selectedNodeId);
  }}

  if(!step){{
    const first=byInstance(DATA.steps,iid)
      .sort((a,b)=>Number(a.step_order)-Number(b.step_order))[0];

    if(first){{
      step=first;
      selectedNodeId=first.node_id;
      selectedInstanceStepId=first.instance_step_id;
    }}
  }}

  if(!step){{
    detail.innerHTML=
      '<div class="empty">실행 step을 선택하세요.</div>';
    return;
  }}

  const inputs=DATA.inputs
    .filter(
      x=>x.instance_step_id===step.instance_step_id
    )
    .sort(
      (a,b)=>
        String(a.event_time||'')
        .localeCompare(String(b.event_time||''))
    );

  detail.innerHTML='';

  const h=document.createElement('h3');

  h.textContent=
    `STEP ${{step.step_order}} · `
    +`${{step.display_label||step.segment_prototype}}`
    +`${{Number(step.repeat_count)>1?` · 반복 ${{step.repeat_count}}회`:''}}`;

  detail.appendChild(h);

  if(inputs.length){{
    const list=document.createElement('div');
    list.className='input-list';

    inputs.forEach(x=>{{
      const a=document.createElement('div');
      a.textContent=x.input_label||'입력값';

      const b=document.createElement('div');
      b.textContent=x.input_value||'';

      const c=document.createElement('div');
      c.className='muted';
      c.textContent=x.input_kind||'';

      list.append(a,b,c);
    }});

    detail.appendChild(list);

  }}else{{
    const e=document.createElement('div');
    e.className='empty';
    e.textContent=
      '이 실행 step에 연결된 input_data가 없습니다.';
    detail.appendChild(e);
  }}

  if(step.representative_evidence){{
    const ev=document.createElement('div');
    ev.className='evidence';
    ev.textContent=
      '대표 evidence: '
      +step.representative_evidence;
    detail.appendChild(ev);
  }}
}}

scenarioSelect.addEventListener(
  'change',
  applyScenarioSelection
);

modelSelect.addEventListener(
  'change',
  renderModel
);

instanceSelect.addEventListener(
  'change',
  ()=>{{
    selectedNodeId=null;
    selectedInstanceStepId=null;
    renderAll();
  }}
);

edgeThreshold.addEventListener(
  'change',
  renderGraph
);

loadScenarios();
loadModels();

if(DATA.scenarios.length){{
  scenarioSelect.selectedIndex=0;
  applyScenarioSelection();
}}
</script>
</body>
</html>
"""

(
    MAIN_RESULTS_DIR
    / 'workflow_preview.html'
).write_text(
    preview_html,
    encoding='utf-8'
)


# ============================================================
# P. Model bundle + manifest
# ============================================================

bundle_files = [
    'event_transformer.pt',
    'event_boundary_model.pt',
    'segment_transformer.pt',
    'workflow_boundary_model.pt',
    'workflow_transition_model.pt',
    'prototype_consensus_bundle.pt',
]

if KEEP_MODEL_BUNDLE:
    bundle = {
        'version':
            V62_VERSION,
        'dataset_name':
            DATASET_NAME,
        'config': {
            'event_window':
                EVENT_WINDOW,
            'event_stride':
                EVENT_STRIDE,
            'd_model':
                D_MODEL,
            'event_layers':
                EVENT_LAYERS,
            'segment_window':
                SEGMENT_WINDOW,
            'segment_stride':
                SEGMENT_STRIDE,
            'semantic_encoder':
                SEMANTIC_MODEL_NAME,
            'semantic_dim':
                int(
                    SEMANTIC_DIM
                ),
            'use_semantic_encoder':
                bool(
                    USE_SEMANTIC_ENCODER
                ),
            'use_salience_gate':
                False,
            'use_vicreg':
                False,
            'semantic_weight':
                float(
                    SEMANTIC_WEIGHT
                ),
            'ablation':
                'V5.7 calibrated segmentation + V6 multi-seed consensus prototypes + customer report',
            'use_transition_contrastive_workflow_boundary':
                bool(
                    USE_TRANSITION_CONTRASTIVE_WORKFLOW_BOUNDARY
                ),
            'workflow_transition_epochs':
                int(
                    WORKFLOW_TRANSITION_EPOCHS
                ),
            'workflow_transition_temperature':
                float(
                    WORKFLOW_TRANSITION_TEMPERATURE
                ),
            'workflow_transition_blend_weight':
                float(
                    WORKFLOW_TRANSITION_BLEND_WEIGHT
                ),
            'threshold_mode':
                THRESHOLD_MODE,
            'threshold_posterior_min':
                float(
                    THRESHOLD_POSTERIOR_MIN
                ),
            'threshold_bic_gain_min':
                float(
                    THRESHOLD_BIC_GAIN_MIN
                ),
            'threshold_min_separation_sigma':
                float(
                    THRESHOLD_MIN_SEPARATION_SIGMA
                ),
            'threshold_robust_z_fallback':
                float(
                    THRESHOLD_ROBUST_Z_FALLBACK
                ),
            'evaluation_v2_primary_mode':
                EVAL_V2_PRIMARY_MODE,
            'use_consensus_prototypes':
                bool(
                    USE_CONSENSUS_PROTOTYPES
                ),
            'prototype_ensemble_seeds':
                [
                    int(
                        x
                    )
                    for x in PROTOTYPE_ENSEMBLE_SEEDS
                ],
            'prototype_opt_steps':
                int(
                    PROTOTYPE_OPT_STEPS
                ),
            'workflow_consensus_k_range':
                [
                    int(
                        WORKFLOW_CONSENSUS_K_MIN
                    ),
                    int(
                        WORKFLOW_CONSENSUS_K_MAX
                    ),
                ],
            'report_target_min_workflows':
                int(
                    REPORT_TARGET_MIN_WORKFLOWS
                ),
            'use_structural_boundary':bool(USE_STRUCTURAL_BOUNDARY),
            'structural_fields':list(struct_cols),
            'struct_dim':int(STRUCT_DIM),
            'struct_target_weight':float(STRUCT_TARGET_WEIGHT),
        },
        'structural_vocabs':{c:struct_vocabs[c].itos for c in struct_cols},
        'checkpoints':
            {}
    }

    for fn in bundle_files:
        p = OUTPUT_DIR / fn

        if p.exists():
            bundle[
                'checkpoints'
            ][
                fn
            ] = torch.load(
                p,
                map_location='cpu'
            )

    torch.save(
        bundle,
        OUTPUT_DIR
        / 'model_bundle.pt'
    )


manifest = {
    'version':
        V62_VERSION,
    'dataset_name':
        DATASET_NAME,
    'output_dir':
        str(
            OUTPUT_DIR
        ),
    'main_results_dir':
        str(
            MAIN_RESULTS_DIR
        ),
    'events':
        int(
            len(
                df
            )
        ),
    'latent_segments':
        int(
            len(
                segdf
            )
        ),
    'latent_workflows':
        int(
            len(
                workflow_df
            )
        ),
    'session_safe_process_instances':
        int(
            len(
                process_instances
            )
        ),
    'process_models':
        int(
            len(
                process_models
            )
        ),
    'process_nodes':
        int(
            len(
                process_nodes
            )
        ),
    'process_edges':
        int(
            len(
                process_edges
            )
        ),
    'process_instance_steps':
        int(
            len(
                process_instance_steps
            )
        ),
    'process_node_inputs':
        int(
            len(
                process_node_inputs
            )
        ),
    'representation_model': {
        'base_pipeline':
            'V4.2',
        'semantic_encoder':
            SEMANTIC_MODEL_NAME,
        'semantic_dim':
            int(
                SEMANTIC_DIM
            ),
        'semantic_weight':
            float(
                SEMANTIC_WEIGHT
            ),
        'semantic_weight_candidates':
            SEMANTIC_WEIGHT_CANDIDATES,
        'learned_salience_gate':
            False,
        'vicreg':
            False,
        'event_embedding_cosine_mean':
            float(
                event_embedding_cosine_mean
            ),
    },
    'boundary_model':{
        'base':'V5.3 contextual neural boundary',
        'structural_aware':bool(USE_STRUCTURAL_BOUNDARY),
        'structural_fields':list(struct_cols),
        'struct_dim':int(STRUCT_DIM),
        'struct_target_weight':float(STRUCT_TARGET_WEIGHT),
        'summary':structural_boundary_summary,
    },
    'workflow_boundary_model': {
        'base':
            'V5.4 contextual workflow boundary',
        'transition_contrastive':
            bool(
                USE_TRANSITION_CONTRASTIVE_WORKFLOW_BOUNDARY
            ),
        'transition_temperature':
            float(
                WORKFLOW_TRANSITION_TEMPERATURE
            ),
        'transition_blend_weight':
            float(
                WORKFLOW_TRANSITION_BLEND_WEIGHT
            ),
        'workflow_representation':
            'V5.4 mean pooling retained',
        'diagnostics':
            workflow_transition_diagnostics,
    },
    'threshold_calibration': {
        'mode':
            THRESHOLD_MODE,
        'posterior_min':
            float(
                THRESHOLD_POSTERIOR_MIN
            ),
        'bic_gain_min':
            float(
                THRESHOLD_BIC_GAIN_MIN
            ),
        'min_separation_sigma':
            float(
                THRESHOLD_MIN_SEPARATION_SIGMA
            ),
        'robust_z_fallback':
            float(
                THRESHOLD_ROBUST_Z_FALLBACK
            ),
        'workflow_signal_summary':
            workflow_threshold_signal_summary
            .replace(
                {
                    np.nan:
                        None
                }
            )
            .to_dict(
                orient='records'
            ),
        'workflow_cut_jaccard':
            workflow_cut_jaccard
            .replace(
                {
                    np.nan:
                        None
                }
            )
            .to_dict(
                orient='records'
            ),
    },
    'prototype_training':
        'original V4.2 objective retained; multi-seed longer optimization + consensus final assignment',
    'prototype_diagnostics':
        prototype_diagnostics,
    'prototype_stability_diagnostics':
        prototype_stability_diagnostics,
    'prototype_consensus': {
        'enabled':
            bool(
                USE_CONSENSUS_PROTOTYPES
            ),
        'seeds':
            [
                int(
                    x
                )
                for x in PROTOTYPE_ENSEMBLE_SEEDS
            ],
        'optimization_steps':
            int(
                PROTOTYPE_OPT_STEPS
            ),
        'segment':
            segment_ensemble[
                'consensus_summary'
            ],
        'workflow':
            workflow_ensemble[
                'consensus_summary'
            ],
    },
    'v6_2_presentation': {
        'full_data_run':
            bool(
                FULL_DATA_RUN
            ),
        'input_base_dir':
            str(
                BASE_DIR
            ),
        'poc_scenario_count':
            int(
                len(
                    POC_SCENARIOS
                )
            ),
        'scenario_comparison_posthoc_only':
            True,
        'hide_unreadable_prototype_only_steps_in_html':
            True,
        'keep_named_low_level_steps':
            True,
        'user_trajectory_layer':
            True,
        'user_id_used_as_embedding_feature':
            False,
        'user_id_source':
            'sessions.session_id -> sessions.user_id, event identity fallback only when mapping absent',
        'trajectory_gap_minutes':
            int(
                USER_TRAJECTORY_GAP_MINUTES
            ),
        'trajectory_max_window_instances':
            int(
                USER_TRAJECTORY_MAX_WINDOW_INSTANCES
            ),
    },
    'user_trajectory_analysis': {
        'enabled':
            bool(
                USE_USER_TRAJECTORY
            ),
        'mapped_session_count':
            int(
                len(
                    session_user_map
                )
            ),
        'user_count':
            int(
                user_trajectory_instances[
                    'trajectory_user_id'
                ].nunique()
            )
            if len(
                user_trajectory_instances
            )
            else 0,
        'trajectory_episode_count':
            int(
                len(
                    user_trajectory_summary
                )
            ),
        'cross_session_episode_count':
            int(
                user_trajectory_summary[
                    'is_cross_session'
                ].sum()
            )
            if len(
                user_trajectory_summary
            )
            else 0,
        'trajectory_window_count':
            int(
                len(
                    user_trajectory_windows
                )
            ),
        'scenario_match_candidate_count':
            int(
                len(
                    scenario_user_trajectory_matches
                )
            ),
        'support_score_threshold':
            float(
                USER_TRAJECTORY_SUPPORT_SCORE
            ),
        'user_id_as_model_feature':
            False,
        'raw_user_id_exported':
            False,
        'customer_user_identifier':
            'stable SHA1 prefix alias USER-XXXXXXXX',
    },
    'customer_reporting': {
        'target_min_workflows':
            int(
                REPORT_TARGET_MIN_WORKFLOWS
            ),
        'selected_workflow_count':
            int(
                len(
                    reportable_workflows
                )
            ),
        'strict_reportable_count':
            int(
                (
                    reportable_workflows[
                        'report_status'
                    ]=='Reportable'
                ).sum()
            )
            if len(
                reportable_workflows
            )
            else 0,
        'report_file':
            'customer_workflow_report.html',
        'reportability_score_threshold':
            float(
                REPORT_SCORE_THRESHOLD
            ),
    },
    'gold_evaluation':
        gold_eval_metrics,
    'evaluation_v2':
        evaluation_v2_metrics,
    'diagnostic_results': [
        'gold_anchor_confusion.csv',
        'gold_anchor_summary.csv',
        'gold_pair_cooccurrence.csv',
        'evaluation_v2_anchor_provenance.csv',
        'evaluation_v2_summary.csv',
        'evaluation_v2_instance_labels.csv',
        'evaluation_v2_pair_cooccurrence.csv',
        'event_threshold_diagnostics.csv',
        'workflow_threshold_diagnostics.csv',
        'workflow_threshold_signal_summary.csv',
        'workflow_cut_jaccard.csv',
        'prototype_ensemble_runs.csv',
        'prototype_seed_pairwise.csv',
        'consensus_k_search.csv',
        'segment_consensus_assignments.csv',
        'workflow_consensus_assignments.csv',
        'workflow_quality_report.html',
        'workflow_transition_training.csv',
        'workflow_boundary_scores.csv',
        'poc_scenario_instance_matches.csv',
        'user_trajectory_instances.csv',
        'user_trajectory_windows.csv',
        'poc_scenario_user_trajectory_matches.csv',
    ]
    if gold_eval_metrics.get(
        'enabled'
    )
    else [],
    'input_policy':
        'QA raw input_data export; no additional masking in V6.2; sessions.user_id used only for higher-level trajectory grouping',
    'main_results': [
        'workflow_preview.html',
        'customer_workflow_report.html',
        'developer_handoff.md',
        'process_models.csv',
        'process_nodes.csv',
        'process_edges.csv',
        'process_instances.csv',
        'process_instance_steps.csv',
        'process_node_inputs.csv',
        'reportable_workflows.csv',
        'reportable_workflow_steps.csv',
        'workflow_reportability.csv',
        'poc_scenario_comparison.csv',
        'user_trajectory_summary.csv',
    ],
    'note':
        'V6.2 skips V6.1 as a delivery version while inheriting its full-data and customer-view changes. '
        'The V6.0 event/segment/workflow model remains unchanged. sessions.user_id is joined only after local '
        'workflow extraction and is used to connect session-safe process instances into generic temporal user '
        'trajectory episodes. User trajectory windows provide a higher-level scenario signal across sessions. '
        'user_id is not added to event embeddings, prototype training or consensus clustering. '
        'The seven PoC Ground Truth scenarios remain post-hoc presentation/evaluation references only.'
}

(
    OUTPUT_DIR
    / 'run_manifest.json'
).write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str
    ),
    encoding='utf-8'
)


# ============================================================
# Q. Remove debug/intermediate artifacts
# ============================================================

keep_root = {
    'main_results',
    'diagnostics',
    'run_manifest.json',
}

if KEEP_MODEL_BUNDLE:
    keep_root.add(
        'model_bundle.pt'
    )

for p in list(
    OUTPUT_DIR.iterdir()
):
    if p.name in keep_root:
        continue

    if p.is_dir():
        shutil.rmtree(
            p
        )
    else:
        p.unlink()


print(
    '\n=== V6.2 DONE ==='
)

print(
    'OUTPUT_DIR:',
    OUTPUT_DIR
)

print(
    'MAIN_RESULTS_DIR:',
    MAIN_RESULTS_DIR
)

print(
    '\n개발자/고객 확인 파일:'
)

for fn in [
    'workflow_preview.html',
    'developer_handoff.md',
    'process_models.csv',
    'process_nodes.csv',
    'process_edges.csv',
    'process_instances.csv',
    'process_instance_steps.csv',
    'process_node_inputs.csv',
]:
    print(
        '-',
        MAIN_RESULTS_DIR
        / fn
    )

print(
    '\n고객용 품질 비교 리포트:'
)

print(
    '-',
    DIAGNOSTICS_DIR
    / 'workflow_quality_report.html'
)

print(
    '\n고객용 업무 프로세스 보고서:'
)

print(
    '-',
    MAIN_RESULTS_DIR
    / 'customer_workflow_report.html'
)

print(
    '\nV6.2 policy:'
)

print(
    '- session-safe display instances'
)

print(
    '- consecutive SP compression'
)

print(
    '- actual edge-based graph preview'
)

print(
    '- readable presentation labels'
)

print(
    '- V5.4 workflow mean pooling retained'
)

print(
    '- V5.6 transition-contrastive workflow boundary signal retained'
)

print(
    '- V5.5 diagnostics UI retained'
)

print(
    '- calibrated event/workflow threshold'
)

print(
    '- Evaluation v2: ARI/NMI/V-measure + anchor sensitivity'
)

print(
    '- prototype seed stability diagnostics'
)

print(
    '- gold process labels remain evaluation-only'
)

print(
    '- V5.7 calibrated threshold retained'
)

print(
    '- multi-seed consensus SP/WP final assignment'
)

print(
    f'- customer report target: at least {REPORT_TARGET_MIN_WORKFLOWS} workflows when available'
)

print(
    '- gold process labels remain evaluation-only'
)

print(
    '- QA raw input_data: no additional masking'
)

print(
    '- V5.4 controlled ablation: V5.2 + reduced semantic influence'
)

print(
    '- semantic encoder:',
    SEMANTIC_MODEL_NAME
)

print(
    '- semantic weight:',
    SEMANTIC_WEIGHT
)

print('- structural boundary summary:',structural_boundary_summary)

print(
    '- gold evaluation metrics:',
    gold_eval_metrics
)

print(
    '- learned salience gate:',
    USE_SALIENCE_GATE
)

print(
    '- VICReg:',
    USE_VICREG
)

print(
    '- event embedding cosine mean:',
    event_embedding_cosine_mean
)

print(
    '- prototype diagnostics:',
    prototype_diagnostics
)


<>:9352: SyntaxWarning: invalid escape sequence '\s'
<>:9354: SyntaxWarning: invalid escape sequence '\d'
<>:9352: SyntaxWarning: invalid escape sequence '\s'
<>:9354: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1271/3989871301.py:9352: SyntaxWarning: invalid escape sequence '\s'
  MAIN_RESULTS_DIR
/tmp/ipykernel_1271/3989871301.py:9354: SyntaxWarning: invalid escape sequence '\d'
  ).write_text(


user trajectory users: 11
user trajectory episodes: 26
cross-session trajectory episodes: 25
\n=== V6.2 POC 7 SCENARIO COMPARISON ===
scenario_id       owner comparison_source best_workflow_prototype                   best_trajectory_workflow_prototypes  supporting_user_count  matched_step_count  ground_truth_step_count  ordered_match_count  coverage_ratio  order_ratio
       SC01        국제교류   user_trajectory                    WP04 WP04 | WP03 | WP06 | WP07 | WP01 | WP02 | WP11 | WP10                      1                   6                        6                    6        1.000000     1.000000
       SC02       교무/수업   user_trajectory                    WP05                                                  WP05                      1                   6                        9                    6        0.666667     0.666667
       SC03    학생처/장학등록   user_trajectory                    WP05                                                  WP05                      1          